# 02 - Data Quality Analysis

**Objective:** Comprehensive data audit: missing values, duplicate records, schema mismatches, and ball-by-ball integrity validation.

---
### supeRR Selector Pressure Analysis


In [1]:
# Setup and imports
import sys
from pathlib import Path

# Ensure src module is accessible
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from superr_selector.data.loader import DataLoader

%matplotlib inline
print('Environment initialized successfully.')


Environment initialized successfully.


## 1. Load Datasets


In [2]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 2. DEFINE PROJECT PATH
# ==========================================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

# ==========================================
# 3. LOAD DATASETS
# ==========================================

datasets = {
    "ball_by_ball": pd.read_csv(DATA_DIR / "ball_by_ball.csv"),
    "competition_team_player_mapping": pd.read_csv(
        DATA_DIR / "competition_team_player_mapping.csv"
    ),
    "competitions": pd.read_csv(DATA_DIR / "competitions.csv"),
    "matches": pd.read_csv(DATA_DIR / "matches.csv"),
    "playing_eleven": pd.read_csv(DATA_DIR / "playing_eleven.csv"),
    "teams": pd.read_csv(DATA_DIR / "teams.csv")
}

# ==========================================
# 4. CREATE DATAFRAME REFERENCES
# ==========================================

ball_df = datasets["ball_by_ball"]
matches_df = datasets["matches"]
competitions_df = datasets["competitions"]
mapping_df = datasets["competition_team_player_mapping"]
playing_eleven_df = datasets["playing_eleven"]

# ==========================================
# 5. VERIFY DATA LOADING
# ==========================================

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

ball_by_ball: (200999, 33)
competition_team_player_mapping: (6413, 6)
competitions: (141, 4)
matches: (883, 22)
playing_eleven: (28381, 9)
teams: (353, 3)


In [3]:
# ==========================================
# STEP 17: INVESTIGATE RUNS AND EXTRAS
# ==========================================

# Calculate extras without modifying the original dataset
extras_total = (
    ball_df["extras1_runs"] + ball_df["extras_2_runs"]
)

# 1. Basic statistics
print("RUNS AND EXTRAS SUMMARY")

print("\nTotal runs column:")
display(ball_df["runs"].describe())

print("\nExtras1 runs:")
display(ball_df["extras1_runs"].value_counts().sort_index())

print("\nExtras2 runs:")
display(ball_df["extras_2_runs"].value_counts().sort_index())


# 2. Compare runs with extras
print("\nCOMPARISON")

print("Total runs recorded in runs column:",
      ball_df["runs"].sum())

print("Total extras1 runs:",
      ball_df["extras1_runs"].sum())

print("Total extras2 runs:",
      ball_df["extras_2_runs"].sum())

print("Combined extras:",
      extras_total.sum())


# 3. Inspect deliveries containing extras
print("\nSAMPLE DELIVERIES WITH EXTRAS")

extras_samples = ball_df.loc[
    extras_total > 0,
    [
        "match_id",
        "innings",
        "overid",
        "over_ball_num",
        "striker_name",
        "bowler_name",
        "runs",
        "extras1_runs",
        "extras_2_runs",
        "extras_type"
    ]
].head(20)

display(extras_samples)


# 4. Check whether runs equals extras
print("\nRUNS VS EXTRAS CHECK")

print(
    "Deliveries where runs == combined extras:",
    (ball_df["runs"] == extras_total).sum()
)

print(
    "Deliveries where runs > combined extras:",
    (ball_df["runs"] > extras_total).sum()
)

print(
    "Deliveries where runs < combined extras:",
    (ball_df["runs"] < extras_total).sum()
)

RUNS AND EXTRAS SUMMARY

Total runs column:


count    200999.000000
mean          1.341051
std           1.730127
min           0.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           6.000000
Name: runs, dtype: float64


Extras1 runs:


extras1_runs
0    189167
1     10672
2       458
3        47
4       381
5       274
Name: count, dtype: int64


Extras2 runs:


extras_2_runs
0    200963
1        20
4        16
Name: count, dtype: int64


COMPARISON
Total runs recorded in runs column: 269550
Total extras1 runs: 14623
Total extras2 runs: 84
Combined extras: 14707

SAMPLE DELIVERIES WITH EXTRAS


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type
39,13511,1,7,4,Eoin Morgan,Akeal Hosein,0,1,0,wide
43,13511,1,8,1,Eoin Morgan,Romario Shepherd,0,1,0,wide
44,13511,1,8,2,Eoin Morgan,Romario Shepherd,0,1,0,wide
143,13511,2,4,5,Brandon King,Adil Rashid,0,4,0,legbyes
158,13511,2,7,2,Shai Hope,Liam Dawson,0,1,0,legbyes
278,13512,1,10,1,Moeen Ali,Kieron Pollard,0,1,0,wide
281,13512,1,10,4,Moeen Ali,Kieron Pollard,0,1,0,wide
292,13512,1,12,1,Moeen Ali,Jason Holder,0,1,0,wide
307,13512,1,14,3,Sam Billings,Kieron Pollard,0,1,0,wide
320,13512,1,16,3,Eoin Morgan,Kieron Pollard,0,1,0,wide



RUNS VS EXTRAS CHECK
Deliveries where runs == combined extras: 68250
Deliveries where runs > combined extras: 121389
Deliveries where runs < combined extras: 11360


In [4]:
# ==========================================
# STEP 18: VALIDATE BATTER RUNS VS EXTRAS
# ==========================================

ball_df["total_extras"] = (
    ball_df["extras1_runs"] + ball_df["extras_2_runs"]
)

# 1. How many deliveries contain extras?
print("Deliveries with extras:")
print((ball_df["total_extras"] > 0).sum())

# 2. Deliveries with extras and zero batter runs
print("\nExtras with zero runs:")
print(
    (
        (ball_df["total_extras"] > 0) &
        (ball_df["runs"] == 0)
    ).sum()
)

# 3. Deliveries with extras and non-zero batter runs
print("\nExtras with non-zero runs:")
print(
    (
        (ball_df["total_extras"] > 0) &
        (ball_df["runs"] > 0)
    ).sum()
)

# 4. Examine deliveries with both batter runs and extras
print("\nDeliveries with both runs and extras:")

display(
    ball_df.loc[
        (ball_df["runs"] > 0) &
        (ball_df["total_extras"] > 0),
        [
            "striker_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "total_extras",
            "extras_type",
            "is_four",
            "is_six"
        ]
    ].head(20)
)

# 5. Calculate possible total delivery runs
ball_df["calculated_total_runs"] = (
    ball_df["runs"] + ball_df["total_extras"]
)

print("\nCalculated total runs across all deliveries:")
print(ball_df["calculated_total_runs"].sum())

Deliveries with extras:
11833

Extras with zero runs:
11360

Extras with non-zero runs:
473

Deliveries with both runs and extras:


,striker_name,runs,extras1_runs,extras_2_runs,total_extras,extras_type,is_four,is_six
1226,Sam Billings,2,1,0,1,other,False,False
1229,Sam Billings,1,1,0,1,Overstep,False,False
1486,Ishan Kishan,1,1,0,1,Overstep,False,False
3309,Mahendra Singh Dhoni,6,1,0,1,other,False,True
4187,Jos Buttler,4,1,0,1,Overstep,True,False
4192,Yashasvi Jaiswal,1,1,0,1,Overstep,False,False
4610,Shahbaz Ahmed,1,1,0,1,Overstep,False,False
5099,Andre Russell,1,1,0,1,Overstep,False,False
5111,Andre Russell,4,1,0,1,Overstep,True,False
5265,Ishan Kishan,4,1,0,1,Overstep,True,False



Calculated total runs across all deliveries:
284257


In [5]:
# ==========================================
# STEP 19: EXTRAS CATEGORY ANALYSIS
# ==========================================

# Frequency of each extras category
print("EXTRAS CATEGORY DISTRIBUTION")

display(
    ball_df.groupby("extras_type", dropna=False).agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        total_extras=("total_extras", "sum")
    ).sort_values("deliveries", ascending=False)
)


# Inspect Overstep deliveries
print("\nOVERSTEP DELIVERY SAMPLES")

display(
    ball_df.loc[
        ball_df["extras_type"] == "Overstep",
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(15)
)

EXTRAS CATEGORY DISTRIBUTION


,deliveries,batter_runs,extras1_runs,extras2_runs,total_extras
extras_type,,,,,
NaN,189167,268285,1,1,2
wide,7714,0,9174,0,9174
legbyes,2748,1,3603,0,3603
Overstep,576,852,576,48,624
byes,529,0,1000,0,1000
other,264,412,264,35,299
Penalty,1,0,5,0,5



OVERSTEP DELIVERY SAMPLES


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
495,13513,1,4,2,Nicholas Pooran,George Garton,0,1,0,False,NaN
1229,13515,2,20,1,Sam Billings,Jason Holder,1,1,0,False,NaN
1485,13523,1,1,4,Ishan Kishan,Akeal Hosein,0,1,4,False,NaN
1486,13523,1,1,5,Ishan Kishan,Akeal Hosein,1,1,0,False,NaN
3191,14094,1,1,1,Ruturaj Gaikwad,Umesh Yadav,0,1,0,False,NaN
4169,14098,1,1,5,Jos Buttler,Bhuvneshwar Kumar,0,1,0,False,NaN
4181,14098,1,3,4,Yashasvi Jaiswal,Bhuvneshwar Kumar,0,1,0,False,NaN
4187,14098,1,4,3,Jos Buttler,Umran Malik,4,1,0,False,NaN
4192,14098,1,5,1,Yashasvi Jaiswal,Washington Sundar,1,1,0,False,NaN
4452,14099,1,6,3,Nitish Rana,Akash Deep,0,1,0,False,NaN


In [6]:
# STEP 20: WICKET ANALYSIS BY EXTRAS CATEGORY

overstep_df = ball_df[ball_df["extras_type"] == "Overstep"]

print("OVERSTEP DELIVERY SUMMARY")
print("Total Overstep deliveries:", len(overstep_df))
print("Overstep deliveries with wickets:", overstep_df["is_wicket"].sum())

print("\nWICKET TYPES ON OVERSTEP DELIVERIES")

display(
    overstep_df[overstep_df["is_wicket"] == True]
    .groupby("dismissal_type", dropna=False)
    .agg(
        wicket_deliveries=("match_id", "size"),
        dismissed_players=("dismissed_player_id", "count")
    )
    .sort_values("wicket_deliveries", ascending=False)
)

print("\nALL WICKET TYPES IN THE DATASET")

display(
    ball_df[ball_df["is_wicket"] == True]
    .groupby("dismissal_type", dropna=False)
    .size()
    .sort_values(ascending=False)
    .to_frame("count")
)

OVERSTEP DELIVERY SUMMARY
Total Overstep deliveries: 576
Overstep deliveries with wickets: 5

WICKET TYPES ON OVERSTEP DELIVERIES


,wicket_deliveries,dismissed_players
dismissal_type,,
RunOut,5,5



ALL WICKET TYPES IN THE DATASET


,count
dismissal_type,
Caught,7557
Bowled,1775
LBW,709
RunOut,600
Stump,214
Hit Wicket,17
Retire Out,6
Obstracting the field,2
Obstructing the field,1


In [7]:
# STEP 21: INVESTIGATE BALL NUMBERING

print("OVER BALL NUMBER DISTRIBUTION")

display(
    ball_df["over_ball_num"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("delivery_count")
)

print("\nINNINGS BALL NUMBER DISTRIBUTION")

display(
    ball_df["innings_ball_num"]
    .describe()
    .to_frame("innings_ball_num_statistics")
)

print("\nOVERSTEP DELIVERY NUMBERING SAMPLES")

display(
    ball_df.loc[
        ball_df["extras_type"] == "Overstep",
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "extras_type",
            "runs",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].head(20)
)

print("\nWIDE DELIVERY NUMBERING SAMPLES")

display(
    ball_df.loc[
        ball_df["extras_type"] == "wide",
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "extras_type",
            "runs",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].head(20)
)

OVER BALL NUMBER DISTRIBUTION


,delivery_count
over_ball_num,
1,32350
2,32257
3,32169
4,32066
5,31937
6,31791
7,6710
8,1364
9,279



INNINGS BALL NUMBER DISTRIBUTION


,innings_ball_num_statistics
count,200999.000000
mean,60.457256
std,35.403761
min,1.000000
25%,30.000000
50%,60.000000
75%,90.000000
max,136.000000



OVERSTEP DELIVERY NUMBERING SAMPLES


,match_id,innings,overid,over_ball_num,innings_ball_num,extras_type,runs,extras1_runs,extras_2_runs
495,13513,1,4,2,20,Overstep,0,1,0
1229,13515,2,20,1,122,Overstep,1,1,0
1485,13523,1,1,4,4,Overstep,0,1,4
1486,13523,1,1,5,5,Overstep,1,1,0
3191,14094,1,1,1,1,Overstep,0,1,0
4169,14098,1,1,5,5,Overstep,0,1,0
4181,14098,1,3,4,17,Overstep,0,1,0
4187,14098,1,4,3,23,Overstep,4,1,0
4192,14098,1,5,1,28,Overstep,1,1,0
4452,14099,1,6,3,33,Overstep,0,1,0



WIDE DELIVERY NUMBERING SAMPLES


,match_id,innings,overid,over_ball_num,innings_ball_num,extras_type,runs,extras1_runs,extras_2_runs
39,13511,1,7,4,40,wide,0,1,0
43,13511,1,8,1,44,wide,0,1,0
44,13511,1,8,2,45,wide,0,1,0
278,13512,1,10,1,55,wide,0,1,0
281,13512,1,10,4,58,wide,0,1,0
292,13512,1,12,1,69,wide,0,1,0
307,13512,1,14,3,84,wide,0,1,0
320,13512,1,16,3,97,wide,0,1,0
330,13512,1,17,6,107,wide,0,1,0
337,13512,1,18,6,114,wide,0,1,0


In [8]:
# STEP 22: VALIDATE INNINGS DELIVERY SEQUENCE

# Sort deliveries in their recorded innings order
sequence_df = ball_df.sort_values(
    ["match_id", "innings", "innings_ball_num"]
).copy()

# Calculate the difference between consecutive delivery numbers
sequence_df["ball_number_diff"] = (
    sequence_df.groupby(["match_id", "innings"])["innings_ball_num"]
    .diff()
)

# Check whether delivery numbers increment by exactly one
sequence_summary = sequence_df.groupby(
    ["match_id", "innings"]
).agg(
    delivery_rows=("innings_ball_num", "size"),
    unique_ball_numbers=("innings_ball_num", "nunique"),
    minimum_ball_number=("innings_ball_num", "min"),
    maximum_ball_number=("innings_ball_num", "max"),
    non_sequential_gaps=(
        "ball_number_diff",
        lambda x: ((x.notna()) & (x != 1)).sum()
    )
)

print("INNINGS SEQUENCE VALIDATION")
display(sequence_summary.describe())

print("\nTOTAL NON-SEQUENTIAL TRANSITIONS:")
print(sequence_summary["non_sequential_gaps"].sum())

print("\nEXAMPLE NON-SEQUENTIAL TRANSITIONS:")

display(
    sequence_df.loc[
        sequence_df["ball_number_diff"].notna()
        & (sequence_df["ball_number_diff"] != 1),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "ball_number_diff",
            "extras_type"
        ]
    ].head(20)
)

INNINGS SEQUENCE VALIDATION


,delivery_rows,unique_ball_numbers,minimum_ball_number,maximum_ball_number,non_sequential_gaps
count,1724.000000,1724.000000,1724.0,1724.000000,1724.0
mean,116.588747,116.588747,1.0,116.588747,0.0
std,19.697000,19.697000,0.0,19.697000,0.0
min,1.000000,1.000000,1.0,1.000000,0.0
25%,117.000000,117.000000,1.0,117.000000,0.0
50%,123.000000,123.000000,1.0,123.000000,0.0
75%,126.000000,126.000000,1.0,126.000000,0.0
max,136.000000,136.000000,1.0,136.000000,0.0



TOTAL NON-SEQUENTIAL TRANSITIONS:
0

EXAMPLE NON-SEQUENTIAL TRANSITIONS:


,match_id,innings,overid,over_ball_num,innings_ball_num,ball_number_diff,extras_type


In [9]:
# STEP 23: INVESTIGATE UNKNOWN EXTRAS

other_df = ball_df[ball_df["extras_type"] == "other"].copy()

print("OTHER EXTRAS SUMMARY")
print("Total deliveries:", len(other_df))
print("Total batter runs:", other_df["runs"].sum())
print("Total extras1 runs:", other_df["extras1_runs"].sum())
print("Total extras2 runs:", other_df["extras_2_runs"].sum())
print("Wickets:", other_df["is_wicket"].sum())

print("\nOTHER EXTRAS DISTRIBUTION")

display(
    other_df.groupby(
        ["extras1_runs", "extras_2_runs", "is_wicket"],
        dropna=False
    )
    .size()
    .to_frame("delivery_count")
    .sort_values("delivery_count", ascending=False)
)

print("\nOTHER EXTRAS SAMPLES")

display(
    other_df[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(25)
)

print("\nDELIVERIES WITH EXTRAS BUT MISSING CATEGORY")

display(
    ball_df[
        ball_df["extras_type"].isna()
        & (
            (ball_df["extras1_runs"] > 0)
            | (ball_df["extras_2_runs"] > 0)
        )
    ][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ]
)

OTHER EXTRAS SUMMARY
Total deliveries: 264
Total batter runs: 412
Total extras1 runs: 264
Total extras2 runs: 35
Wickets: 3

OTHER EXTRAS DISTRIBUTION


delivery_count
extras1_runs extras_2_runs is_wicket                
1            0             False                 247
             1             False                   7
             4             False                   7
             0             True                    3


OTHER EXTRAS SAMPLES


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,1,10,1,57,Rovman Powell,Tymal Mills,0,1,0,False,NaN
1079,13515,1,16,6,99,Rovman Powell,Saqib Mahmood,0,1,0,False,NaN
1226,13515,2,19,5,119,Sam Billings,Sheldon Cottrell,2,1,0,False,NaN
2588,13614,1,18,5,111,Dasun Shanaka,Mitchell Starc,0,1,4,False,NaN
2609,13614,2,2,1,7,Ashton Agar,Nuwan Thushara,0,1,0,False,NaN
3309,14094,1,19,7,119,Mahendra Singh Dhoni,Shivam Mavi,6,1,0,False,NaN
4413,14098,2,20,1,119,Bhuvneshwar Kumar,Nathan Coulter Nile,0,1,0,False,NaN
5012,14101,1,17,4,105,Kagiso Rabada,Tim Southee,0,1,0,False,NaN
5365,14102,2,19,9,122,Kieron Pollard,Prasidh Krishna,0,1,0,True,RunOut
6575,14107,2,16,5,95,Patrick Cummins,Daniel Sams,2,1,0,False,NaN



DELIVERIES WITH EXTRAS BUT MISSING CATEGORY


,match_id,innings,overid,over_ball_num,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
36854,17555,1,5,4,0,0,1,False,NaN
107489,36097,2,8,6,0,1,0,False,NaN


In [10]:
# STEP 24: INSPECT OTHER-EXTRAS WICKETS

print("WICKET DETAILS FOR OTHER EXTRAS")

display(
    other_df.loc[
        other_df["is_wicket"] == True,
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "dismissal_type",
            "dismissed_player_id"
        ]
    ].to_string(index=False)
)

print("\nCLEANER OTHER-EXTRAS DISTRIBUTION")

display(
    other_df.groupby(
        ["extras1_runs", "extras_2_runs", "is_wicket"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
)

WICKET DETAILS FOR OTHER EXTRAS


' match_id  innings  overid  over_ball_num  innings_ball_num         striker_name     bowler_name  runs  extras1_runs  extras_2_runs dismissal_type  dismissed_player_id\n    14102        2      19              9               122       Kieron Pollard Prasidh Krishna     0             1              0         RunOut               5034.0\n    14145        1      19              3               114 Namboori Tilak Varma Lockie Ferguson     0             1              0         RunOut               7897.0\n    48015        2      10              1                57         Hammad Mirza     Saad Masood     0             1              0         RunOut              27137.0'


CLEANER OTHER-EXTRAS DISTRIBUTION


,extras1_runs,extras_2_runs,is_wicket,delivery_count
0,1,0,False,247
2,1,1,False,7
3,1,4,False,7
1,1,0,True,3


In [11]:
# STEP 25: COMPLETE EXTRAS CATEGORY AUDIT

extras_audit = (
    ball_df.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        wickets=("is_wicket", "sum")
    )
    .reset_index()
)

extras_audit["total_extras"] = (
    extras_audit["extras1_runs"]
    + extras_audit["extras2_runs"]
)

extras_audit["total_delivery_runs"] = (
    extras_audit["batter_runs"]
    + extras_audit["total_extras"]
)

extras_audit["delivery_percentage"] = (
    extras_audit["deliveries"] / len(ball_df) * 100
)

display(
    extras_audit.sort_values(
        "deliveries",
        ascending=False
    ).round(2)
)

print("\nTOTAL DELIVERY RUNS:")
print(extras_audit["total_delivery_runs"].sum())

print("\nTOTAL RECORDED DELIVERIES:")
print(extras_audit["deliveries"].sum())

,extras_type,deliveries,batter_runs,extras1_runs,extras2_runs,wickets,total_extras,total_delivery_runs,delivery_percentage
6,NaN,189167,268285,1,1,10832,2,268287,94.11
5,wide,7714,0,9174,0,40,9174,9174,3.84
3,legbyes,2748,1,3603,0,0,3603,3604,1.37
0,Overstep,576,852,576,48,5,624,1476,0.29
2,byes,529,0,1000,0,1,1000,1000,0.26
4,other,264,412,264,35,3,299,711,0.13
1,Penalty,1,0,5,0,0,5,5,0.00



TOTAL DELIVERY RUNS:
284257

TOTAL RECORDED DELIVERIES:
200999


In [12]:
# STEP 26: RECONCILE WICKET COUNTS

print("TOTAL WICKETS FROM is_wicket:")
print(ball_df["is_wicket"].sum())

print("\nTOTAL WICKETS BY EXTRAS CATEGORY:")
display(
    ball_df[ball_df["is_wicket"] == True]
    .groupby("extras_type", dropna=False)
    .size()
    .to_frame("wicket_count")
)

print("\nTOTAL WICKETS BY DISMISSAL TYPE:")
display(
    ball_df[ball_df["is_wicket"] == True]
    .groupby("dismissal_type", dropna=False)
    .size()
    .to_frame("wicket_count")
)

print("\nWICKET RECORDS WITH MISSING DISMISSAL TYPE:")
display(
    ball_df[
        (ball_df["is_wicket"] == True)
        & (ball_df["dismissal_type"].isna())
    ][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "extras_type",
            "dismissed_player_id"
        ]
    ]
)

TOTAL WICKETS FROM is_wicket:
10881

TOTAL WICKETS BY EXTRAS CATEGORY:


,wicket_count
extras_type,
Overstep,5
byes,1
other,3
wide,40
NaN,10832



TOTAL WICKETS BY DISMISSAL TYPE:


,wicket_count
dismissal_type,
Bowled,1775
Caught,7557
Hit Wicket,17
LBW,709
Obstracting the field,2
Obstructing the field,1
Retire Out,6
RunOut,600
Stump,214



WICKET RECORDS WITH MISSING DISMISSAL TYPE:


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,extras_type,dismissed_player_id


In [13]:
# STEP 27: OVER NUMBERING VALIDATION

over_summary = (
    ball_df.groupby(["match_id", "innings"])
    .agg(
        total_deliveries=("innings_ball_num", "size"),
        unique_overs=("overid", "nunique"),
        minimum_over=("overid", "min"),
        maximum_over=("overid", "max"),
        minimum_ball=("over_ball_num", "min"),
        maximum_ball=("over_ball_num", "max")
    )
)

print("OVER NUMBERING SUMMARY")
display(over_summary.describe().round(2))

print("\nOVER NUMBERING RANGE")
print("Minimum overid:", ball_df["overid"].min())
print("Maximum overid:", ball_df["overid"].max())

print("\nOVER-BALL NUMBER RANGE")
print("Minimum over_ball_num:", ball_df["over_ball_num"].min())
print("Maximum over_ball_num:", ball_df["over_ball_num"].max())

print("\nEXAMPLE DELIVERIES WHERE over_ball_num > 6")

display(
    ball_df[ball_df["over_ball_num"] > 6][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "extras_type",
            "runs"
        ]
    ].head(15)
)

OVER NUMBERING SUMMARY


,total_deliveries,unique_overs,minimum_over,maximum_over,minimum_ball,maximum_ball
count,1724.00,1724.00,1724.0,1724.00,1724.0,1724.00
mean,116.59,18.76,1.0,18.76,1.0,7.66
std,19.70,3.01,0.0,3.01,0.0,0.97
min,1.00,1.00,1.0,1.00,1.0,1.00
25%,117.00,19.00,1.0,19.00,1.0,7.00
50%,123.00,20.00,1.0,20.00,1.0,8.00
75%,126.00,20.00,1.0,20.00,1.0,8.00
max,136.00,20.00,1.0,20.00,1.0,13.00



OVER NUMBERING RANGE
Minimum overid: 1
Maximum overid: 20

OVER-BALL NUMBER RANGE
Minimum over_ball_num: 1
Maximum over_ball_num: 13

EXAMPLE DELIVERIES WHERE over_ball_num > 6


,match_id,innings,overid,over_ball_num,innings_ball_num,extras_type,runs
42,13511,1,7,7,43,NaN,0
49,13511,1,8,7,50,NaN,0
50,13511,1,8,8,51,NaN,0
284,13512,1,10,7,61,NaN,1
285,13512,1,10,8,62,NaN,0
298,13512,1,12,7,75,NaN,0
311,13512,1,14,7,88,NaN,1
324,13512,1,16,7,101,NaN,1
331,13512,1,17,7,108,NaN,4
338,13512,1,18,7,115,NaN,4


In [14]:
# STEP 28: ANALYZE DELIVERIES AFTER THE SIXTH BALL

extra_deliveries = ball_df[
    ball_df["over_ball_num"] > 6
].copy()

print("DELIVERIES WITH over_ball_num > 6")
print("Total:", len(extra_deliveries))

print("\nDISTRIBUTION BY EXTRAS CATEGORY")

display(
    extra_deliveries.groupby(
        "extras_type",
        dropna=False
    )
    .agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        wickets=("is_wicket", "sum")
    )
    .sort_values("deliveries", ascending=False)
)

print("\nDELIVERY NUMBER DISTRIBUTION BY EXTRAS CATEGORY")

display(
    pd.crosstab(
        extra_deliveries["over_ball_num"],
        extra_deliveries["extras_type"],
        dropna=False
    )
)

print("\nSAMPLES WITH OVER_BALL_NUM > 6 AND MISSING EXTRAS CATEGORY")

display(
    extra_deliveries[
        extra_deliveries["extras_type"].isna()
    ][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(20)
)

DELIVERIES WITH over_ball_num > 6
Total: 8429

DISTRIBUTION BY EXTRAS CATEGORY


,deliveries,batter_runs,extras1_runs,extras2_runs,wickets
extras_type,,,,,
NaN,7735,11294,0,0,480
wide,473,0,560,0,3
legbyes,142,0,197,0,0
byes,37,0,57,0,0
Overstep,22,39,22,0,0
other,20,45,20,0,1



DELIVERY NUMBER DISTRIBUTION BY EXTRAS CATEGORY


extras_type,Overstep,byes,legbyes,other,wide,NaN
over_ball_num,,,,,,
7,15,31,116,14,370,6164
8,4,5,23,5,74,1253
9,3,1,3,1,24,247
10,0,0,0,0,3,54
11,0,0,0,0,0,15
12,0,0,0,0,2,0
13,0,0,0,0,0,2



SAMPLES WITH OVER_BALL_NUM > 6 AND MISSING EXTRAS CATEGORY


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,is_wicket,dismissal_type
42,13511,1,7,7,43,Sam Billings,Akeal Hosein,0,True,Stump
49,13511,1,8,7,50,Liam Dawson,Romario Shepherd,0,False,NaN
50,13511,1,8,8,51,Liam Dawson,Romario Shepherd,0,False,NaN
284,13512,1,10,7,61,Jason Roy,Kieron Pollard,1,False,NaN
285,13512,1,10,8,62,Moeen Ali,Kieron Pollard,0,False,NaN
298,13512,1,12,7,75,Moeen Ali,Jason Holder,0,False,NaN
311,13512,1,14,7,88,Eoin Morgan,Kieron Pollard,1,False,NaN
324,13512,1,16,7,101,Chris Jordan,Kieron Pollard,1,False,NaN
331,13512,1,17,7,108,Eoin Morgan,Romario Shepherd,4,False,NaN
338,13512,1,18,7,115,Liam Dawson,Jason Holder,4,False,NaN


In [15]:
# STEP 29: INSPECT A COMPLETE OVER

sample_over = ball_df[
    (ball_df["match_id"] == 13512) &
    (ball_df["innings"] == 1) &
    (ball_df["overid"] == 10)
].copy()

display(
    sample_over[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "is_four",
            "is_six",
            "is_wicket",
            "dismissal_type",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].to_string(index=False)
)

' match_id  innings  overid  over_ball_num  innings_ball_num striker_name    bowler_name  runs  is_four  is_six  is_wicket dismissal_type extras_type  extras1_runs  extras_2_runs\n    13512        1      10              1                55    Moeen Ali Kieron Pollard     0    False   False      False            NaN        wide             1              0\n    13512        1      10              2                56    Moeen Ali Kieron Pollard     0    False   False      False            NaN         NaN             0              0\n    13512        1      10              3                57    Moeen Ali Kieron Pollard     2    False   False      False            NaN         NaN             0              0\n    13512        1      10              4                58    Moeen Ali Kieron Pollard     0    False   False      False            NaN        wide             1              0\n    13512        1      10              5                59    Moeen Ali Kieron Pollard     2    False  

In [16]:
# STEP 30: AUDIT CANDIDATE ILLEGAL DELIVERIES

candidate_illegal = ball_df[
    ball_df["extras_type"].isin(["wide", "Overstep"])
].copy()

display(
    candidate_illegal.groupby("extras_type").agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        wickets=("is_wicket", "sum")
    )
)

print("WICKET DISMISSAL TYPES IN THESE CATEGORIES")

display(
    candidate_illegal[
        candidate_illegal["is_wicket"] == True
    ][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "dismissal_type"
        ]
    ].head(20)
)

,deliveries,batter_runs,extras1_runs,extras2_runs,wickets
extras_type,,,,,
Overstep,576,852,576,48,5
wide,7714,0,9174,0,40


WICKET DISMISSAL TYPES IN THESE CATEGORIES


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,dismissal_type
3241,14094,1,8,6,Robin Uthappa,Varun Chakravarthy,0,wide,Stump
20026,14164,1,20,6,Romario Shepherd,Nathan Ellis,0,Overstep,RunOut
20247,15187,1,20,6,Jos Buttler,Yash Dayal,1,Overstep,RunOut
20248,15187,1,20,7,Ravichandran Ashwin,Yash Dayal,0,wide,RunOut
23402,15609,1,10,1,Milton Shumba,Mohammad Nabi,0,wide,Stump
23716,15610,1,20,1,Najibullah Zadran,Tendai Chatara,0,wide,RunOut
24021,15611,2,8,3,Sikandar Raza,Sharafuddin Ashraf,0,wide,Stump
35456,16866,1,19,5,Rizwan CP,Ehsan Khan,0,wide,Stump
35701,16867,1,20,6,Adwitya Bhargava,Mohamed Shafeeq,0,wide,RunOut
44693,19959,1,11,6,Bjorn Fortuin,Odean Smith,0,Overstep,RunOut


In [17]:
# STEP 31: INVESTIGATE THE OTHER EXTRAS CATEGORY

other_deliveries = ball_df[
    ball_df["extras_type"] == "other"
].copy()

print("TOTAL OTHER DELIVERIES:", len(other_deliveries))

print("\nDISTRIBUTION OF OVER BALL NUMBERS")

display(
    other_deliveries["over_ball_num"]
    .value_counts()
    .sort_index()
)

print("\nOTHER CATEGORY SUMMARY")

display(
    other_deliveries.groupby(
        ["runs", "extras1_runs", "extras_2_runs", "is_wicket"],
        dropna=False
    )
    .size()
    .reset_index(name="deliveries")
    .sort_values("deliveries", ascending=False)
)

print("\nSAMPLE OF OTHER DELIVERIES")

display(
    other_deliveries[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(25)
)

TOTAL OTHER DELIVERIES: 264

DISTRIBUTION OF OVER BALL NUMBERS


over_ball_num
1    44
2    35
3    33
4    44
5    47
6    41
7    14
8     5
9     1
Name: count, dtype: int64


OTHER CATEGORY SUMMARY


,runs,extras1_runs,extras_2_runs,is_wicket,deliveries
0,0,1,0,False,111
4,1,1,0,False,49
7,4,1,0,False,42
8,6,1,0,False,26
5,2,1,0,False,18
2,0,1,1,False,7
3,0,1,4,False,7
1,0,1,0,True,3
6,3,1,0,False,1



SAMPLE OF OTHER DELIVERIES


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,1,10,1,57,Rovman Powell,Tymal Mills,0,1,0,False,NaN
1079,13515,1,16,6,99,Rovman Powell,Saqib Mahmood,0,1,0,False,NaN
1226,13515,2,19,5,119,Sam Billings,Sheldon Cottrell,2,1,0,False,NaN
2588,13614,1,18,5,111,Dasun Shanaka,Mitchell Starc,0,1,4,False,NaN
2609,13614,2,2,1,7,Ashton Agar,Nuwan Thushara,0,1,0,False,NaN
3309,14094,1,19,7,119,Mahendra Singh Dhoni,Shivam Mavi,6,1,0,False,NaN
4413,14098,2,20,1,119,Bhuvneshwar Kumar,Nathan Coulter Nile,0,1,0,False,NaN
5012,14101,1,17,4,105,Kagiso Rabada,Tim Southee,0,1,0,False,NaN
5365,14102,2,19,9,122,Kieron Pollard,Prasidh Krishna,0,1,0,True,RunOut
6575,14107,2,16,5,95,Patrick Cummins,Daniel Sams,2,1,0,False,NaN


In [18]:
# STEP 32: INSPECT PENALTY DELIVERY

penalty_deliveries = ball_df[
    ball_df["extras_type"] == "Penalty"
].copy()

display(
    penalty_deliveries[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "is_wicket",
            "dismissal_type",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].to_string(index=False)
)

print("Number of penalty records:", len(penalty_deliveries))

' match_id  innings  overid  over_ball_num  innings_ball_num striker_name  bowler_name  runs  is_wicket dismissal_type extras_type  extras1_runs  extras_2_runs\n    16753        2       6              4                34 Harry Tector Fareed Malik     0      False            NaN     Penalty             5              0'

Number of penalty records: 1


In [19]:
# STEP 33: DELIVERY CLASSIFICATION AUDIT

classification_audit = pd.DataFrame([
    {
        "extras_type": "wide",
        "count": 7714,
        "candidate_legal_ball": "No",
        "status": "Observed; standard cricket scoring interpretation"
    },
    {
        "extras_type": "Overstep",
        "count": 576,
        "candidate_legal_ball": "Likely No",
        "status": "Requires confirmation of dataset terminology"
    },
    {
        "extras_type": "legbyes",
        "count": 2748,
        "candidate_legal_ball": "Yes",
        "status": "Standard cricket scoring interpretation"
    },
    {
        "extras_type": "byes",
        "count": 529,
        "candidate_legal_ball": "Yes",
        "status": "Standard cricket scoring interpretation"
    },
    {
        "extras_type": "other",
        "count": 264,
        "candidate_legal_ball": "Unresolved",
        "status": "Heterogeneous category; investigate further"
    },
    {
        "extras_type": "Penalty",
        "count": 1,
        "candidate_legal_ball": "Unresolved",
        "status": "Single record; scoring context unavailable"
    },
    {
        "extras_type": "Missing",
        "count": int(ball_df["extras_type"].isna().sum()),
        "candidate_legal_ball": "Likely Yes",
        "status": "Ordinary deliveries observed; validate globally"
    }
])

display(classification_audit)

,extras_type,count,candidate_legal_ball,status
0,wide,7714,No,Observed; standard cricket scoring interpretation
1,Overstep,576,Likely No,Requires confirmation of dataset terminology
2,legbyes,2748,Yes,Standard cricket scoring interpretation
3,byes,529,Yes,Standard cricket scoring interpretation
4,other,264,Unresolved,Heterogeneous category; investigate further
5,Penalty,1,Unresolved,Single record; scoring context unavailable
6,Missing,189167,Likely Yes,Ordinary deliveries observed; validate globally


In [20]:
# STEP 34: OVER-LEVEL DELIVERY SEQUENCE AUDIT

over_audit = (
    ball_df.groupby(["match_id", "innings", "overid"])
    .agg(
        recorded_deliveries=("over_ball_num", "size"),
        unique_delivery_numbers=("over_ball_num", "nunique"),
        min_delivery_number=("over_ball_num", "min"),
        max_delivery_number=("over_ball_num", "max"),
        wide_count=("extras_type", lambda x: (x == "wide").sum()),
        overstep_count=("extras_type", lambda x: (x == "Overstep").sum()),
        other_count=("extras_type", lambda x: (x == "other").sum())
    )
    .reset_index()
)

print("TOTAL OVERS:", len(over_audit))

print("\nOVERS WITH MORE THAN 6 RECORDED DELIVERIES:")
display(
    over_audit[
        over_audit["recorded_deliveries"] > 6
    ].head(20)
)

print("\nOVERS WITH DUPLICATE DELIVERY NUMBERS:")
display(
    over_audit[
        over_audit["unique_delivery_numbers"]
        != over_audit["recorded_deliveries"]
    ].head(20)
)

TOTAL OVERS: 32350

OVERS WITH MORE THAN 6 RECORDED DELIVERIES:


,match_id,innings,overid,recorded_deliveries,unique_delivery_numbers,min_delivery_number,max_delivery_number,wide_count,overstep_count,other_count
6,13511,1,7,7,7,1,7,1,0,0
7,13511,1,8,8,8,1,8,2,0,0
47,13512,1,10,8,8,1,8,2,0,0
49,13512,1,12,7,7,1,7,1,0,0
51,13512,1,14,7,7,1,7,1,0,0
53,13512,1,16,7,7,1,7,1,0,0
54,13512,1,17,7,7,1,7,1,0,0
55,13512,1,18,7,7,1,7,1,0,0
56,13512,1,19,7,7,1,7,1,0,0
58,13512,2,1,7,7,1,7,1,0,0



OVERS WITH DUPLICATE DELIVERY NUMBERS:


,match_id,innings,overid,recorded_deliveries,unique_delivery_numbers,min_delivery_number,max_delivery_number,wide_count,overstep_count,other_count


In [21]:
# STEP 35: CHECK CONTINUITY OF OVER DELIVERY NUMBERS

def check_over_sequence(group):
    actual = sorted(group["over_ball_num"].dropna().astype(int).tolist())
    expected = list(range(1, max(actual) + 1)) if actual else []

    return pd.Series({
        "starts_at_1": bool(actual and actual[0] == 1),
        "continuous_sequence": actual == expected,
        "actual_sequence": actual
    })


sequence_audit = (
    ball_df.groupby(["match_id", "innings", "overid"])
    .apply(check_over_sequence, include_groups=False)
    .reset_index()
)

print("TOTAL OVERS:", len(sequence_audit))

print("\nOVERS NOT STARTING AT 1:")
display(sequence_audit[~sequence_audit["starts_at_1"]].head(20))

print("\nOVERS WITH NON-CONTINUOUS SEQUENCES:")
display(
    sequence_audit[
        ~sequence_audit["continuous_sequence"]
    ].head(20)
)

print("\nTOTAL INVALID OVER SEQUENCES:")
print(
    len(sequence_audit[
        (~sequence_audit["starts_at_1"]) |
        (~sequence_audit["continuous_sequence"])
    ])
)

TOTAL OVERS: 32350

OVERS NOT STARTING AT 1:


,match_id,innings,overid,starts_at_1,continuous_sequence,actual_sequence



OVERS WITH NON-CONTINUOUS SEQUENCES:


,match_id,innings,overid,starts_at_1,continuous_sequence,actual_sequence



TOTAL INVALID OVER SEQUENCES:
0


In [22]:
# STEP 37: INNINGS-LEVEL DELIVERY SUMMARY

innings_audit = (
    ball_df.groupby(["match_id", "innings"])
    .agg(
        total_recorded_deliveries=("innings_ball_num", "size"),
        unique_delivery_numbers=("innings_ball_num", "nunique"),
        first_delivery_number=("innings_ball_num", "min"),
        last_delivery_number=("innings_ball_num", "max"),
        total_overs_recorded=("overid", "nunique"),
        maximum_over_number=("overid", "max"),
        total_wickets=("is_wicket", "sum"),
        total_batter_runs=("runs", "sum")
    )
    .reset_index()
)

print("TOTAL INNINGS:", len(innings_audit))

print("\nINNINGS WITH MORE THAN 120 RECORDED DELIVERIES:")
display(
    innings_audit[
        innings_audit["total_recorded_deliveries"] > 120
    ].head(20)
)

print("\nINNINGS SUMMARY:")
display(
    innings_audit[
        [
            "total_recorded_deliveries",
            "total_overs_recorded",
            "total_wickets"
        ]
    ].describe()
)

print("\nINNINGS WITH DUPLICATE DELIVERY NUMBERS:")
display(
    innings_audit[
        innings_audit["unique_delivery_numbers"]
        != innings_audit["total_recorded_deliveries"]
    ].head(20)
)

TOTAL INNINGS: 1724

INNINGS WITH MORE THAN 120 RECORDED DELIVERIES:


,match_id,innings,total_recorded_deliveries,unique_delivery_numbers,first_delivery_number,last_delivery_number,total_overs_recorded,maximum_over_number,total_wickets,total_batter_runs
0,13511,1,121,121,1,121,20,20,10,100
2,13512,1,128,128,1,128,20,20,8,163
3,13512,2,124,124,1,124,20,20,8,158
4,13513,1,125,125,1,125,20,20,5,211
5,13513,2,127,127,1,127,20,20,9,192
6,13514,1,128,128,1,128,20,20,6,182
7,13514,2,125,125,1,125,20,20,5,150
8,13515,1,127,127,1,127,20,20,4,168
9,13515,2,127,127,1,127,20,20,10,150
10,13522,1,131,131,1,131,20,20,7,140



INNINGS SUMMARY:


,total_recorded_deliveries,total_overs_recorded,total_wickets
count,1724.000000,1724.000000,1724.000000
mean,116.588747,18.764501,6.311485
std,19.697000,3.012875,2.580535
min,1.000000,1.000000,0.000000
25%,117.000000,19.000000,4.000000
50%,123.000000,20.000000,6.000000
75%,126.000000,20.000000,8.000000
max,136.000000,20.000000,10.000000



INNINGS WITH DUPLICATE DELIVERY NUMBERS:


,match_id,innings,total_recorded_deliveries,unique_delivery_numbers,first_delivery_number,last_delivery_number,total_overs_recorded,maximum_over_number,total_wickets,total_batter_runs


In [23]:
# STEP 38: VALIDATE INNINGS DELIVERY SEQUENCE

innings_sequence_audit = (
    ball_df.groupby(["match_id", "innings"])
    .agg(
        actual_count=("innings_ball_num", "size"),
        unique_count=("innings_ball_num", "nunique"),
        first_ball=("innings_ball_num", "min"),
        last_ball=("innings_ball_num", "max")
    )
    .reset_index()
)

innings_sequence_audit["starts_at_1"] = (
    innings_sequence_audit["first_ball"] == 1
)

innings_sequence_audit["continuous_sequence"] = (
    innings_sequence_audit["actual_count"]
    == innings_sequence_audit["last_ball"]
)

print("TOTAL INNINGS:", len(innings_sequence_audit))

print("\nINNINGS NOT STARTING AT 1:")
display(
    innings_sequence_audit[
        ~innings_sequence_audit["starts_at_1"]
    ]
)

print("\nINNINGS WITH NON-CONTINUOUS SEQUENCES:")
display(
    innings_sequence_audit[
        ~innings_sequence_audit["continuous_sequence"]
    ]
)

print("\nTOTAL INVALID INNINGS:")
print(
    len(
        innings_sequence_audit[
            (~innings_sequence_audit["starts_at_1"])
            | (~innings_sequence_audit["continuous_sequence"])
        ]
    )
)

TOTAL INNINGS: 1724

INNINGS NOT STARTING AT 1:


,match_id,innings,actual_count,unique_count,first_ball,last_ball,starts_at_1,continuous_sequence



INNINGS WITH NON-CONTINUOUS SEQUENCES:


,match_id,innings,actual_count,unique_count,first_ball,last_ball,starts_at_1,continuous_sequence



TOTAL INVALID INNINGS:
0


In [24]:
# STEP 39: INSPECT OVERSTEP DELIVERIES

overstep_df = ball_df[
    ball_df["extras_type"] == "Overstep"
].copy()

print("TOTAL OVERSTEP DELIVERIES:", len(overstep_df))

print("\nOVERSTEP DISTRIBUTION:")
display(
    overstep_df[
        [
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].value_counts(dropna=False).reset_index(name="count")
)

print("\nSAMPLE OVERSTEP RECORDS:")
display(
    overstep_df[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(15)
)

TOTAL OVERSTEP DELIVERIES: 576

OVERSTEP DISTRIBUTION:


,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type,count
0,0,1,0,False,NaN,215
1,1,1,0,False,NaN,173
2,4,1,0,False,NaN,81
3,6,1,0,False,NaN,47
4,2,1,0,False,NaN,32
5,0,1,1,False,NaN,12
6,0,1,4,False,NaN,9
7,0,1,0,True,RunOut,3
8,3,1,0,False,NaN,2
9,1,1,0,True,RunOut,1



SAMPLE OVERSTEP RECORDS:


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
495,13513,1,4,2,20,Nicholas Pooran,George Garton,0,1,0,False,NaN
1229,13515,2,20,1,122,Sam Billings,Jason Holder,1,1,0,False,NaN
1485,13523,1,1,4,4,Ishan Kishan,Akeal Hosein,0,1,4,False,NaN
1486,13523,1,1,5,5,Ishan Kishan,Akeal Hosein,1,1,0,False,NaN
3191,14094,1,1,1,1,Ruturaj Gaikwad,Umesh Yadav,0,1,0,False,NaN
4169,14098,1,1,5,5,Jos Buttler,Bhuvneshwar Kumar,0,1,0,False,NaN
4181,14098,1,3,4,17,Yashasvi Jaiswal,Bhuvneshwar Kumar,0,1,0,False,NaN
4187,14098,1,4,3,23,Jos Buttler,Umran Malik,4,1,0,False,NaN
4192,14098,1,5,1,28,Yashasvi Jaiswal,Washington Sundar,1,1,0,False,NaN
4452,14099,1,6,3,33,Nitish Rana,Akash Deep,0,1,0,False,NaN


In [25]:
# STEP 40: INSPECT OTHER EXTRAS CATEGORY

other_df = ball_df[
    ball_df["extras_type"] == "other"
].copy()

print("TOTAL OTHER DELIVERIES:", len(other_df))

print("\nOTHER CATEGORY DISTRIBUTION:")
display(
    other_df[
        [
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].value_counts(dropna=False).reset_index(name="count")
)

print("\nOVER-BALL NUMBER DISTRIBUTION:")
display(
    other_df["over_ball_num"]
    .value_counts()
    .sort_index()
    .rename_axis("over_ball_num")
    .reset_index(name="count")
)

print("\nSAMPLE OTHER RECORDS:")
display(
    other_df[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ].head(20)
)

TOTAL OTHER DELIVERIES: 264

OTHER CATEGORY DISTRIBUTION:


,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type,count
0,0,1,0,False,NaN,111
1,1,1,0,False,NaN,49
2,4,1,0,False,NaN,42
3,6,1,0,False,NaN,26
4,2,1,0,False,NaN,18
5,0,1,1,False,NaN,7
6,0,1,4,False,NaN,7
7,0,1,0,True,RunOut,3
8,3,1,0,False,NaN,1



OVER-BALL NUMBER DISTRIBUTION:


,over_ball_num,count
0,1,44
1,2,35
2,3,33
3,4,44
4,5,47
5,6,41
6,7,14
7,8,5
8,9,1



SAMPLE OTHER RECORDS:


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,1,10,1,57,Rovman Powell,Tymal Mills,0,1,0,False,NaN
1079,13515,1,16,6,99,Rovman Powell,Saqib Mahmood,0,1,0,False,NaN
1226,13515,2,19,5,119,Sam Billings,Sheldon Cottrell,2,1,0,False,NaN
2588,13614,1,18,5,111,Dasun Shanaka,Mitchell Starc,0,1,4,False,NaN
2609,13614,2,2,1,7,Ashton Agar,Nuwan Thushara,0,1,0,False,NaN
3309,14094,1,19,7,119,Mahendra Singh Dhoni,Shivam Mavi,6,1,0,False,NaN
4413,14098,2,20,1,119,Bhuvneshwar Kumar,Nathan Coulter Nile,0,1,0,False,NaN
5012,14101,1,17,4,105,Kagiso Rabada,Tim Southee,0,1,0,False,NaN
5365,14102,2,19,9,122,Kieron Pollard,Prasidh Krishna,0,1,0,True,RunOut
6575,14107,2,16,5,95,Patrick Cummins,Daniel Sams,2,1,0,False,NaN


In [26]:
# STEP 41: CHECK EXTRAS CATEGORY BY COMPETITION AND SEASON

category_df = ball_df[
    ball_df["extras_type"].isin(["Overstep", "other"])
].copy()

category_df = category_df.merge(
    matches_df[["match_id", "comp_id"]],
    on="match_id",
    how="left"
)

category_df = category_df.merge(
    competitions_df[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left"
)

category_summary = (
    category_df.groupby(
        ["extras_type", "match_format", "season"],
        dropna=False
    )
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
    .reset_index()
    .sort_values(["extras_type", "season"])
)

display(category_summary)

print("\nTOTALS BY CATEGORY:")
display(
    category_df.groupby("extras_type")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        competitions=("comp_id", "nunique")
    )
)

,extras_type,match_format,season,deliveries,matches,competitions
0,Overstep,T20,2022,47,34,1
5,Overstep,T20I,2022,61,36,21
6,Overstep,T20I,2022/23,6,4,3
1,Overstep,T20,2023,64,36,1
7,Overstep,T20I,2023,46,30,14
2,Overstep,T20,2024,65,42,2
8,Overstep,T20I,2024,65,42,27
9,Overstep,T20I,2024/25,6,3,1
3,Overstep,T20,2025,53,34,2
10,Overstep,T20I,2025,78,55,23



TOTALS BY CATEGORY:


,deliveries,matches,competitions
extras_type,,,
Overstep,576,378,114
other,264,215,83


In [27]:
# STEP 42: CHECK MATCHES CONTAINING BOTH CATEGORIES

category_match_counts = (
    category_df.groupby("match_id")["extras_type"]
    .agg(
        category_count="nunique",
        categories=lambda x: ", ".join(sorted(x.unique()))
    )
    .reset_index()
)

both_categories_matches = category_match_counts[
    category_match_counts["category_count"] == 2
]

print("MATCHES WITH BOTH OVERSTEP AND OTHER:",
      len(both_categories_matches))

print("\nSAMPLE MATCHES:")
display(both_categories_matches.head(20))

print("\nMATCHES WITH ONLY ONE CATEGORY:")
display(
    category_match_counts["categories"]
    .value_counts()
    .rename_axis("categories")
    .reset_index(name="matches")
)

MATCHES WITH BOTH OVERSTEP AND OTHER: 107

SAMPLE MATCHES:


,match_id,category_count,categories
0,13513,2,"Overstep, other"
1,13515,2,"Overstep, other"
4,14094,2,"Overstep, other"
5,14098,2,"Overstep, other"
7,14101,2,"Overstep, other"
8,14102,2,"Overstep, other"
19,14115,2,"Overstep, other"
22,14122,2,"Overstep, other"
29,14134,2,"Overstep, other"
34,14141,2,"Overstep, other"



MATCHES WITH ONLY ONE CATEGORY:


,categories,matches
0,Overstep,271
1,other,108
2,"Overstep, other",107


In [28]:
# STEP 43: CHECK BOTH CATEGORIES WITHIN THE SAME OVER

over_category_audit = (
    category_df.groupby(["match_id", "innings", "overid"])
    .agg(
        category_count=("extras_type", "nunique"),
        categories=(
            "extras_type",
            lambda x: ", ".join(sorted(x.unique()))
        ),
        total_deliveries=("extras_type", "size")
    )
    .reset_index()
)

both_categories_overs = over_category_audit[
    over_category_audit["category_count"] == 2
]

print("TOTAL OVERS WITH BOTH CATEGORIES:",
      len(both_categories_overs))

print("\nSAMPLE OVERS CONTAINING BOTH:")
display(both_categories_overs.head(20))

print("\nCATEGORY COMBINATIONS BY OVER:")
display(
    over_category_audit["categories"]
    .value_counts()
    .rename_axis("categories")
    .reset_index(name="overs")
)

TOTAL OVERS WITH BOTH CATEGORIES: 8

SAMPLE OVERS CONTAINING BOTH:


,match_id,innings,overid,category_count,categories,total_deliveries
122,16656,1,3,2,"Overstep, other",2
179,20184,2,13,2,"Overstep, other",2
276,21585,1,17,2,"Overstep, other",2
403,31709,1,12,2,"Overstep, other",2
595,42743,2,2,2,"Overstep, other",2
606,43171,1,20,2,"Overstep, other",2
711,50188,2,17,2,"Overstep, other",2
771,50942,1,15,2,"Overstep, other",2



CATEGORY COMBINATIONS BY OVER:


,categories,overs
0,Overstep,540
1,other,246
2,"Overstep, other",8


In [29]:
# STEP 44: INSPECT ALL EIGHT MIXED OVERS

mixed_over_keys = both_categories_overs[
    ["match_id", "innings", "overid"]
]

mixed_over_details = (
    ball_df.merge(
        mixed_over_keys,
        on=["match_id", "innings", "overid"],
        how="inner"
    )
    .sort_values(
        ["match_id", "innings", "overid", "over_ball_num"]
    )
)

display(
    mixed_over_details[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "extras_type",
            "is_wicket",
            "dismissal_type"
        ]
    ].to_string(index=False)
)

' match_id  innings  overid  over_ball_num  innings_ball_num    striker_name         bowler_name  runs  extras1_runs  extras_2_runs extras_type  is_wicket dismissal_type\n    16656        1       3              1                14    Rohit Sharma       Haroon Arshad     6             0              0         NaN      False            NaN\n    16656        1       3              2                15    Rohit Sharma       Haroon Arshad     0             0              0         NaN      False            NaN\n    16656        1       3              3                16    Rohit Sharma       Haroon Arshad     1             0              0         NaN      False            NaN\n    16656        1       3              4                17    Lokesh Rahul       Haroon Arshad     2             1              0    Overstep      False            NaN\n    16656        1       3              5                18    Lokesh Rahul       Haroon Arshad     0             1              0       other      F

In [30]:
# STEP 45: INSPECT PENALTY DELIVERY

penalty_df = ball_df[
    ball_df["extras_type"] == "Penalty"
].copy()

print("TOTAL PENALTY DELIVERIES:", len(penalty_df))

display(
    penalty_df[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "extras_type",
            "is_wicket",
            "dismissal_type"
        ]
    ]
)

# Inspect the complete over containing the penalty record
if not penalty_df.empty:
    penalty_match = penalty_df.iloc[0]["match_id"]
    penalty_innings = penalty_df.iloc[0]["innings"]
    penalty_over = penalty_df.iloc[0]["overid"]

    print("\nCOMPLETE OVER:")
    display(
        ball_df[
            (ball_df["match_id"] == penalty_match)
            & (ball_df["innings"] == penalty_innings)
            & (ball_df["overid"] == penalty_over)
        ][
            [
                "over_ball_num",
                "innings_ball_num",
                "striker_name",
                "bowler_name",
                "runs",
                "extras1_runs",
                "extras_2_runs",
                "extras_type",
                "is_wicket",
                "dismissal_type"
            ]
        ]
    )

TOTAL PENALTY DELIVERIES: 1


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_wicket,dismissal_type
34203,16753,2,6,4,34,Harry Tector,Fareed Malik,0,5,0,Penalty,False,NaN



COMPLETE OVER:


,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_wicket,dismissal_type
34200,1,31,Harry Tector,Fareed Malik,1,0,0,NaN,False,NaN
34201,2,32,George Dockrell,Fareed Malik,2,0,0,NaN,False,NaN
34202,3,33,George Dockrell,Fareed Malik,1,0,0,NaN,False,NaN
34203,4,34,Harry Tector,Fareed Malik,0,5,0,Penalty,False,NaN
34204,5,35,Harry Tector,Fareed Malik,4,0,0,NaN,False,NaN
34205,6,36,Harry Tector,Fareed Malik,0,0,0,NaN,True,Caught


In [31]:
# STEP 45: INSPECT PENALTY DELIVERY

penalty_df = ball_df[
    ball_df["extras_type"] == "Penalty"
].copy()

print("TOTAL PENALTY DELIVERIES:", len(penalty_df))

display(
    penalty_df[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "extras_type",
            "is_wicket",
            "dismissal_type"
        ]
    ]
)

# Inspect the complete over containing the penalty record
if not penalty_df.empty:
    penalty_match = penalty_df.iloc[0]["match_id"]
    penalty_innings = penalty_df.iloc[0]["innings"]
    penalty_over = penalty_df.iloc[0]["overid"]

    print("\nCOMPLETE OVER:")
    display(
        ball_df[
            (ball_df["match_id"] == penalty_match)
            & (ball_df["innings"] == penalty_innings)
            & (ball_df["overid"] == penalty_over)
        ][
            [
                "over_ball_num",
                "innings_ball_num",
                "striker_name",
                "bowler_name",
                "runs",
                "extras1_runs",
                "extras_2_runs",
                "extras_type",
                "is_wicket",
                "dismissal_type"
            ]
        ]
    )

TOTAL PENALTY DELIVERIES: 1


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_wicket,dismissal_type
34203,16753,2,6,4,34,Harry Tector,Fareed Malik,0,5,0,Penalty,False,NaN



COMPLETE OVER:


,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,extras_type,is_wicket,dismissal_type
34200,1,31,Harry Tector,Fareed Malik,1,0,0,NaN,False,NaN
34201,2,32,George Dockrell,Fareed Malik,2,0,0,NaN,False,NaN
34202,3,33,George Dockrell,Fareed Malik,1,0,0,NaN,False,NaN
34203,4,34,Harry Tector,Fareed Malik,0,5,0,Penalty,False,NaN
34204,5,35,Harry Tector,Fareed Malik,4,0,0,NaN,False,NaN
34205,6,36,Harry Tector,Fareed Malik,0,0,0,NaN,True,Caught


In [32]:
# STEP 46: INSPECT MISSING EXTRAS CATEGORY WITH POSITIVE EXTRAS

missing_category_df = ball_df[
    ball_df["extras_type"].isna()
].copy()

missing_positive_extras = missing_category_df[
    (missing_category_df["extras1_runs"] > 0)
    | (missing_category_df["extras_2_runs"] > 0)
]

print(
    "MISSING CATEGORY WITH POSITIVE EXTRAS:",
    len(missing_positive_extras)
)

display(
    missing_positive_extras[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras1_runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ]
)

print("\nMISSING CATEGORY WITH ZERO EXTRAS:")
print(
    len(missing_category_df)
    - len(missing_positive_extras)
)

MISSING CATEGORY WITH POSITIVE EXTRAS: 2


,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras1_runs,extras_2_runs,is_wicket,dismissal_type
36854,17555,1,5,4,29,Dawid Malan,Haris Rauf,0,0,1,False,NaN
107489,36097,2,8,6,49,Rassie van der Dussen,Romario Shepherd,0,1,0,False,NaN



MISSING CATEGORY WITH ZERO EXTRAS:
189165


In [33]:

# STEP 47: Provisional extras and legal-delivery classification

import pandas as pd
import numpy as np

# Work on a copy. Never modify the original raw dataset.
ball_extras_audit = ball.copy()

# Check required columns
required_cols = ["extras_type", "extras1_runs", "extras_2_runs"]
missing_cols = [col for col in required_cols if col not in ball_extras_audit.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Normalize extras labels without changing the original column
extras_label = (
    ball_extras_audit["extras_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Calculate total extras for classification
extras_1 = pd.to_numeric(
    ball_extras_audit["extras1_runs"], errors="coerce"
).fillna(0)

extras_2 = pd.to_numeric(
    ball_extras_audit["extras_2_runs"], errors="coerce"
).fillna(0)

total_extras = extras_1 + extras_2

# Initialize all records as unresolved
ball_extras_audit["extras_classification"] = "unresolved"
ball_extras_audit["is_legal_ball_provisional"] = pd.Series(
    pd.NA, index=ball_extras_audit.index, dtype="boolean"
)

# Classify known categories
wide_mask = extras_label.eq("wide")
noball_mask = extras_label.eq("overstep")
bye_mask = extras_label.eq("byes")
legbye_mask = extras_label.eq("legbyes")

ball_extras_audit.loc[wide_mask, "extras_classification"] = "wide"
ball_extras_audit.loc[noball_mask, "extras_classification"] = "no_ball"
ball_extras_audit.loc[bye_mask, "extras_classification"] = "bye"
ball_extras_audit.loc[legbye_mask, "extras_classification"] = "legbye"

# Wides and no-balls do not count as valid balls
ball_extras_audit.loc[
    wide_mask | noball_mask, "is_legal_ball_provisional"
] = False

# Byes and leg-byes are legal-ball categories
ball_extras_audit.loc[
    bye_mask | legbye_mask, "is_legal_ball_provisional"
] = True

# Missing extras category + zero extras:
# Treat as an ordinary-ball candidate, based on the dataset pattern.
ordinary_mask = extras_label.isna() & total_extras.eq(0)

ball_extras_audit.loc[
    ordinary_mask, "extras_classification"
] = "ordinary_candidate"

ball_extras_audit.loc[
    ordinary_mask, "is_legal_ball_provisional"
] = True

# Everything else remains unresolved:
# - other
# - Penalty
# - missing category with positive extras
# - unexpected categories

# Display classification summary
print("EXTRAS CLASSIFICATION SUMMARY")
print("=" * 55)

print(
    ball_extras_audit["extras_classification"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nPROVISIONAL LEGAL-BALL STATUS")
print("=" * 55)

print(
    ball_extras_audit["is_legal_ball_provisional"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nUNRESOLVED RECORDS")
print("=" * 55)

print(
    ball_extras_audit.loc[
        ball_extras_audit["is_legal_ball_provisional"].isna(),
        ["extras_type", "extras1_runs", "extras_2_runs"]
    ].value_counts(dropna=False).head(15).to_string()
)

print("\nTotal records:", len(ball_extras_audit))
print(
    "Original DataFrame unchanged:",
    len(ball) == len(ball_extras_audit)
)


NameError: name 'ball' is not defined

In [34]:

# Check which DataFrames are currently available

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")


ball_df: (200999, 35)
matches_df: (883, 22)
competitions_df: (141, 4)
mapping_df: (6413, 6)
playing_eleven_df: (28381, 9)
df: (353, 3)
extras_samples: (20, 10)
overstep_df: (576, 35)
sequence_df: (200999, 36)
sequence_summary: (1724, 5)
other_df: (264, 35)
extras_audit: (7, 9)
over_summary: (1724, 6)
extra_deliveries: (8429, 35)
sample_over: (8, 35)
candidate_illegal: (8290, 35)
other_deliveries: (264, 35)
penalty_deliveries: (1, 35)
classification_audit: (7, 4)
over_audit: (32350, 10)
sequence_audit: (32350, 6)
innings_audit: (1724, 10)
innings_sequence_audit: (1724, 8)
category_df: (840, 39)
category_summary: (25, 6)
category_match_counts: (486, 3)
both_categories_matches: (107, 3)
over_category_audit: (794, 6)
both_categories_overs: (8, 6)
mixed_over_keys: (8, 3)
mixed_over_details: (66, 35)
penalty_df: (1, 35)
missing_category_df: (189167, 35)
missing_positive_extras: (2, 35)


In [35]:
# Step 47: Provisional legal-delivery classification
# We preserve uncertain categories instead of guessing their legal status.

ball_df = ball_df.copy()

# Normalize extras_type for consistent classification
extras_category = (
    ball_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Identify whether any extras were recorded
has_extras = (
    ball_df["extras1_runs"].fillna(0)
    + ball_df["extras_2_runs"].fillna(0)
) > 0

# Start with every delivery marked unresolved
ball_df["legal_status"] = "unresolved"
ball_df["classification_basis"] = "Requires further verification"

# Categories supported by the project masterplan and standard scoring
ball_df.loc[
    extras_category.isin(["wide", "overstep"]),
    ["legal_status", "classification_basis"]
] = ["illegal", "Wide / Overstep (no-ball)"]

ball_df.loc[
    extras_category.isin(["legbyes", "byes"]),
    ["legal_status", "classification_basis"]
] = ["legal", "Leg-byes / Byes"]

# Missing extras category with zero extras:
# Treat as an ordinary-ball candidate, but keep the assumption visible.
ordinary_candidate = (extras_category == "") & (~has_extras)

ball_df.loc[
    ordinary_candidate,
    ["legal_status", "classification_basis"]
] = ["legal_assumed", "Missing extras category; zero extras"]

# Missing category with positive extras remains unresolved.
# "other" and "Penalty" also remain unresolved.

# Audit the classification
classification_result = (
    ball_df.groupby(
        ["legal_status", "classification_basis"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
)

display(classification_result)

print("\nOverall legal-status counts:")
display(ball_df["legal_status"].value_counts(dropna=False))

print("\nTotal deliveries:", len(ball_df))
print("Unresolved deliveries:", (ball_df["legal_status"] == "unresolved").sum())

,legal_status,classification_basis,delivery_count
2,legal_assumed,Missing extras category; zero extras,189165
0,illegal,Wide / Overstep (no-ball),8290
1,legal,Leg-byes / Byes,3277
3,unresolved,Requires further verification,267



Overall legal-status counts:


legal_status
legal_assumed    189165
illegal            8290
legal              3277
unresolved          267
Name: count, dtype: int64


Total deliveries: 200999
Unresolved deliveries: 267


In [36]:
# Step 48: Inspect all unresolved delivery categories

unresolved_df = ball_df[
    ball_df["legal_status"] == "unresolved"
].copy()

print("Unresolved delivery count:", len(unresolved_df))

print("\nUnresolved extras categories:")
display(
    unresolved_df["extras_type"]
    .fillna("<MISSING>")
    .value_counts(dropna=False)
    .rename_axis("extras_type")
    .reset_index(name="delivery_count")
)

print("\nUnresolved runs and wicket combinations:")
display(
    unresolved_df.groupby(
        ["extras_type", "runs", "extras1_runs", "extras_2_runs", "is_wicket"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
)

print("\nSample unresolved records:")
display(
    unresolved_df[
        [
            "match_id", "innings", "overid", "over_ball_num",
            "striker_name", "bowler_name", "runs",
            "extras_type", "extras1_runs", "extras_2_runs",
            "is_wicket", "dismissal_type"
        ]
    ].head(20)
)

Unresolved delivery count: 267

Unresolved extras categories:


,extras_type,delivery_count
0,other,264
1,<MISSING>,2
2,Penalty,1



Unresolved runs and wicket combinations:


,extras_type,runs,extras1_runs,extras_2_runs,is_wicket,delivery_count
1,other,0,1,0,False,111
5,other,1,1,0,False,49
8,other,4,1,0,False,42
9,other,6,1,0,False,26
6,other,2,1,0,False,18
4,other,0,1,4,False,7
3,other,0,1,1,False,7
2,other,0,1,0,True,3
0,Penalty,0,5,0,False,1
7,other,3,1,0,False,1



Sample unresolved records:


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,is_wicket,dismissal_type
532,13513,1,10,1,Rovman Powell,Tymal Mills,0,other,1,0,False,NaN
1079,13515,1,16,6,Rovman Powell,Saqib Mahmood,0,other,1,0,False,NaN
1226,13515,2,19,5,Sam Billings,Sheldon Cottrell,2,other,1,0,False,NaN
2588,13614,1,18,5,Dasun Shanaka,Mitchell Starc,0,other,1,4,False,NaN
2609,13614,2,2,1,Ashton Agar,Nuwan Thushara,0,other,1,0,False,NaN
3309,14094,1,19,7,Mahendra Singh Dhoni,Shivam Mavi,6,other,1,0,False,NaN
4413,14098,2,20,1,Bhuvneshwar Kumar,Nathan Coulter Nile,0,other,1,0,False,NaN
5012,14101,1,17,4,Kagiso Rabada,Tim Southee,0,other,1,0,False,NaN
5365,14102,2,19,9,Kieron Pollard,Prasidh Krishna,0,other,1,0,True,RunOut
6575,14107,2,16,5,Patrick Cummins,Daniel Sams,2,other,1,0,False,NaN


In [37]:
# Step 49: Calculate minimum and maximum possible legal balls per innings

innings_ball_audit = (
    ball_df.groupby(["match_id", "innings"])
    .agg(
        recorded_deliveries=("legal_status", "size"),
        confirmed_or_assumed_legal=(
            "legal_status",
            lambda s: s.isin(["legal", "legal_assumed"]).sum()
        ),
        unresolved_deliveries=(
            "legal_status",
            lambda s: (s == "unresolved").sum()
        ),
        classified_illegal=(
            "legal_status",
            lambda s: (s == "illegal").sum()
        )
    )
    .reset_index()
)

# Lower bound: unresolved deliveries do not count as legal
innings_ball_audit["legal_balls_min"] = (
    innings_ball_audit["confirmed_or_assumed_legal"]
)

# Upper bound: assume every unresolved delivery was legal
innings_ball_audit["legal_balls_max"] = (
    innings_ball_audit["confirmed_or_assumed_legal"]
    + innings_ball_audit["unresolved_deliveries"]
)

innings_ball_audit["uncertainty_width"] = (
    innings_ball_audit["legal_balls_max"]
    - innings_ball_audit["legal_balls_min"]
)

print("Total innings:", len(innings_ball_audit))
print(
    "Innings containing unresolved deliveries:",
    (innings_ball_audit["unresolved_deliveries"] > 0).sum()
)
print(
    "Total unresolved deliveries:",
    innings_ball_audit["unresolved_deliveries"].sum()
)

print("\nLegal-ball range summary:")
display(
    innings_ball_audit[
        [
            "legal_balls_min",
            "legal_balls_max",
            "uncertainty_width"
        ]
    ].describe()
)

print("\nInnings with the most unresolved deliveries:")
display(
    innings_ball_audit.sort_values(
        ["unresolved_deliveries", "match_id"],
        ascending=[False, True]
    ).head(15)
)

Total innings: 1724
Innings containing unresolved deliveries: 233
Total unresolved deliveries: 267

Legal-ball range summary:


,legal_balls_min,legal_balls_max,uncertainty_width
count,1724.00000,1724.000000,1724.000000
mean,111.62529,111.780162,0.154872
std,18.61352,18.651486,0.423928
min,1.00000,1.000000,0.000000
25%,113.00000,113.000000,0.000000
50%,120.00000,120.000000,0.000000
75%,120.00000,120.000000,0.000000
max,120.00000,124.000000,4.000000



Innings with the most unresolved deliveries:


,match_id,innings,recorded_deliveries,confirmed_or_assumed_legal,unresolved_deliveries,classified_illegal,legal_balls_min,legal_balls_max,uncertainty_width
903,36096,1,126,120,4,2,120,124,4
524,20249,2,113,108,3,2,108,111,3
861,32618,2,132,120,3,9,120,123,3
926,36562,2,126,116,3,7,116,119,3
1255,42784,2,127,119,3,5,119,122,3
1274,45391,2,121,114,3,4,114,117,3
55,14108,2,122,118,2,2,118,120,2
100,14131,1,133,120,2,11,120,122,2
155,14159,2,130,120,2,8,120,122,2
203,15796,2,132,120,2,10,120,122,2


In [38]:
# Step 50: Investigate innings with more than 120 possible legal balls

exceptional_innings = innings_ball_audit[
    innings_ball_audit["legal_balls_max"] > 120
].copy()

print("Innings with upper bound above 120:", len(exceptional_innings))

display(
    exceptional_innings[
        [
            "match_id",
            "innings",
            "recorded_deliveries",
            "confirmed_or_assumed_legal",
            "unresolved_deliveries",
            "classified_illegal",
            "legal_balls_min",
            "legal_balls_max"
        ]
    ].sort_values(
        "legal_balls_max",
        ascending=False
    ).head(20)
)

# Inspect the actual delivery records for the innings with the highest upper bound
worst_case = exceptional_innings.sort_values(
    "legal_balls_max",
    ascending=False
).iloc[0]

worst_case_deliveries = ball_df[
    (ball_df["match_id"] == worst_case["match_id"])
    & (ball_df["innings"] == worst_case["innings"])
].copy()

print(
    f"\nDelivery-level inspection: match {worst_case['match_id']}, "
    f"innings {worst_case['innings']}"
)

display(
    worst_case_deliveries[
        [
            "overid",
            "over_ball_num",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "legal_status",
            "classification_basis",
            "is_wicket",
            "dismissal_type"
        ]
    ].tail(20)
)

Innings with upper bound above 120: 150


,match_id,innings,recorded_deliveries,confirmed_or_assumed_legal,unresolved_deliveries,classified_illegal,legal_balls_min,legal_balls_max
903,36096,1,126,120,4,2,120,124
861,32618,2,132,120,3,9,120,123
203,15796,2,132,120,2,10,120,122
322,17626,1,126,120,2,4,120,122
155,14159,2,130,120,2,8,120,122
100,14131,1,133,120,2,11,120,122
921,36560,1,124,120,2,2,120,122
1190,40763,1,131,120,2,9,120,122
1255,42784,2,127,119,3,5,119,122
1586,50359,2,125,120,2,3,120,122



Delivery-level inspection: match 36096, innings 1


,overid,over_ball_num,runs,extras_type,extras1_runs,extras_2_runs,legal_status,classification_basis,is_wicket,dismissal_type
107179,18,3,1,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107180,18,4,4,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107181,18,5,1,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107182,18,6,0,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107183,19,1,4,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107184,19,2,0,other,1,0,unresolved,Requires further verification,False,NaN
107185,19,3,1,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107186,19,4,4,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107187,19,5,1,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN
107188,19,6,6,NaN,0,0,legal_assumed,Missing extras category; zero extras,False,NaN


In [39]:
# Step 51: Join exceptional innings with match and competition metadata

exceptional_context = exceptional_innings.merge(
    matches_df[
        [
            "match_id",
            "comp_id",
            "match_date",
            "stage",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "result"
        ]
    ],
    on="match_id",
    how="left"
).merge(
    competitions_df[
        ["comp_id", "comp_name", "match_format", "season"]
    ],
    on="comp_id",
    how="left"
)

print("Competition formats among exceptional innings:")
display(
    exceptional_context["match_format"]
    .value_counts(dropna=False)
    .rename_axis("match_format")
    .reset_index(name="innings_count")
)

print("\nExceptional innings with match context:")
display(
    exceptional_context[
        [
            "match_id",
            "innings",
            "match_date",
            "comp_name",
            "match_format",
            "season",
            "stage",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target",
            "recorded_deliveries",
            "legal_balls_min",
            "legal_balls_max",
            "unresolved_deliveries",
            "result"
        ]
    ].sort_values(
        ["legal_balls_max", "unresolved_deliveries"],
        ascending=False
    ).head(25)
)

Competition formats among exceptional innings:


,match_format,innings_count
0,T20,76
1,T20I,74



Exceptional innings with match context:


,match_id,innings,match_date,comp_name,match_format,season,stage,first_inn_revised_overs,second_inn_revised_overs,revised_target,recorded_deliveries,legal_balls_min,legal_balls_max,unresolved_deliveries,result
83,36096,1,2024-08-24,South Africa in West Indies T20I Series 2024,T20I,2024,Others,NaN,NaN,NaN,126,120,124,4,West Indies Won by 7 wickets
78,32618,2,2024-05-26,South Africa in West Indies T20I Series [May 2...,T20I,2024,Others,NaN,NaN,NaN,132,120,123,3,West Indies Won by 16 runs
109,42784,2,2025-05-19,Bangladesh in United Arab Emirates T20I Series...,T20I,2025,League,NaN,NaN,NaN,127,119,122,3,United Arab Emirates Won by 2 wickets
9,14131,1,2022-04-25,IPL 2022,T20,2022,League,NaN,NaN,NaN,133,120,122,2,Punjab Kings Won by 11 Runs
13,14159,2,2022-05-17,IPL 2022,T20,2022,League,NaN,NaN,NaN,130,120,122,2,Sunrisers Hyderabad Won by 3 Runs
17,15796,2,2022-06-28,India in Ireland T20I Series 2022,T20I,2022,League,NaN,NaN,NaN,132,120,122,2,India Won by 4 Runs
26,17626,1,2022-10-02,South Africa in India T20I Series 2022/23,T20I,2022/23,League,NaN,NaN,NaN,126,120,122,2,India Won by 16 Runs
40,20204,1,2023-04-17,IPL 2023,T20,2023,League,NaN,NaN,NaN,126,120,122,2,Chennai Super Kings Won by 8 Runs
49,20244,2,2023-05-17,IPL 2023,T20,2023,League,NaN,NaN,NaN,124,120,122,2,Delhi Capitals Won by 15 runs
74,31286,2,2024-05-18,IPL 2024,T20,2024,League,NaN,NaN,NaN,126,120,122,2,Royal Challengers Bengaluru Won by 27 runs


In [40]:
# Step 52: Locate unresolved deliveries relative to the 120-ball limit

exceptional_keys = exceptional_innings[["match_id", "innings"]]

exceptional_deliveries = ball_df.merge(
    exceptional_keys,
    on=["match_id", "innings"],
    how="inner"
).sort_values(
    ["match_id", "innings", "overid", "over_ball_num"]
).copy()

# Count classified legal deliveries cumulatively within each innings
exceptional_deliveries["known_legal_before"] = (
    exceptional_deliveries.groupby(["match_id", "innings"])[
        "legal_status"
    ]
    .transform(
        lambda s: s.isin(["legal", "legal_assumed"]).cumsum().shift(
            fill_value=0
        )
    )
)

# Keep unresolved records and locate them relative to the 120-ball threshold
unresolved_positions = exceptional_deliveries[
    exceptional_deliveries["legal_status"] == "unresolved"
].copy()

unresolved_positions["position_group"] = unresolved_positions[
    "known_legal_before"
].apply(
    lambda n: (
        "Before 120 known legal balls"
        if n < 120
        else "At/after 120 known legal balls"
    )
)

print("Unresolved delivery positions:")
display(
    unresolved_positions["position_group"]
    .value_counts()
    .rename_axis("position_group")
    .reset_index(name="delivery_count")
)

print("\nUnresolved deliveries after 120 known legal balls:")
display(
    unresolved_positions[
        unresolved_positions["known_legal_before"] >= 120
    ][
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "extras_type",
            "runs",
            "known_legal_before"
        ]
    ].head(30)
)

print("\nExample: match 36096, innings 1 — full unresolved positions:")
display(
    unresolved_positions[
        (unresolved_positions["match_id"] == 36096)
        & (unresolved_positions["innings"] == 1)
    ][
        [
            "overid",
            "over_ball_num",
            "extras_type",
            "runs",
            "known_legal_before"
        ]
    ]
)

Unresolved delivery positions:


,position_group,delivery_count
0,Before 120 known legal balls,173



Unresolved deliveries after 120 known legal balls:


,match_id,innings,overid,over_ball_num,extras_type,runs,known_legal_before



Example: match 36096, innings 1 — full unresolved positions:


,overid,over_ball_num,extras_type,runs,known_legal_before
10605,15,2,other,0,85
10630,19,2,other,0,109
10635,19,7,other,1,113
10636,19,8,other,4,113


In [41]:
# Step 53: Full unresolved-delivery audit across all innings

all_deliveries = ball_df.sort_values(
    ["match_id", "innings", "overid", "over_ball_num"]
).copy()

# Cumulative count of legal and assumed-legal deliveries before each record
all_deliveries["known_legal_before"] = (
    all_deliveries.groupby(["match_id", "innings"])["legal_status"]
    .transform(
        lambda s: s.isin(["legal", "legal_assumed"]).cumsum().shift(
            fill_value=0
        )
    )
)

# Final classified legal-ball count for each innings
final_legal_counts = (
    all_deliveries.groupby(["match_id", "innings"])["legal_status"]
    .apply(lambda s: s.isin(["legal", "legal_assumed"]).sum())
    .rename("final_known_legal_balls")
    .reset_index()
)

# Select every unresolved delivery
all_unresolved = all_deliveries[
    all_deliveries["legal_status"] == "unresolved"
].copy()

all_unresolved = all_unresolved.merge(
    final_legal_counts,
    on=["match_id", "innings"],
    how="left"
)

# Classify its position relative to the final known legal-ball count
all_unresolved["position_group"] = np.select(
    [
        all_unresolved["known_legal_before"] >= 120,
        all_unresolved["final_known_legal_balls"] >= 120
    ],
    [
        "At/after 120 known legal balls",
        "Before 120; innings reaches 120 known legal balls"
    ],
    default="Before 120; innings ends below 120 known legal balls"
)

print("Total unresolved deliveries:", len(all_unresolved))

print("\nPosition distribution across ALL unresolved deliveries:")
display(
    all_unresolved["position_group"]
    .value_counts()
    .rename_axis("position_group")
    .reset_index(name="delivery_count")
)

print("\nUnresolved categories by position:")
display(
    all_unresolved.groupby(
        ["extras_type", "position_group"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values("delivery_count", ascending=False)
)

print("\nUnresolved deliveries by match format:")
unresolved_with_format = all_unresolved.merge(
    matches_df[["match_id", "comp_id"]],
    on="match_id",
    how="left"
).merge(
    competitions_df[["comp_id", "match_format"]],
    on="comp_id",
    how="left"
)

display(
    unresolved_with_format.groupby(
        ["match_format", "extras_type"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
)

print("\nAll unresolved records at or after 120 known legal balls:")
display(
    all_unresolved[
        all_unresolved["known_legal_before"] >= 120
    ][
        [
            "match_id", "innings", "overid", "over_ball_num",
            "extras_type", "runs", "known_legal_before",
            "final_known_legal_balls"
        ]
    ]
)

Total unresolved deliveries: 267

Position distribution across ALL unresolved deliveries:


,position_group,delivery_count
0,Before 120; innings reaches 120 known legal balls,168
1,Before 120; innings ends below 120 known legal...,99



Unresolved categories by position:


,extras_type,position_group,delivery_count
2,other,Before 120; innings reaches 120 known legal balls,168
1,other,Before 120; innings ends below 120 known legal...,96
3,NaN,Before 120; innings ends below 120 known legal...,2
0,Penalty,Before 120; innings ends below 120 known legal...,1



Unresolved deliveries by match format:


,match_format,extras_type,delivery_count
0,T20,other,127
1,T20I,Penalty,1
2,T20I,other,137
3,T20I,NaN,2



All unresolved records at or after 120 known legal balls:


,match_id,innings,overid,over_ball_num,extras_type,runs,known_legal_before,final_known_legal_balls


In [42]:
# Step 54: Inspect unresolved delivery positions within overs

all_unresolved["over_ball_group"] = np.where(
    all_unresolved["over_ball_num"] <= 6,
    "Recorded position 1-6",
    "Recorded position 7+"
)

print("Unresolved deliveries by recorded over position:")
display(
    all_unresolved.groupby(
        ["extras_type", "over_ball_group"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(
        ["extras_type", "over_ball_group"],
        na_position="last"
    )
)

print("\nExact recorded-position distribution:")
display(
    all_unresolved.groupby(
        ["extras_type", "over_ball_num"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(
        ["extras_type", "over_ball_num"],
        na_position="last"
    )
)

print("\nUnresolved deliveries by over number:")
display(
    all_unresolved.groupby(
        ["extras_type", "overid"],
        dropna=False
    )
    .size()
    .reset_index(name="delivery_count")
    .sort_values(
        ["extras_type", "overid"],
        na_position="last"
    )
)

Unresolved deliveries by recorded over position:


,extras_type,over_ball_group,delivery_count
0,Penalty,Recorded position 1-6,1
1,other,Recorded position 1-6,244
2,other,Recorded position 7+,20
3,NaN,Recorded position 1-6,2



Exact recorded-position distribution:


,extras_type,over_ball_num,delivery_count
0,Penalty,4,1
1,other,1,44
2,other,2,35
3,other,3,33
4,other,4,44
5,other,5,47
6,other,6,41
7,other,7,14
8,other,8,5
9,other,9,1



Unresolved deliveries by over number:


,extras_type,overid,delivery_count
0,Penalty,6,1
1,other,1,6
2,other,2,8
3,other,3,6
4,other,4,5
5,other,5,12
6,other,6,9
7,other,7,4
8,other,8,7
9,other,9,13


In [43]:
# Step 55: Build cumulative legal-ball count bounds

ball_df = ball_df.sort_values(
    ["match_id", "innings", "overid", "over_ball_num"]
).copy()

# Minimum: count only legal and legal_assumed deliveries
ball_df["legal_ball_increment_min"] = (
    ball_df["legal_status"].isin(["legal", "legal_assumed"]).astype(int)
)

# Maximum: also allow unresolved deliveries to be legal
ball_df["legal_ball_increment_max"] = (
    ball_df["legal_status"]
    .isin(["legal", "legal_assumed", "unresolved"])
    .astype(int)
)

# Cumulative counts before and after each delivery
group_keys = ["match_id", "innings"]

ball_df["legal_balls_min_before"] = (
    ball_df.groupby(group_keys)["legal_ball_increment_min"]
    .cumsum()
    - ball_df["legal_ball_increment_min"]
)

ball_df["legal_balls_max_before"] = (
    ball_df.groupby(group_keys)["legal_ball_increment_max"]
    .cumsum()
    - ball_df["legal_ball_increment_max"]
)

ball_df["legal_balls_min_after"] = (
    ball_df.groupby(group_keys)["legal_ball_increment_min"].cumsum()
)

ball_df["legal_balls_max_after"] = (
    ball_df.groupby(group_keys)["legal_ball_increment_max"].cumsum()
)

# Audit the final counters against the earlier innings-level summary
counter_audit = (
    ball_df.groupby(group_keys)
    .agg(
        final_min=("legal_balls_min_after", "max"),
        final_max=("legal_balls_max_after", "max")
    )
    .reset_index()
    .merge(
        innings_ball_audit[
            ["match_id", "innings", "legal_balls_min", "legal_balls_max"]
        ],
        on=group_keys,
        how="left"
    )
)

print("Counter consistency checks:")
print(
    "Minimum counters match:",
    (counter_audit["final_min"] == counter_audit["legal_balls_min"]).all()
)
print(
    "Maximum counters match:",
    (counter_audit["final_max"] == counter_audit["legal_balls_max"]).all()
)

print("\nFinal legal-ball counter summary:")
display(
    counter_audit[
        ["final_min", "final_max"]
    ].describe()
)

print("\nSample delivery-level counters:")
display(
    ball_df[
        [
            "match_id", "innings", "overid", "over_ball_num",
            "extras_type", "legal_status",
            "legal_balls_min_before", "legal_balls_max_before",
            "legal_balls_min_after", "legal_balls_max_after"
        ]
    ].head(15)
)

Counter consistency checks:
Minimum counters match: True
Maximum counters match: True

Final legal-ball counter summary:


,final_min,final_max
count,1724.00000,1724.000000
mean,111.62529,111.780162
std,18.61352,18.651486
min,1.00000,1.000000
25%,113.00000,113.000000
50%,120.00000,120.000000
75%,120.00000,120.000000
max,120.00000,124.000000



Sample delivery-level counters:


,match_id,innings,overid,over_ball_num,extras_type,legal_status,legal_balls_min_before,legal_balls_max_before,legal_balls_min_after,legal_balls_max_after
0,13511,1,1,1,NaN,legal_assumed,0,0,1,1
1,13511,1,1,2,NaN,legal_assumed,1,1,2,2
2,13511,1,1,3,NaN,legal_assumed,2,2,3,3
3,13511,1,1,4,NaN,legal_assumed,3,3,4,4
4,13511,1,1,5,NaN,legal_assumed,4,4,5,5
5,13511,1,1,6,NaN,legal_assumed,5,5,6,6
6,13511,1,2,1,NaN,legal_assumed,6,6,7,7
7,13511,1,2,2,NaN,legal_assumed,7,7,8,8
8,13511,1,2,3,NaN,legal_assumed,8,8,9,9
9,13511,1,2,4,NaN,legal_assumed,9,9,10,10


In [44]:
# Audit batter runs and extras before calculating innings scores

run_columns = ["runs", "extras1_runs", "extras_2_runs"]

print("Column data types:")
display(ball_df[run_columns].dtypes.to_frame("dtype"))

print("\nMissing values:")
display(ball_df[run_columns].isna().sum().to_frame("missing_count"))

print("\nNegative values:")
display((ball_df[run_columns] < 0).sum().to_frame("negative_count"))

print("\nDescriptive statistics:")
display(ball_df[run_columns].describe().T)

# Calculate total runs recorded on each delivery
run_audit = ball_df[run_columns].fillna(0).copy()
run_audit["total_delivery_runs"] = run_audit.sum(axis=1)

print("\nRun totals:")
display(
    run_audit[run_columns + ["total_delivery_runs"]]
    .sum()
    .to_frame("total_runs")
)

print("\nDelivery-level run distribution:")
display(
    run_audit["total_delivery_runs"]
    .value_counts()
    .sort_index()
    .to_frame("delivery_count")
)

print("\nRows with missing or negative run values:")
invalid_runs = ball_df[
    ball_df[run_columns].isna().any(axis=1)
    | (ball_df[run_columns] < 0).any(axis=1)
]

print("Invalid rows:", len(invalid_runs))
display(invalid_runs[["match_id", "innings", "overid", "over_ball_num"] + run_columns].head(20))

Column data types:


,dtype
runs,int64
extras1_runs,int64
extras_2_runs,int64



Missing values:


,missing_count
runs,0
extras1_runs,0
extras_2_runs,0



Negative values:


,negative_count
runs,0
extras1_runs,0
extras_2_runs,0


... (4 outputs left out)


Delivery-level run distribution:


,delivery_count
total_delivery_runs,
0,68027
1,82209
2,13196
3,751
4,23564
5,438
6,12741
7,73



Rows with missing or negative run values:
Invalid rows: 0


,match_id,innings,overid,over_ball_num,runs,extras1_runs,extras_2_runs


In [45]:
# Create total runs per delivery and cumulative innings scores

ball_df = ball_df.copy()

ball_df["total_delivery_runs"] = (
    ball_df["runs"]
    + ball_df["extras1_runs"]
    + ball_df["extras_2_runs"]
)

group_keys = ["match_id", "innings"]

# Cumulative score after each recorded delivery
ball_df["innings_score_after"] = (
    ball_df.groupby(group_keys)["total_delivery_runs"].cumsum()
)

# Score before the current delivery
ball_df["innings_score_before"] = (
    ball_df["innings_score_after"] - ball_df["total_delivery_runs"]
)

# Independent innings-total audit
innings_score_audit = (
    ball_df.groupby(group_keys)
    .agg(
        calculated_total=("total_delivery_runs", "sum"),
        final_cumulative_score=("innings_score_after", "max"),
        recorded_deliveries=("total_delivery_runs", "size")
    )
    .reset_index()
)

innings_score_audit["score_matches"] = (
    innings_score_audit["calculated_total"]
    == innings_score_audit["final_cumulative_score"]
)

print("All innings scores reconcile:",
      innings_score_audit["score_matches"].all())

print("Number of innings:", len(innings_score_audit))

print("\nInnings total score summary:")
display(innings_score_audit["calculated_total"].describe())

print("\nSample delivery-level scores:")
display(
    ball_df[
        [
            "match_id", "innings", "overid", "over_ball_num",
            "total_delivery_runs",
            "innings_score_before", "innings_score_after"
        ]
    ].head(15)
)

All innings scores reconcile: True
Number of innings: 1724

Innings total score summary:


count    1724.000000
mean      164.882251
std        42.055950
min         0.000000
25%       142.000000
50%       166.000000
75%       193.000000
max       304.000000
Name: calculated_total, dtype: float64


Sample delivery-level scores:


,match_id,innings,overid,over_ball_num,total_delivery_runs,innings_score_before,innings_score_after
0,13511,1,1,1,0,0,0
1,13511,1,1,2,0,0,0
2,13511,1,1,3,0,0,0
3,13511,1,1,4,0,0,0
4,13511,1,1,5,6,0,6
5,13511,1,1,6,0,6,6
6,13511,1,2,1,0,6,6
7,13511,1,2,2,0,6,6
8,13511,1,2,3,0,6,6
9,13511,1,2,4,4,6,10


In [46]:
# Inspect unusual innings totals and compare scores by competition format

innings_review = innings_score_audit.merge(
    matches_df[["match_id", "comp_id", "match_date", "result"]],
    on="match_id",
    how="left"
)

innings_review = innings_review.merge(
    competitions_df[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left"
)

print("Zero-run innings:")
zero_score_innings = innings_review[
    innings_review["calculated_total"] == 0
]
display(zero_score_innings)

print("\nInnings with scores above 250:")
high_score_innings = innings_review[
    innings_review["calculated_total"] > 250
].sort_values("calculated_total", ascending=False)

display(high_score_innings)

print("\nScore distribution by competition format:")
display(
    innings_review.groupby("match_format")["calculated_total"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nInnings count per match:")
innings_per_match = (
    innings_review.groupby("match_id")
    .size()
    .value_counts()
    .sort_index()
    .to_frame("match_count")
)

display(innings_per_match)

print("\nMissing competition metadata:",
      innings_review["match_format"].isna().sum())

Zero-run innings:


,match_id,innings,calculated_total,final_cumulative_score,recorded_deliveries,score_matches,comp_id,match_date,result,comp_name,match_format,season
1450,48027,3,0,0,2,True,3289,2025-11-21,Bangladesh A Won The Super Over,Asia Cup Rising Stars 2025,T20,2025



Innings with scores above 250:


,match_id,innings,calculated_total,final_cumulative_score,recorded_deliveries,score_matches,comp_id,match_date,result,comp_name,match_format,season
917,36209,1,304,304,132,True,2628,2025-09-12,England Won by 146 runs,South Africa in England T20I Series 2025,T20I,2025
1426,48016,1,297,297,126,True,3289,2025-11-14,India A Won by 148 runs,Asia Cup Rising Stars 2025,T20,2025
967,37278,1,297,297,126,True,2512,2024-10-12,India Won by 133 runs,Bangladesh in India T20I Series 2024,T20I,2024
745,31248,1,287,287,132,True,2193,2024-04-15,Sunrisers Hyderabad Won by 25 runs,IPL 2024,T20,2024
1056,40695,1,286,286,133,True,2725,2025-03-23,Sunrisers Hyderabad Won by 44 runs,IPL 2025,T20,2025
1006,37943,1,283,283,133,True,2718,2024-11-15,India Won by 135 runs,India in South Africa T20I Series 2024,T20I,2024
1172,40753,1,278,278,131,True,2725,2025-05-25,Sunrisers Hyderabad Won by 110 runs,IPL 2025,T20,2025
689,30063,1,277,277,130,True,2193,2024-03-27,Sunrisers Hyderabad won by 31 runs,IPL 2024,T20,2024
705,30071,1,272,272,134,True,2193,2024-04-03,Kolkata Knight Riders Won by 106 runs,IPL 2024,T20,2024
1474,48595,1,271,271,128,True,3329,2026-01-31,India Won by 46 runs,New Zealand in India T20I Series 2026,T20I,2026



Score distribution by competition format:


,count,mean,median,min,max
match_format,,,,,
T20,798,174.08,175.0,0,297
T20I,926,156.96,158.0,2,304



Innings count per match:


,match_count
1,22
2,839
4,6



Missing competition metadata: 0


In [47]:
# Identify matches with an unusual number of recorded innings

match_innings_counts = (
    innings_review.groupby("match_id")
    .agg(
        innings_count=("innings", "nunique"),
        innings_numbers=("innings", lambda x: sorted(x.tolist())),
        total_recorded_runs=("calculated_total", "sum")
    )
    .reset_index()
)

unusual_match_ids = match_innings_counts[
    match_innings_counts["innings_count"].isin([1, 4])
]["match_id"]

unusual_matches = (
    innings_review[
        innings_review["match_id"].isin(unusual_match_ids)
    ]
    .sort_values(["match_id", "innings"])
)

print("Matches with only one recorded innings:",
      (match_innings_counts["innings_count"] == 1).sum())

display(
    unusual_matches[
        unusual_matches["match_id"].isin(
            match_innings_counts.loc[
                match_innings_counts["innings_count"] == 1, "match_id"
            ]
        )
    ][
        [
            "match_id", "innings", "calculated_total",
            "recorded_deliveries", "match_date",
            "comp_name", "result"
        ]
    ].to_string(index=False)
)

print("\nMatches with four recorded innings:")

display(
    unusual_matches[
        unusual_matches["match_id"].isin(
            match_innings_counts.loc[
                match_innings_counts["innings_count"] == 4, "match_id"
            ]
        )
    ][
        [
            "match_id", "innings", "calculated_total",
            "recorded_deliveries", "match_date",
            "comp_name", "result"
        ]
    ].to_string(index=False)
)

Matches with only one recorded innings: 22


' match_id  innings  calculated_total  recorded_deliveries match_date                                          comp_name          result\n    15410        1                28                   22 2022-06-19             South Africa in India T20I Series 2022 Match Abandoned\n    15546        1               105                   82 2022-07-02         Bangladesh in West Indies T20I Series 2022       No Result\n    20226        1               125                  118 2023-05-03                                           IPL 2023 Match Abandoned\n    21406        1               164                  118 2023-04-20           New Zealand in Pakistan T20I Series 2023 Match Abandoned\n    28333        1                72                   66 2023-12-29         Bangladesh in New Zealand T20I Series 2023       No Result\n    31707        1                 2                    2 2024-04-18           New Zealand in Pakistan T20I Series 2024       No Result\n    35424        1               169    


Matches with four recorded innings:


' match_id  innings  calculated_total  recorded_deliveries match_date                           comp_name                                                            result\n    34429        1               137                  125 2024-07-30 India in Sri Lanka T20I Series 2024  India Tied with Srilanka (India Won the Super Over by 2 wickets)\n    34429        2               137                  127 2024-07-30 India in Sri Lanka T20I Series 2024  India Tied with Srilanka (India Won the Super Over by 2 wickets)\n    34429        3                 2                    4 2024-07-30 India in Sri Lanka T20I Series 2024  India Tied with Srilanka (India Won the Super Over by 2 wickets)\n    34429        4                 4                    1 2024-07-30 India in Sri Lanka T20I Series 2024  India Tied with Srilanka (India Won the Super Over by 2 wickets)\n    40725        1               188                  128 2025-04-16                            IPL 2025 Delhi Capitals Tied with Rajasthan

In [48]:
# Classify match structures and audit their result categories

match_structure = (
    innings_review.groupby("match_id")
    .agg(
        innings_count=("innings", "nunique"),
        innings_numbers=("innings", lambda x: tuple(sorted(x))),
        result=("result", "first"),
        comp_name=("comp_name", "first"),
        match_format=("match_format", "first")
    )
    .reset_index()
)

print("Match structure counts:")
display(
    match_structure["innings_count"]
    .value_counts()
    .sort_index()
    .to_frame("match_count")
)

print("\nResult categories for matches with exactly two innings:")
two_innings_results = (
    match_structure[match_structure["innings_count"] == 2]
    ["result"]
    .value_counts(dropna=False)
    .to_frame("match_count")
)

display(two_innings_results)

print("\nMatches with two innings but abandoned/no-result/tied result:")
special_result_matches = match_structure[
    (match_structure["innings_count"] == 2)
    & (
        match_structure["result"].astype(str).str.contains(
            "abandon|no result|tied", case=False, na=False
        )
    )
]

display(special_result_matches)

print("\nMatches with missing result:")
display(
    match_structure[match_structure["result"].isna()]
)

Match structure counts:


,match_count
innings_count,
1,22
2,839
4,6



Result categories for matches with exactly two innings:


,match_count
result,
India Won by 7 wickets,8
Delhi Capitals Won by 6 wickets,7
Kolkata Knight Riders Won by 8 wickets,6
India Won by 6 Wickets,5
Gujarat Titans Won by 7 wickets,5
...,...
India Won by 127 runs,1
India Won by 2 runs,1
Pakistan Won by 82 runs,1



Matches with two innings but abandoned/no-result/tied result:


,match_id,innings_count,innings_numbers,result,comp_name,match_format
177,17765,2,"(1, 2)",No Result,England in Australia T20I Series 2022,T20I
179,18532,2,"(1, 2)",Match Tied,India in New Zealand T20I Series 2022/23,T20I
574,40737,2,"(1, 2)",No Result,IPL 2025,T20



Matches with missing result:


,match_id,innings_count,innings_numbers,result,comp_name,match_format


In [49]:
# Inspect match metadata and innings details for the two No Result matches
# and the one tied match.

exception_ids = [17765, 18532, 40737]

match_columns = [
    "match_id", "match_date", "comp_id",
    "home_team", "away_team", "result",
    "first_inn_revised_overs", "second_inn_revised_overs",
    "revised_target", "win_team_id"
]

print("Match-level metadata:")
display(
    matches_df.loc[
        matches_df["match_id"].isin(exception_ids), match_columns
    ].sort_values("match_id")
)

print("\nInnings-level details:")
exception_innings = (
    innings_review[innings_review["match_id"].isin(exception_ids)]
    .merge(
        innings_ball_audit[
            [
                "match_id", "innings",
                "legal_balls_min", "legal_balls_max",
                "unresolved_deliveries"
            ]
        ],
        on=["match_id", "innings"],
        how="left"
    )
    .sort_values(["match_id", "innings"])
)

display(
    exception_innings[
        [
            "match_id", "innings", "calculated_total",
            "recorded_deliveries", "legal_balls_min",
            "legal_balls_max", "unresolved_deliveries",
            "match_date", "comp_name", "result"
        ]
    ]
)

print("\nDelivery sequence endpoints:")
delivery_endpoints = (
    ball_df[ball_df["match_id"].isin(exception_ids)]
    .groupby(["match_id", "innings"])
    .agg(
        first_over=("overid", "min"),
        last_over=("overid", "max"),
        last_recorded_ball=("over_ball_num", "last"),
        last_delivery_time=("delivery_ist_time", "last")
    )
    .reset_index()
    .sort_values(["match_id", "innings"])
)

display(delivery_endpoints)

Match-level metadata:


,match_id,match_date,comp_id,home_team,away_team,result,first_inn_revised_overs,second_inn_revised_overs,revised_target,win_team_id
177,17765,2022-10-14,1611,Australia,England,No Result,72.0,72.0,130.0,NaN
179,18532,2022-11-22,1646,New Zealand,India,Match Tied,NaN,NaN,NaN,NaN
569,40737,2025-04-26,2725,Kolkata Knight Riders,Punjab Kings,No Result,NaN,NaN,NaN,NaN



Innings-level details:


,match_id,innings,calculated_total,recorded_deliveries,legal_balls_min,legal_balls_max,unresolved_deliveries,match_date,comp_name,result
0,17765,1,112,75,72,72,0,2022-10-14,England in Australia T20I Series 2022,No Result
1,17765,2,30,25,23,23,0,2022-10-14,England in Australia T20I Series 2022,No Result
2,18532,1,160,123,118,118,0,2022-11-22,India in New Zealand T20I Series 2022/23,Match Tied
3,18532,2,75,56,54,54,0,2022-11-22,India in New Zealand T20I Series 2022/23,Match Tied
4,40737,1,201,123,120,120,0,2025-04-26,IPL 2025,No Result
5,40737,2,7,7,6,6,0,2025-04-26,IPL 2025,No Result



Delivery sequence endpoints:


,match_id,innings,first_over,last_over,last_recorded_ball,last_delivery_time
0,17765,1,1,12,6,2022-10-14 16:08:56 UTC
1,17765,2,1,4,5,2022-10-14 16:38:41 UTC
2,18532,1,1,20,4,2022-11-22 14:21:57 UTC
3,18532,2,1,9,6,2022-11-22 15:20:45 UTC
4,40737,1,1,20,6,2025-04-26 15:42:27 UTC
5,40737,2,1,1,7,2025-04-26 16:02:01 UTC


In [50]:
# Create primary-analysis eligibility flags at innings level

eligible_innings = innings_review.copy()

result_text = (
    eligible_innings["result"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Only regulation innings are eligible; Super Overs are excluded
eligible_innings["is_regulation_innings"] = (
    eligible_innings["innings"].isin([1, 2])
)

# Exclude abandoned and no-result matches from the primary analysis
eligible_innings["is_completed_match_result"] = ~result_text.str.contains(
    r"no result|abandoned",
    regex=True
)

# A tied match can contribute first-innings observations,
# but its chase is excluded because the revised target is unavailable.
eligible_innings["has_reliable_chase_target"] = True

tied_chase_mask = (
    result_text.str.contains("tied", regex=False)
    & (eligible_innings["innings"] == 2)
)

eligible_innings.loc[
    tied_chase_mask, "has_reliable_chase_target"
] = False

# Primary eligibility:
# - regulation innings only
# - completed match result
# - reliable chase target for second innings
eligible_innings["primary_analysis_eligible"] = (
    eligible_innings["is_regulation_innings"]
    & eligible_innings["is_completed_match_result"]
    & eligible_innings["has_reliable_chase_target"]
)

print("Primary-analysis eligibility:")
display(
    eligible_innings["primary_analysis_eligible"]
    .value_counts()
    .rename_axis("eligible")
    .to_frame("innings_count")
)

print("\nExclusion reasons:")
display(
    eligible_innings.loc[
        ~eligible_innings["primary_analysis_eligible"],
        [
            "match_id", "innings", "result",
            "calculated_total", "recorded_deliveries",
            "is_regulation_innings",
            "is_completed_match_result",
            "has_reliable_chase_target"
        ]
    ].head(30)
)

print("\nEligibility by competition format:")
display(
    eligible_innings.groupby("match_format")[
        "primary_analysis_eligible"
    ].agg(["count", "sum"])
)

Primary-analysis eligibility:


,innings_count
eligible,
True,1681
False,43



Exclusion reasons:


,match_id,innings,result,calculated_total,recorded_deliveries,is_regulation_innings,is_completed_match_result,has_reliable_chase_target
182,15410,1,Match Abandoned,28,22,True,False,True
189,15546,1,No Result,105,82,True,False,True
352,17765,1,No Result,112,75,True,False,True
353,17765,2,No Result,30,25,True,False,True
357,18532,2,Match Tied,75,56,True,True,False
478,20226,1,Match Abandoned,125,118,True,False,True
551,21406,1,Match Abandoned,164,118,True,False,True
632,28333,1,No Result,72,66,True,False,True
829,31707,1,No Result,2,2,True,False,True
889,34429,2,India Tied with Srilanka (India Won the Super ...,137,127,True,True,False



Eligibility by competition format:


,count,sum
match_format,,
T20,798,782
T20I,926,899


In [51]:
# Correct primary-analysis eligibility rules

eligible_innings = innings_review.copy()

result_text = (
    eligible_innings["result"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Only regulation innings are eligible
eligible_innings["is_regulation_innings"] = (
    eligible_innings["innings"].isin([1, 2])
)

# Exclude abandoned and no-result matches
eligible_innings["is_completed_match_result"] = ~result_text.str.contains(
    r"no result|abandoned",
    regex=True
)

# Super Over matches have valid regulation innings 1 and 2.
# The separate Match Tied record (match 18532) has an incomplete
# second innings and no revised-target metadata.
eligible_innings["has_reliable_chase_target"] = True

unverified_tied_chase = (
    (eligible_innings["match_id"] == 18532)
    & (eligible_innings["innings"] == 2)
)

eligible_innings.loc[
    unverified_tied_chase,
    "has_reliable_chase_target"
] = False

eligible_innings["primary_analysis_eligible"] = (
    eligible_innings["is_regulation_innings"]
    & eligible_innings["is_completed_match_result"]
    & eligible_innings["has_reliable_chase_target"]
)

print("Corrected eligibility counts:")
display(
    eligible_innings["primary_analysis_eligible"]
    .value_counts()
    .rename_axis("eligible")
    .to_frame("innings_count")
)

print("\nExcluded innings by reason:")
display(
    eligible_innings.loc[
        ~eligible_innings["primary_analysis_eligible"],
        ["match_id", "innings", "result",
         "is_regulation_innings",
         "is_completed_match_result",
         "has_reliable_chase_target"]
    ].groupby(
        ["is_regulation_innings",
         "is_completed_match_result",
         "has_reliable_chase_target"],
        dropna=False
    ).size().to_frame("innings_count")
)

print("\nSuper Over regulation innings retained:")
display(
    eligible_innings[
        eligible_innings["result"].astype(str).str.contains(
            "super over", case=False, na=False
        )
        & eligible_innings["primary_analysis_eligible"]
    ][["match_id", "innings", "calculated_total", "result"]]
)

print("\nFormat-level eligibility:")
display(
    eligible_innings.groupby("match_format")[
        "primary_analysis_eligible"
    ].agg(["count", "sum"])
)

Corrected eligibility counts:


,innings_count
eligible,
True,1685
False,39



Excluded innings by reason:


innings_count
is_regulation_innings is_completed_match_result has_reliable_chase_target               
False                 True                      True                                  12
True                  False                     True                                  26
                      True                      False                                  1


Super Over regulation innings retained:


,match_id,innings,calculated_total,result
539,21176,1,196,Sri Lanka won the super over
540,21176,2,196,Sri Lanka won the super over
645,28962,1,212,India Won Super Over By 10 Runs
646,28962,2,212,India Won Super Over By 10 Runs
888,34429,1,137,India Tied with Srilanka (India Won the Super ...
889,34429,2,137,India Tied with Srilanka (India Won the Super ...
1116,40725,1,188,Delhi Capitals Tied with Rajasthan Royals (DC ...
1117,40725,2,188,Delhi Capitals Tied with Rajasthan Royals (DC ...
1321,45512,1,202,India won the Super Over(Match Tied)
1322,45512,2,202,India won the Super Over(Match Tied)



Format-level eligibility:


,count,sum
match_format,,
T20,798,784
T20I,926,901


In [52]:
# Attach innings-level eligibility to the delivery-level dataset

eligibility_lookup = eligible_innings[
    [
        "match_id",
        "innings",
        "primary_analysis_eligible"
    ]
].copy()

# Confirm one eligibility record per match and innings
assert not eligibility_lookup.duplicated(
    ["match_id", "innings"]
).any(), "Duplicate innings keys found!"

ball_df = ball_df.merge(
    eligibility_lookup,
    on=["match_id", "innings"],
    how="left",
    validate="many_to_one"
)

# Audit the merge
print("Total delivery records:", len(ball_df))

print(
    "Deliveries with missing eligibility:",
    ball_df["primary_analysis_eligible"].isna().sum()
)

print("\nDelivery-level eligibility:")
display(
    ball_df["primary_analysis_eligible"]
    .value_counts(dropna=False)
    .rename_axis("primary_analysis_eligible")
    .to_frame("delivery_count")
)

print("\nEligible deliveries by format:")
eligible_delivery_summary = (
    ball_df.groupby(
        ["match_format", "primary_analysis_eligible"]
    )
    .size()
    .unstack(fill_value=0)
)

display(eligible_delivery_summary)

# Confirm original delivery count was preserved
assert len(ball_df) == 200999
assert ball_df["primary_analysis_eligible"].notna().all()

print("\nMerge validation passed.")

Total delivery records: 200999
Deliveries with missing eligibility: 0

Delivery-level eligibility:


,delivery_count
primary_analysis_eligible,
True,199152
False,1847



Eligible deliveries by format:


KeyError: 'match_format'

In [53]:
# Build a match-to-format lookup using match and competition metadata

match_format_lookup = (
    matches_df[["match_id", "comp_id"]]
    .merge(
        competitions_df[["comp_id", "match_format"]],
        on="comp_id",
        how="left",
        validate="many_to_one"
    )[["match_id", "match_format"]]
)

# Confirm one format record per match
assert not match_format_lookup["match_id"].duplicated().any()

# Attach format to ball-by-ball data
ball_df = ball_df.merge(
    match_format_lookup,
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Audit
print("Total delivery records:", len(ball_df))
print(
    "Missing match formats:",
    ball_df["match_format"].isna().sum()
)

print("\nEligible deliveries by format:")
eligible_delivery_summary = (
    ball_df.groupby(
        ["match_format", "primary_analysis_eligible"]
    )
    .size()
    .unstack(fill_value=0)
)

display(eligible_delivery_summary)

# Confirm no delivery records were duplicated or lost
assert len(ball_df) == 200999
assert ball_df["match_format"].notna().all()

print("\nMatch-format merge validation passed.")

Total delivery records: 200999
Missing match formats: 0

Eligible deliveries by format:


primary_analysis_eligible,False,True
match_format,,
T20,482,94190
T20I,1365,104962



Match-format merge validation passed.


In [54]:
# Wicket and dismissal audit on the primary analysis population

eligible_df = ball_df[
    ball_df["primary_analysis_eligible"]
].copy()

# 1. Wicket totals
print("Eligible delivery records:", len(eligible_df))
print("Wicket deliveries:", eligible_df["is_wicket"].sum())

print(
    "Wicket deliveries with missing dismissal type:",
    eligible_df.loc[
        eligible_df["is_wicket"],
        "dismissal_type"
    ].isna().sum()
)

print(
    "Wicket deliveries with missing dismissed player ID:",
    eligible_df.loc[
        eligible_df["is_wicket"],
        "dismissed_player_id"
    ].isna().sum()
)

# 2. Dismissal type distribution
print("\nDismissal type distribution:")

dismissal_summary = (
    eligible_df.loc[eligible_df["is_wicket"]]
    .groupby("dismissal_type", dropna=False)
    .agg(
        wicket_deliveries=("is_wicket", "size"),
        unique_dismissed_players=("dismissed_player_id", "nunique")
    )
    .sort_values("wicket_deliveries", ascending=False)
)

display(dismissal_summary)

# 3. Check for inconsistent wicket flags and dismissal labels
non_wicket_with_type = eligible_df[
    (~eligible_df["is_wicket"])
    & eligible_df["dismissal_type"].notna()
]

wicket_without_type = eligible_df[
    eligible_df["is_wicket"]
    & eligible_df["dismissal_type"].isna()
]

print("\nConsistency checks:")
print("Non-wicket deliveries with a dismissal type:", len(non_wicket_with_type))
print("Wicket deliveries without a dismissal type:", len(wicket_without_type))

# 4. Inspect any exceptional wicket records
print("\nWicket records with missing dismissed player ID:")

display(
    eligible_df.loc[
        eligible_df["is_wicket"]
        & eligible_df["dismissed_player_id"].isna(),
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "dismissal_type"
        ]
    ]
)

Eligible delivery records: 199152
Wicket deliveries: 10778
Wicket deliveries with missing dismissal type: 0
Wicket deliveries with missing dismissed player ID: 1

Dismissal type distribution:


,wicket_deliveries,unique_dismissed_players
dismissal_type,,
Caught,7482,722
Bowled,1758,517
LBW,706,347
RunOut,594,318
Stump,212,156
Hit Wicket,17,14
Retire Out,6,5
Obstracting the field,2,2
Obstructing the field,1,1



Consistency checks:
Non-wicket deliveries with a dismissal type: 0
Wicket deliveries without a dismissal type: 0

Wicket records with missing dismissed player ID:


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,dismissal_type
127899,40709,2,19,5,Hardik Pandya,Shardul Thakur,Retire Out


In [55]:
# Create a normalized dismissal label for classification

dismissal_normalized = (
    eligible_df["dismissal_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Standard bowler-credited dismissal types
bowler_credited_types = {
    "caught",
    "bowled",
    "lbw",
    "stump",
    "hit wicket"
}

# Create bowler-credit flag
eligible_df["bowler_credited_wicket"] = (
    eligible_df["is_wicket"]
    & dismissal_normalized.isin(bowler_credited_types)
).astype(int)

# Audit the classification
bowler_wicket_audit = (
    eligible_df.loc[eligible_df["is_wicket"]]
    .groupby("dismissal_type", dropna=False)
    .agg(
        total_wickets=("is_wicket", "size"),
        bowler_credited_wickets=("bowler_credited_wicket", "sum")
    )
)

display(bowler_wicket_audit)

print("\nTotal team wickets:", int(eligible_df["is_wicket"].sum()))
print(
    "Bowler-credited wickets:",
    int(eligible_df["bowler_credited_wicket"].sum())
)
print(
    "Non-bowler-credited wickets:",
    int(
        eligible_df["is_wicket"].sum()
        - eligible_df["bowler_credited_wicket"].sum()
    )
)

# Verify no non-wicket delivery receives bowler credit
assert (
    eligible_df.loc[
        ~eligible_df["is_wicket"],
        "bowler_credited_wicket"
    ].sum() == 0
)

print("\nBowler-wicket classification validation passed.")

,total_wickets,bowler_credited_wickets
dismissal_type,,
Bowled,1758,1758
Caught,7482,7482
Hit Wicket,17,17
LBW,706,706
Obstracting the field,2,0
Obstructing the field,1,0
Retire Out,6,0
RunOut,594,0
Stump,212,212



Total team wickets: 10778
Bowler-credited wickets: 10175
Non-bowler-credited wickets: 603

Bowler-wicket classification validation passed.


In [56]:
# Audit extras specifically within the eligible analysis population

extras_audit_eligible = (
    eligible_df
    .groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        total_delivery_runs=("total_delivery_runs", "sum"),
        wickets=("is_wicket", "sum")
    )
    .sort_values("deliveries", ascending=False)
)

display(extras_audit_eligible)

print("\nEligible extras totals:")
print(
    "Batter runs:",
    int(eligible_df["runs"].sum())
)
print(
    "Extras1 runs:",
    int(eligible_df["extras1_runs"].sum())
)
print(
    "Extras2 runs:",
    int(eligible_df["extras_2_runs"].sum())
)
print(
    "Total delivery runs:",
    int(eligible_df["total_delivery_runs"].sum())
)

print(
    "\nEligible deliveries with unresolved extras type:",
    int(
        eligible_df["extras_type"].isin(["other", "Penalty"]).sum()
        + (
            eligible_df["extras_type"].isna()
            & (
                (eligible_df["extras1_runs"] + eligible_df["extras_2_runs"]) > 0
            )
        ).sum()
    )
)

,deliveries,batter_runs,extras1_runs,extras2_runs,total_delivery_runs,wickets
extras_type,,,,,,
NaN,187418,265886,1,1,265888,10729
wide,7655,0,9088,0,9088,40
legbyes,2721,1,3566,0,3567,0
Overstep,568,842,568,48,1458,5
byes,529,0,1000,0,1000,1
other,260,404,260,35,699,3
Penalty,1,0,5,0,5,0



Eligible extras totals:
Batter runs: 267133
Extras1 runs: 14488
Extras2 runs: 84
Total delivery runs: 281705

Eligible deliveries with unresolved extras type: 263


In [57]:
# Inspect unresolved extras before defining bowler runs conceded

unresolved_extras = eligible_df[
    eligible_df["extras_type"].isin(["other", "Penalty"])
    | (
        eligible_df["extras_type"].isna()
        & (
            (eligible_df["extras1_runs"] + eligible_df["extras_2_runs"]) > 0
        )
    )
].copy()

print("Unresolved deliveries:", len(unresolved_extras))

print("\nBreakdown by extras type:")
display(
    unresolved_extras.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        batter_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        total_runs=("total_delivery_runs", "sum"),
        wickets=("is_wicket", "sum")
    )
)

print("\nUnresolved records — inspect all rows:")
display(
    unresolved_extras[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "total_delivery_runs",
            "dismissal_type"
        ]
    ].sort_values(["extras_type", "match_id", "innings", "overid", "over_ball_num"])
)

Unresolved deliveries: 263

Breakdown by extras type:


,deliveries,batter_runs,extras1_runs,extras2_runs,total_runs,wickets
extras_type,,,,,,
Penalty,1,0,5,0,5,0
other,260,404,260,35,699,3
NaN,2,0,1,1,2,0



Unresolved records — inspect all rows:


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,total_delivery_runs,dismissal_type
34203,16753,2,6,4,Harry Tector,Fareed Malik,0,Penalty,5,0,5,NaN
532,13513,1,10,1,Rovman Powell,Tymal Mills,0,other,1,0,1,NaN
1079,13515,1,16,6,Rovman Powell,Saqib Mahmood,0,other,1,0,1,NaN
1226,13515,2,19,5,Sam Billings,Sheldon Cottrell,2,other,1,0,3,NaN
2588,13614,1,18,5,Dasun Shanaka,Mitchell Starc,0,other,1,4,5,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
198534,54213,1,16,2,Glenn Phillips,Nitish Reddy,4,other,1,0,5,NaN
199755,55078,1,20,4,Eshan Malinga,Razaullah Razaullah,0,other,1,1,2,NaN
200926,55561,1,12,5,Newman Nyamhuri,Abdullah Ahmadzai,0,other,1,0,1,NaN
36854,17555,1,5,4,Dawid Malan,Haris Rauf,0,NaN,0,1,1,NaN


In [58]:
# Calculate confirmed bowler runs without guessing unresolved extras

category = (
    eligible_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

eligible_df["confirmed_bowler_extras"] = 0

# All wide runs are charged to the bowler
wide_mask = category == "wide"
eligible_df.loc[
    wide_mask,
    "confirmed_bowler_extras"
] = eligible_df.loc[wide_mask, "extras1_runs"]

# Overstep is identified as a no-ball in the project master plan.
# Its extras1_runs represent the one-run no-ball penalty.
no_ball_mask = category == "overstep"
eligible_df.loc[
    no_ball_mask,
    "confirmed_bowler_extras"
] = eligible_df.loc[no_ball_mask, "extras1_runs"]

# Batter runs are charged to the bowler.
eligible_df["confirmed_bowler_runs"] = (
    eligible_df["runs"]
    + eligible_df["confirmed_bowler_extras"]
)

# Summarize known and unresolved components
print("Confirmed batter runs:", int(eligible_df["runs"].sum()))
print(
    "Confirmed bowler extras:",
    int(eligible_df["confirmed_bowler_extras"].sum())
)
print(
    "Confirmed bowler runs:",
    int(eligible_df["confirmed_bowler_runs"].sum())
)

print("\nUnresolved extra runs not included in confirmed bowler runs:")

unresolved_extra_summary = (
    unresolved_extras
    .groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        extras1_runs=("extras1_runs", "sum"),
        extras2_runs=("extras_2_runs", "sum"),
        team_runs=("total_delivery_runs", "sum")
    )
)

display(unresolved_extra_summary)

print(
    "\nConfirmed bowler runs are a lower-bound measure,",
    "not yet the final economy-rate input."
)

Confirmed batter runs: 267133
Confirmed bowler extras: 9656
Confirmed bowler runs: 276789

Unresolved extra runs not included in confirmed bowler runs:


,deliveries,extras1_runs,extras2_runs,team_runs
extras_type,,,,
Penalty,1,5,0,5
other,260,260,35,699
NaN,2,1,1,2



Confirmed bowler runs are a lower-bound measure, not yet the final economy-rate input.


In [59]:
# Investigate the meaning and distribution of extras2_runs

print("Eligible deliveries with extras2_runs > 0:")

extras2_detail = eligible_df[
    eligible_df["extras_2_runs"] > 0
].copy()

print("Number of records:", len(extras2_detail))
print("Total extras2 runs:", int(extras2_detail["extras_2_runs"].sum()))

print("\nDistribution by extras type:")

display(
    extras2_detail.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        extras2_runs=("extras_2_runs", "sum"),
        batter_runs=("runs", "sum"),
        extras1_runs=("extras1_runs", "sum"),
        total_runs=("total_delivery_runs", "sum")
    )
)

print("\nDetailed records:")

display(
    extras2_detail[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras1_runs",
            "extras_2_runs",
            "total_delivery_runs"
        ]
    ].sort_values(["extras_type", "match_id", "innings"])
)

Eligible deliveries with extras2_runs > 0:
Number of records: 36
Total extras2 runs: 84

Distribution by extras type:


,deliveries,extras2_runs,batter_runs,extras1_runs,total_runs
extras_type,,,,,
Overstep,21,48,0,21,69
other,14,35,0,14,49
NaN,1,1,0,0,1



Detailed records:


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,runs,extras_type,extras1_runs,extras_2_runs,total_delivery_runs
1485,13523,1,1,4,Ishan Kishan,Akeal Hosein,0,Overstep,1,4,5
7607,14112,1,11,5,David Warner,Varun Chakravarthy,0,Overstep,1,4,5
24453,15796,2,14,4,Lorcan Tucker,Umran Malik,0,Overstep,1,1,2
28010,16424,2,6,1,Anamul Haque,Tanaka Chivanga,0,Overstep,1,4,5
47440,20186,2,20,2,Krishnappa Gowtham,Tushar Deshpande,0,Overstep,1,1,2
50553,20199,2,10,4,Nitish Rana,Marco Jansen,0,Overstep,1,1,2
50935,20201,1,11,5,Krunal Pandya,Sikandar Raza,0,Overstep,1,4,5
55141,20218,1,16,5,Marcus Stoinis,Kagiso Rabada,0,Overstep,1,1,2
83106,30068,1,6,2,David Warner,Mustafizur Rahman,0,Overstep,1,1,2
84026,30071,2,15,3,Tristan Stubbs,Varun Chakravarthy,0,Overstep,1,4,5


In [60]:
# Lower and upper bounds for bowler runs conceded

category = (
    eligible_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Start from the confirmed lower-bound measure
eligible_df["bowler_runs_lower"] = (
    eligible_df["confirmed_bowler_runs"]
)

# Identify extra components whose bowler treatment is unresolved:
# 1. All extras on "other" records
# 2. All extras on "Penalty" records
# 3. Positive extras on records with missing extras_type
# 4. extras2_runs on Overstep records
unresolved_other = category == "other"
unresolved_penalty = category == "penalty"
unresolved_missing = (
    (category == "")
    & (
        (eligible_df["extras1_runs"] + eligible_df["extras_2_runs"]) > 0
    )
)
unresolved_overstep_extras2 = (
    (category == "overstep")
    & (eligible_df["extras_2_runs"] > 0)
)

uncertain_extra_mask = (
    unresolved_other
    | unresolved_penalty
    | unresolved_missing
    | unresolved_overstep_extras2
)

eligible_df["potentially_chargeable_unresolved_extras"] = 0

eligible_df.loc[
    uncertain_extra_mask,
    "potentially_chargeable_unresolved_extras"
] = (
    eligible_df.loc[uncertain_extra_mask, "extras1_runs"]
    + eligible_df.loc[uncertain_extra_mask, "extras_2_runs"]
)

# Upper bound assumes every unresolved extra component is charged
eligible_df["bowler_runs_upper"] = (
    eligible_df["bowler_runs_lower"]
    + eligible_df["potentially_chargeable_unresolved_extras"]
)

# Summary
print("Confirmed bowler runs (lower bound):",
      int(eligible_df["bowler_runs_lower"].sum()))

print("Potentially chargeable unresolved extras:",
      int(eligible_df["potentially_chargeable_unresolved_extras"].sum()))

print("Bowler runs (upper bound):",
      int(eligible_df["bowler_runs_upper"].sum()))

print("\nUncertainty by extras category:")

uncertainty_summary = (
    eligible_df.loc[uncertain_extra_mask]
    .groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        potentially_chargeable_extras=(
            "potentially_chargeable_unresolved_extras", "sum"
        )
    )
)

display(uncertainty_summary)

# Validation
assert (
    eligible_df["bowler_runs_upper"]
    >= eligible_df["bowler_runs_lower"]
).all()

print("\nBowler-runs bounds validation passed.")

Confirmed bowler runs (lower bound): 276789
Potentially chargeable unresolved extras: 371
Bowler runs (upper bound): 277160

Uncertainty by extras category:


,deliveries,potentially_chargeable_extras
extras_type,,
Overstep,21,69
Penalty,1,5
other,260,295
NaN,2,2



Bowler-runs bounds validation passed.


In [61]:
# Correct unresolved extras calculation without double-counting Overstep penalties

category = (
    eligible_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

unresolved_other = category == "other"
unresolved_penalty = category == "penalty"

unresolved_missing = (
    (category == "")
    & (
        (eligible_df["extras1_runs"] + eligible_df["extras_2_runs"]) > 0
    )
)

unresolved_overstep_extras2 = (
    (category == "overstep")
    & (eligible_df["extras_2_runs"] > 0)
)

uncertain_extra_mask = (
    unresolved_other
    | unresolved_penalty
    | unresolved_missing
    | unresolved_overstep_extras2
)

eligible_df["potentially_chargeable_unresolved_extras"] = 0

# For Overstep, extras1_runs is already included in the lower bound.
eligible_df.loc[
    unresolved_overstep_extras2,
    "potentially_chargeable_unresolved_extras"
] = eligible_df.loc[
    unresolved_overstep_extras2,
    "extras_2_runs"
]

# For other, Penalty and missing-category records, neither extras field
# is included in the confirmed lower bound.
other_unresolved_mask = (
    unresolved_other | unresolved_penalty | unresolved_missing
)

eligible_df.loc[
    other_unresolved_mask,
    "potentially_chargeable_unresolved_extras"
] = (
    eligible_df.loc[other_unresolved_mask, "extras1_runs"]
    + eligible_df.loc[other_unresolved_mask, "extras_2_runs"]
)

eligible_df["bowler_runs_upper"] = (
    eligible_df["bowler_runs_lower"]
    + eligible_df["potentially_chargeable_unresolved_extras"]
)

print(
    "Confirmed bowler runs (lower bound):",
    int(eligible_df["bowler_runs_lower"].sum())
)

print(
    "Unresolved extras:",
    int(eligible_df["potentially_chargeable_unresolved_extras"].sum())
)

print(
    "Bowler runs (upper bound):",
    int(eligible_df["bowler_runs_upper"].sum())
)

print("\nCorrected uncertainty by category:")

display(
    eligible_df.loc[uncertain_extra_mask]
    .groupby("extras_type", dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        unresolved_runs=(
            "potentially_chargeable_unresolved_extras", "sum"
        )
    )
)

assert (
    eligible_df["bowler_runs_upper"]
    >= eligible_df["bowler_runs_lower"]
).all()

assert eligible_df["potentially_chargeable_unresolved_extras"].sum() == 350

print("\nCorrected bowler-runs bounds validated.")

Confirmed bowler runs (lower bound): 276789
Unresolved extras: 350
Bowler runs (upper bound): 277139

Corrected uncertainty by category:


,deliveries,unresolved_runs
extras_type,,
Overstep,21,48
Penalty,1,5
other,260,295
NaN,2,2



Corrected bowler-runs bounds validated.


In [62]:
# Calculate minimum and maximum legal balls bowled by each bowler

eligible_df["legal_ball_min_flag"] = (
    eligible_df["legal_status"].isin(["legal", "legal_assumed"])
).astype(int)

eligible_df["legal_ball_uncertain_flag"] = (
    eligible_df["legal_status"] == "unresolved"
).astype(int)

eligible_df["legal_ball_max_flag"] = (
    eligible_df["legal_ball_min_flag"]
    + eligible_df["legal_ball_uncertain_flag"]
)

# Aggregate at bowler level
bowler_ball_audit = (
    eligible_df.groupby(["bowler_id", "bowler_name"], dropna=False)
    .agg(
        deliveries=("match_id", "size"),
        legal_balls_min=("legal_ball_min_flag", "sum"),
        unresolved_deliveries=("legal_ball_uncertain_flag", "sum"),
        legal_balls_max=("legal_ball_max_flag", "sum"),
        illegal_deliveries=(
            "legal_status",
            lambda s: (s == "illegal").sum()
        )
    )
    .reset_index()
)

# Sort for inspection
bowler_ball_audit = bowler_ball_audit.sort_values(
    "deliveries",
    ascending=False
).reset_index(drop=True)

print("Unique bowlers:", bowler_ball_audit["bowler_id"].nunique())
print("Total bowler delivery records:", bowler_ball_audit["deliveries"].sum())

print("\nLegal-ball audit totals:")
print(
    "Minimum legal balls:",
    int(bowler_ball_audit["legal_balls_min"].sum())
)
print(
    "Unresolved deliveries:",
    int(bowler_ball_audit["unresolved_deliveries"].sum())
)
print(
    "Maximum legal balls:",
    int(bowler_ball_audit["legal_balls_max"].sum())
)
print(
    "Illegal deliveries:",
    int(bowler_ball_audit["illegal_deliveries"].sum())
)

print("\nTop 15 bowlers by delivery volume:")
display(bowler_ball_audit.head(15))

# Validation: each delivery must belong to exactly one status group
assert (
    bowler_ball_audit["deliveries"].sum()
    == len(eligible_df)
)

assert (
    bowler_ball_audit["legal_balls_max"]
    >= bowler_ball_audit["legal_balls_min"]
).all()

assert (
    bowler_ball_audit["legal_balls_min"].sum()
    == eligible_df["legal_ball_min_flag"].sum()
)

assert (
    bowler_ball_audit["unresolved_deliveries"].sum()
    == eligible_df["legal_ball_uncertain_flag"].sum()
)

print("\nBowler legal-ball aggregation validated.")

Unique bowlers: 647
Total bowler delivery records: 199152

Legal-ball audit totals:
Minimum legal balls: 190666
Unresolved deliveries: 263
Maximum legal balls: 190929
Illegal deliveries: 8223

Top 15 bowlers by delivery volume:


,bowler_id,bowler_name,deliveries,legal_balls_min,unresolved_deliveries,legal_balls_max,illegal_deliveries
0,6464,Arshdeep Singh,3414,3190,2,3192,222
1,4722,Rashid Khan,2780,2747,0,2747,33
2,2876,Axar Patel,2695,2658,0,2658,37
3,7503,Ravi Bishnoi,2429,2327,3,2330,99
4,360,Bhuvneshwar Kumar,2314,2221,1,2222,92
5,19316,Maheesh Theekshana,2297,2240,0,2240,57
6,1612,Yuzvendra Chahal,2218,2138,0,2138,80
7,6256,Varun Chakravarthy,2158,2104,0,2104,54
8,4900,Wanindu Hasaranga,2053,2034,0,2034,19
9,2633,Jason Holder,2032,1928,1,1929,103



Bowler legal-ball aggregation validated.


In [63]:
# Inspect bye/leg-bye deliveries where the batter-runs field is non-zero

bye_anomaly = eligible_df.loc[
    eligible_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["legbyes", "byes"])
    & (eligible_df["runs"] > 0),
    [
        "match_id",
        "innings",
        "overid",
        "over_ball_num",
        "striker_name",
        "bowler_name",
        "extras_type",
        "runs",
        "extras1_runs",
        "extras_2_runs",
        "total_delivery_runs",
        "is_four",
        "is_six"
    ]
]

print("Bye/leg-bye deliveries with non-zero batter runs:", len(bye_anomaly))
display(bye_anomaly)

Bye/leg-bye deliveries with non-zero batter runs: 1


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,extras_type,runs,extras1_runs,extras_2_runs,total_delivery_runs,is_four,is_six
113765,37099,1,14,1,Nimesh Vimukthi,Qais Ahmad,legbyes,1,0,0,1,False,False


In [64]:
# Flag the isolated bye/leg-bye scoring inconsistency

category = (
    eligible_df["extras_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

bye_run_anomaly_mask = (
    category.isin(["byes", "legbyes"])
    & (eligible_df["runs"] > 0)
)

eligible_df["ambiguous_bye_runs"] = 0

eligible_df.loc[
    bye_run_anomaly_mask,
    "ambiguous_bye_runs"
] = eligible_df.loc[bye_run_anomaly_mask, "runs"]

# Recalculate the lower bound:
# exclude runs on bye/leg-bye deliveries from bowler-conceded runs.
eligible_df["bowler_runs_lower"] = (
    eligible_df["confirmed_bowler_runs"]
    - eligible_df["ambiguous_bye_runs"]
)

# Recalculate the upper bound:
# include all unresolved extras and the ambiguous bye/leg-bye run.
eligible_df["bowler_runs_upper"] = (
    eligible_df["bowler_runs_lower"]
    + eligible_df["potentially_chargeable_unresolved_extras"]
    + eligible_df["ambiguous_bye_runs"]
)

print("Bowler runs lower bound:",
      int(eligible_df["bowler_runs_lower"].sum()))

print("Unresolved extras:",
      int(eligible_df["potentially_chargeable_unresolved_extras"].sum()))

print("Ambiguous bye/leg-bye runs:",
      int(eligible_df["ambiguous_bye_runs"].sum()))

print("Bowler runs upper bound:",
      int(eligible_df["bowler_runs_upper"].sum()))

print("\nFlagged delivery:")
display(
    eligible_df.loc[
        bye_run_anomaly_mask,
        [
            "match_id", "innings", "overid", "over_ball_num",
            "striker_name", "bowler_name", "extras_type",
            "runs", "ambiguous_bye_runs"
        ]
    ]
)

assert (
    eligible_df["bowler_runs_upper"]
    >= eligible_df["bowler_runs_lower"]
).all()

assert eligible_df["ambiguous_bye_runs"].sum() == 1

print("\nBowler-runs bounds updated and validated.")

Bowler runs lower bound: 276788
Unresolved extras: 350
Ambiguous bye/leg-bye runs: 1
Bowler runs upper bound: 277139

Flagged delivery:


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,extras_type,runs,ambiguous_bye_runs
113765,37099,1,14,1,Nimesh Vimukthi,Qais Ahmad,legbyes,1,1



Bowler-runs bounds updated and validated.


In [65]:
# Aggregate runs and wickets for each bowler

bowler_run_audit = (
    eligible_df.groupby(["bowler_id", "bowler_name"], dropna=False)
    .agg(
        bowler_runs_min=("bowler_runs_lower", "sum"),
        bowler_runs_max=("bowler_runs_upper", "sum"),
        bowler_credited_wickets=("bowler_credited_wicket", "sum")
    )
    .reset_index()
)

# Combine runs, legal balls and delivery counts
bowler_stats = bowler_ball_audit.merge(
    bowler_run_audit,
    on=["bowler_id", "bowler_name"],
    how="inner",
    validate="one_to_one"
)

# Economy-rate bounds
bowler_stats["economy_min"] = (
    bowler_stats["bowler_runs_min"]
    / bowler_stats["legal_balls_max"]
    * 6
)

bowler_stats["economy_max"] = (
    bowler_stats["bowler_runs_max"]
    / bowler_stats["legal_balls_min"]
    * 6
)

# Overs represented as legal balls / 6
bowler_stats["overs_min"] = (
    bowler_stats["legal_balls_min"] / 6
)

bowler_stats["overs_max"] = (
    bowler_stats["legal_balls_max"] / 6
)

# Basic validation
assert len(bowler_stats) == 647
assert bowler_stats["bowler_runs_min"].sum() == 276788
assert bowler_stats["bowler_runs_max"].sum() == 277139
assert bowler_stats["legal_balls_min"].sum() == 190666
assert bowler_stats["legal_balls_max"].sum() == 190929
assert (bowler_stats["economy_max"] >= bowler_stats["economy_min"]).all()

print("Bowler-level records:", len(bowler_stats))
print("Total bowler runs lower bound:", bowler_stats["bowler_runs_min"].sum())
print("Total bowler runs upper bound:", bowler_stats["bowler_runs_max"].sum())

print("\nTop 20 bowlers by legal balls bowled:")
display(
    bowler_stats.sort_values(
        "legal_balls_min",
        ascending=False
    )[
        [
            "bowler_id",
            "bowler_name",
            "deliveries",
            "legal_balls_min",
            "legal_balls_max",
            "bowler_runs_min",
            "bowler_runs_max",
            "bowler_credited_wickets",
            "economy_min",
            "economy_max"
        ]
    ].head(20).round(3)
)

print("\nBowler-level performance table validated.")

Bowler-level records: 647
Total bowler runs lower bound: 276788
Total bowler runs upper bound: 277139

Top 20 bowlers by legal balls bowled:


,bowler_id,bowler_name,deliveries,legal_balls_min,legal_balls_max,bowler_runs_min,bowler_runs_max,bowler_credited_wickets,economy_min,economy_max
0,6464,Arshdeep Singh,3414,3190,3192,4862,4864,192,9.139,9.149
1,4722,Rashid Khan,2780,2747,2747,3403,3403,153,7.433,7.433
2,2876,Axar Patel,2695,2658,2658,3487,3487,120,7.871,7.871
3,7503,Ravi Bishnoi,2429,2327,2330,3234,3237,124,8.328,8.346
5,19316,Maheesh Theekshana,2297,2240,2240,2891,2891,91,7.744,7.744
4,360,Bhuvneshwar Kumar,2314,2221,2222,3068,3069,115,8.284,8.291
6,1612,Yuzvendra Chahal,2218,2138,2138,3071,3071,124,8.618,8.618
7,6256,Varun Chakravarthy,2158,2104,2104,2794,2802,134,7.968,7.990
8,4900,Wanindu Hasaranga,2053,2034,2034,2690,2690,133,7.935,7.935
9,2633,Jason Holder,2032,1928,1929,2891,2892,109,8.992,9.000



Bowler-level performance table validated.


In [66]:
# Measure how much unresolved data affects each bowler's statistics

bowler_stats["economy_interval_width"] = (
    bowler_stats["economy_max"] - bowler_stats["economy_min"]
)

bowler_stats["runs_interval_width"] = (
    bowler_stats["bowler_runs_max"] - bowler_stats["bowler_runs_min"]
)

bowler_stats["legal_ball_uncertainty_pct"] = (
    bowler_stats["unresolved_deliveries"]
    / bowler_stats["legal_balls_min"].replace(0, float("nan"))
) * 100

# Number of distinct matches in which each bowler bowled
bowler_match_counts = (
    eligible_df.groupby(["bowler_id", "bowler_name"])["match_id"]
    .nunique()
    .reset_index(name="matches_bowled")
)

bowler_stats = bowler_stats.merge(
    bowler_match_counts,
    on=["bowler_id", "bowler_name"],
    how="left",
    validate="one_to_one"
)

print("Economy interval width summary:")
display(
    bowler_stats["economy_interval_width"]
    .describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])
    .to_frame("economy_interval_width")
    .round(5)
)

print("\nBowlers with the widest economy uncertainty:")
display(
    bowler_stats.sort_values(
        "economy_interval_width",
        ascending=False
    )[
        [
            "bowler_id",
            "bowler_name",
            "matches_bowled",
            "deliveries",
            "legal_balls_min",
            "legal_balls_max",
            "unresolved_deliveries",
            "bowler_runs_min",
            "bowler_runs_max",
            "economy_min",
            "economy_max",
            "economy_interval_width"
        ]
    ].head(20).round(4)
)

print("\nUncertainty coverage:")
print(
    "Bowlers with at least one unresolved delivery:",
    int((bowler_stats["unresolved_deliveries"] > 0).sum())
)
print(
    "Bowlers with economy interval width above 0.05:",
    int((bowler_stats["economy_interval_width"] > 0.05).sum())
)
print(
    "Bowlers with economy interval width above 0.10:",
    int((bowler_stats["economy_interval_width"] > 0.10).sum())
)

print("\nUncertainty audit completed.")

Economy interval width summary:


,economy_interval_width
count,647.00000
mean,0.03165
std,0.11519
min,0.00000
50%,0.00000
75%,0.00000
90%,0.07195
95%,0.16799
99%,0.59402
max,1.21569



Bowlers with the widest economy uncertainty:


,bowler_id,bowler_name,matches_bowled,deliveries,legal_balls_min,legal_balls_max,unresolved_deliveries,bowler_runs_min,bowler_runs_max,economy_min,economy_max,economy_interval_width
569,22565,Muhammad Waseem,2,19,17,18,1,44,45,14.6667,15.8824,1.2157
566,820,Nathan Coulter Nile,1,20,18,19,1,47,48,14.8421,16.0000,1.1579
555,17919,Mohammad Saleem Safi,1,22,18,19,1,42,43,13.2632,14.3333,1.0702
559,20989,Vivrant Sharma,2,21,18,19,1,36,37,11.3684,12.3333,0.9649
520,21786,Tanaka Chivanga,2,31,27,27,0,47,51,10.4444,11.3333,0.8889
420,6162,Kulwant Khejroliya,4,62,56,58,2,122,124,12.6207,13.2857,0.6650
480,71727,Razaullah Razaullah,2,39,37,38,1,65,67,10.2632,10.8649,0.6017
438,34522,Brandon McMullen,3,52,48,50,2,67,69,8.0400,8.6250,0.5850
464,20229,Mohammad Ali,3,44,42,43,1,85,86,11.8605,12.2857,0.4252
477,507,Mohammad Mahmudullah,5,41,38,39,1,56,57,8.6154,9.0000,0.3846



Uncertainty coverage:
Bowlers with at least one unresolved delivery: 146
Bowlers with economy interval width above 0.05: 88
Bowlers with economy interval width above 0.10: 51

Uncertainty audit completed.


In [67]:
# Build format-specific bowler performance records

required_columns = [
    "bowler_id", "bowler_name", "match_format", "match_id",
    "legal_ball_min_flag", "legal_ball_max_flag",
    "bowler_runs_lower", "bowler_runs_upper",
    "bowler_credited_wicket"
]

missing_columns = [
    col for col in required_columns
    if col not in eligible_df.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

bowler_format_stats = (
    eligible_df.groupby(
        ["bowler_id", "bowler_name", "match_format"],
        dropna=False
    )
    .agg(
        matches_bowled=("match_id", "nunique"),
        deliveries=("match_id", "size"),
        legal_balls_min=("legal_ball_min_flag", "sum"),
        legal_balls_max=("legal_ball_max_flag", "sum"),
        runs_min=("bowler_runs_lower", "sum"),
        runs_max=("bowler_runs_upper", "sum"),
        wickets=("bowler_credited_wicket", "sum")
    )
    .reset_index()
)

# Economy-rate bounds within each format
bowler_format_stats["economy_min"] = (
    bowler_format_stats["runs_min"]
    / bowler_format_stats["legal_balls_max"].replace(0, float("nan"))
    * 6
)

bowler_format_stats["economy_max"] = (
    bowler_format_stats["runs_max"]
    / bowler_format_stats["legal_balls_min"].replace(0, float("nan"))
    * 6
)

bowler_format_stats["economy_interval_width"] = (
    bowler_format_stats["economy_max"]
    - bowler_format_stats["economy_min"]
)

print("Format-specific bowler records:")
display(
    bowler_format_stats.groupby("match_format")
    .agg(
        bowlers=("bowler_id", "nunique"),
        bowler_format_records=("bowler_id", "size"),
        matches_bowled=("matches_bowled", "sum"),
        legal_balls_min=("legal_balls_min", "sum"),
        wickets=("wickets", "sum")
    )
)

print("\nSample-size distribution by format (matches bowled per bowler):")

for fmt in sorted(bowler_format_stats["match_format"].dropna().unique()):
    print(f"\n{fmt}")
    display(
        bowler_format_stats.loc[
            bowler_format_stats["match_format"] == fmt,
            "matches_bowled"
        ]
        .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
        .to_frame("matches_bowled")
        .round(2)
    )

print("\nLowest-volume format-specific records:")
display(
    bowler_format_stats.sort_values(
        ["match_format", "legal_balls_min"]
    )[
        [
            "bowler_id", "bowler_name", "match_format",
            "matches_bowled", "deliveries",
            "legal_balls_min", "wickets",
            "economy_min", "economy_max"
        ]
    ].head(20).round(3)
)

print("\nFormat-specific aggregation completed.")

Format-specific bowler records:


,bowlers,bowler_format_records,matches_bowled,legal_balls_min,wickets
match_format,,,,,
T20,352,352,4671,90017,4543
T20I,479,479,5302,100649,5632



Sample-size distribution by format (matches bowled per bowler):

T20


,matches_bowled
count,352.00
mean,13.27
std,16.98
min,1.00
25%,2.00
50%,5.50
75%,18.00
90%,41.80
95%,52.45
max,77.00



T20I


,matches_bowled
count,479.00
mean,11.07
std,13.38
min,1.00
25%,2.00
50%,6.00
75%,15.00
90%,29.20
95%,40.10
max,75.00



Lowest-volume format-specific records:


,bowler_id,bowler_name,match_format,matches_bowled,deliveries,legal_balls_min,wickets,economy_min,economy_max
250,4117,Ishan Kishan,T20,1,1,1,0,24.000,24.000
422,6466,Atharwa Taide,T20,1,1,1,0,24.000,24.000
458,6819,Yashasvi Jaiswal,T20,1,1,1,0,36.000,36.000
237,3858,Jatinder Singh,T20,1,2,2,0,21.000,21.000
674,25213,Dewald Brevis,T20,1,3,3,1,16.000,16.000
55,1195,Ben Stokes,T20,1,6,6,0,18.000,18.000
87,2214,Gulbadin Naib,T20,1,6,6,0,12.000,12.000
187,3383,Kinchit Shah,T20,1,6,6,0,31.000,31.000
212,3647,Shreyas Iyer,T20,1,6,6,0,7.000,7.000
296,4779,Pieter Willem Mulder,T20,1,6,6,0,16.000,16.000



Format-specific aggregation completed.


In [68]:
# Compare candidate minimum sample requirements for bowlers

candidate_thresholds = [
    (5, 120),
    (10, 120),
    (10, 240),
    (15, 240),
    (15, 360),
    (20, 360)
]

threshold_results = []

for fmt in sorted(bowler_format_stats["match_format"].dropna().unique()):
    format_data = bowler_format_stats[
        bowler_format_stats["match_format"] == fmt
    ].copy()

    total_bowlers = len(format_data)

    for min_matches, min_legal_balls in candidate_thresholds:
        eligible_bowlers = format_data[
            (format_data["matches_bowled"] >= min_matches)
            & (format_data["legal_balls_min"] >= min_legal_balls)
        ]

        threshold_results.append({
            "match_format": fmt,
            "minimum_matches": min_matches,
            "minimum_legal_balls": min_legal_balls,
            "eligible_bowlers": len(eligible_bowlers),
            "excluded_bowlers": total_bowlers - len(eligible_bowlers),
            "eligible_percentage": (
                len(eligible_bowlers) / total_bowlers * 100
            )
        })

threshold_comparison = pd.DataFrame(threshold_results)

print("Candidate sample-threshold comparison:")
display(
    threshold_comparison.round({
        "eligible_percentage": 2
    })
)

print("\nEligible bowlers under a provisional 10-match / 240-ball rule:")

for fmt in sorted(bowler_format_stats["match_format"].dropna().unique()):
    subset = bowler_format_stats[
        (bowler_format_stats["match_format"] == fmt)
        & (bowler_format_stats["matches_bowled"] >= 10)
        & (bowler_format_stats["legal_balls_min"] >= 240)
    ]

    print(f"{fmt}: {len(subset)} bowlers")
    print(
        "  Median matches:",
        round(subset["matches_bowled"].median(), 2)
        if len(subset) else "No qualifying bowlers"
    )
    print(
        "  Median legal balls:",
        round(subset["legal_balls_min"].median(), 2)
        if len(subset) else "No qualifying bowlers"
    )

print("\nThreshold coverage analysis completed.")

Candidate sample-threshold comparison:


,match_format,minimum_matches,minimum_legal_balls,eligible_bowlers,excluded_bowlers,eligible_percentage
0,T20,5,120,157,195,44.60
1,T20,10,120,125,227,35.51
2,T20,10,240,107,245,30.40
3,T20,15,240,92,260,26.14
4,T20,15,360,81,271,23.01
5,T20,20,360,75,277,21.31
6,T20I,5,120,211,268,44.05
7,T20I,10,120,164,315,34.24
8,T20I,10,240,129,350,26.93
9,T20I,15,240,115,364,24.01



Eligible bowlers under a provisional 10-match / 240-ball rule:
T20: 107 bowlers
  Median matches: 29.0
  Median legal balls: 561.0
T20I: 129 bowlers
  Median matches: 23.0
  Median legal balls: 460.0

Threshold coverage analysis completed.


In [69]:
# Identify which competitions contribute to the T20 sample

match_competition = matches_df[
    ["match_id", "comp_id"]
].drop_duplicates()

assert match_competition["match_id"].is_unique

eligible_competitions = eligible_df.merge(
    match_competition,
    on="match_id",
    how="left",
    validate="many_to_one"
)

eligible_competitions = eligible_competitions.merge(
    competitions_df[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_competition")
)

assert eligible_competitions["comp_name"].notna().all()

competition_audit = (
    eligible_competitions.groupby(
        ["comp_id", "comp_name", "match_format"],
        dropna=False
    )
    .agg(
        eligible_deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
    .sort_values(["match_format", "matches"], ascending=[True, False])
)

print("Competition coverage in eligible delivery data:")
display(competition_audit)

print("\nT20 competitions only:")
display(
    competition_audit[
        competition_audit["match_format"] == "T20"
    ].reset_index(drop=True)
)

Competition coverage in eligible delivery data:


,comp_id,comp_name,match_format,eligible_deliveries,matches,bowlers
3,1358,IPL 2022,T20,17909,74,125
31,1715,IPL 2023,T20,17745,73,136
118,3298,IPL 2026,T20,17501,73,125
53,2193,IPL 2024,T20,17103,71,137
84,2725,IPL 2025,T20,16959,71,126
...,...,...,...,...,...,...
127,3446,India in Ireland T20I Series 2026,T20I,496,2,14
88,2801,Ireland in Zimbabwe T20I Series 2025,T20I,243,1,12
89,2873,West Indies in Ireland T20I Series 2025,T20I,257,1,11
110,3226,England in New Zealand T20I Series 2025,T20I,242,1,11



T20 competitions only:


,comp_id,comp_name,match_format,eligible_deliveries,matches,bowlers
0,1358,IPL 2022,T20,17909,74,125
1,1715,IPL 2023,T20,17745,73,136
2,3298,IPL 2026,T20,17501,73,125
3,2193,IPL 2024,T20,17103,71,137
4,2725,IPL 2025,T20,16959,71,126
5,2670,Mens T20 Emerging Teams Asia Cup 2024,T20,3519,15,72
6,3289,Asia Cup Rising Stars 2025,T20,3454,15,70


In [70]:
# Restrict the primary analysis to IPL and T20I

ipl_competition_ids = [1358, 1715, 2193, 2725, 3298]

primary_scope_df = eligible_competitions.loc[
    (
        eligible_competitions["comp_id"].isin(ipl_competition_ids)
    )
    |
    (
        eligible_competitions["match_format"] == "T20I"
    )
].copy()

# Verify that all five IPL seasons are included
ipl_scope = primary_scope_df[
    primary_scope_df["comp_id"].isin(ipl_competition_ids)
]

t20i_scope = primary_scope_df[
    primary_scope_df["match_format"] == "T20I"
]

print("Primary analysis scope:")
print("IPL deliveries:", len(ipl_scope))
print("IPL matches:", ipl_scope["match_id"].nunique())
print("T20I deliveries:", len(t20i_scope))
print("T20I matches:", t20i_scope["match_id"].nunique())

print("\nIPL season coverage:")
display(
    ipl_scope.groupby(["comp_id", "comp_name"])
    .agg(
        eligible_deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
    .sort_values("comp_name")
)

print("\nExcluded non-IPL T20 competitions:")
display(
    competition_audit[
        (competition_audit["match_format"] == "T20")
        & (~competition_audit["comp_id"].isin(ipl_competition_ids))
    ]
)

# Build new format-specific bowler records from the correct scope
bowler_scope_stats = (
    primary_scope_df.groupby(
        ["bowler_id", "bowler_name", "match_format"],
        dropna=False
    )
    .agg(
        matches_bowled=("match_id", "nunique"),
        deliveries=("match_id", "size"),
        legal_balls_min=("legal_ball_min_flag", "sum"),
        legal_balls_max=("legal_ball_max_flag", "sum"),
        runs_min=("bowler_runs_lower", "sum"),
        runs_max=("bowler_runs_upper", "sum"),
        wickets=("bowler_credited_wicket", "sum")
    )
    .reset_index()
)

bowler_scope_stats["economy_min"] = (
    bowler_scope_stats["runs_min"]
    / bowler_scope_stats["legal_balls_max"].replace(0, float("nan"))
    * 6
)

bowler_scope_stats["economy_max"] = (
    bowler_scope_stats["runs_max"]
    / bowler_scope_stats["legal_balls_min"].replace(0, float("nan"))
    * 6
)

print("\nCandidate thresholds in the correct analysis scope:")

scope_threshold_results = []

for fmt in ["T20", "T20I"]:
    format_data = bowler_scope_stats[
        bowler_scope_stats["match_format"] == fmt
    ]

    for min_matches, min_balls in candidate_thresholds:
        qualifying = format_data[
            (format_data["matches_bowled"] >= min_matches)
            & (format_data["legal_balls_min"] >= min_balls)
        ]

        scope_threshold_results.append({
            "match_format": fmt,
            "minimum_matches": min_matches,
            "minimum_legal_balls": min_balls,
            "eligible_bowlers": len(qualifying),
            "eligible_percentage": (
                len(qualifying) / len(format_data) * 100
            )
        })

scope_threshold_comparison = pd.DataFrame(scope_threshold_results)

display(
    scope_threshold_comparison.round(
        {"eligible_percentage": 2}
    )
)

print("\nPrimary analysis scope validation:")
print("Formats:", sorted(primary_scope_df["match_format"].unique()))
print("Unique competitions:", primary_scope_df["comp_id"].nunique())

assert set(primary_scope_df["match_format"].unique()) == {"T20", "T20I"}
assert set(ipl_scope["comp_id"].unique()) == set(ipl_competition_ids)
assert not (
    primary_scope_df["comp_id"].isin([2670, 3289])
).any()

print("\nIPL + T20I analysis scope validated.")


Primary analysis scope:
IPL deliveries: 87217
IPL matches: 362
T20I deliveries: 104962
T20I matches: 451

IPL season coverage:


,comp_id,comp_name,eligible_deliveries,matches,bowlers
0,1358,IPL 2022,17909,74,125
1,1715,IPL 2023,17745,73,136
2,2193,IPL 2024,17103,71,137
3,2725,IPL 2025,16959,71,126
4,3298,IPL 2026,17501,73,125



Excluded non-IPL T20 competitions:


,comp_id,comp_name,match_format,eligible_deliveries,matches,bowlers
79,2670,Mens T20 Emerging Teams Asia Cup 2024,T20,3519,15,72
117,3289,Asia Cup Rising Stars 2025,T20,3454,15,70



Candidate thresholds in the correct analysis scope:


,match_format,minimum_matches,minimum_legal_balls,eligible_bowlers,eligible_percentage
0,T20,5,120,145,55.77
1,T20,10,120,123,47.31
2,T20,10,240,103,39.62
3,T20,15,240,89,34.23
4,T20,15,360,80,30.77
5,T20,20,360,75,28.85
6,T20I,5,120,211,44.05
7,T20I,10,120,164,34.24
8,T20I,10,240,129,26.93
9,T20I,15,240,115,24.01



Primary analysis scope validation:
Formats: ['T20', 'T20I']
Unique competitions: 136

IPL + T20I analysis scope validated.


In [71]:
# Apply the provisional bowling sample eligibility rule

MIN_BOWLING_MATCHES = 10
MIN_BOWLING_LEGAL_BALLS = 240

bowler_scope_stats["sample_eligible"] = (
    (bowler_scope_stats["matches_bowled"] >= MIN_BOWLING_MATCHES)
    & (bowler_scope_stats["legal_balls_min"] >= MIN_BOWLING_LEGAL_BALLS)
)

qualified_bowlers = bowler_scope_stats[
    bowler_scope_stats["sample_eligible"]
].copy()

# Separate IPL and T20I candidate pools
ipl_bowler_candidates = qualified_bowlers[
    qualified_bowlers["match_format"] == "T20"
].copy()

t20i_bowler_candidates = qualified_bowlers[
    qualified_bowlers["match_format"] == "T20I"
].copy()

print("Bowling sample eligibility rule:")
print(f"Minimum matches: {MIN_BOWLING_MATCHES}")
print(f"Minimum legal balls: {MIN_BOWLING_LEGAL_BALLS}")

print("\nQualified bowler candidates:")
print("IPL:", len(ipl_bowler_candidates))
print("T20I:", len(t20i_bowler_candidates))

print("\nIPL candidate sample summary:")
display(
    ipl_bowler_candidates[
        ["matches_bowled", "legal_balls_min", "wickets"]
    ].describe().round(2)
)

print("\nT20I candidate sample summary:")
display(
    t20i_bowler_candidates[
        ["matches_bowled", "legal_balls_min", "wickets"]
    ].describe().round(2)
)

# Verify that no ineligible bowler entered the candidate pools
assert (
    ipl_bowler_candidates["matches_bowled"] >= MIN_BOWLING_MATCHES
).all()

assert (
    ipl_bowler_candidates["legal_balls_min"] >= MIN_BOWLING_LEGAL_BALLS
).all()

assert (
    t20i_bowler_candidates["matches_bowled"] >= MIN_BOWLING_MATCHES
).all()

assert (
    t20i_bowler_candidates["legal_balls_min"] >= MIN_BOWLING_LEGAL_BALLS
).all()

assert len(ipl_bowler_candidates) == 103
assert len(t20i_bowler_candidates) == 129

print("\nQualified bowler pools created and validated.")

Bowling sample eligibility rule:
Minimum matches: 10
Minimum legal balls: 240

Qualified bowler candidates:
IPL: 103
T20I: 129

IPL candidate sample summary:


,matches_bowled,legal_balls_min,wickets
count,103.00,103.00,103.00
mean,34.29,691.25,35.13
std,17.65,399.44,20.77
min,11.00,240.00,5.00
25%,21.00,369.50,18.00
50%,29.00,561.00,28.00
75%,44.00,956.00,46.00
max,77.00,1718.00,94.00



T20I candidate sample summary:


,matches_bowled,legal_balls_min,wickets
count,129.00,129.00,129.00
mean,28.63,573.87,32.96
std,14.34,310.77,18.59
min,11.00,240.00,9.00
25%,17.00,340.00,20.00
50%,23.00,460.00,27.00
75%,37.00,755.00,40.00
max,75.00,1627.00,111.00



Qualified bowler pools created and validated.


In [72]:
# Audit target and revised-over metadata before defining bowling pressure

primary_match_ids = primary_scope_df["match_id"].drop_duplicates()

scope_matches = matches_df[
    matches_df["match_id"].isin(primary_match_ids)
].copy()

print("MATCH METADATA COVERAGE")
print("Matches in primary scope:", scope_matches["match_id"].nunique())

metadata_cols = [
    "first_inn_revised_overs",
    "second_inn_revised_overs",
    "revised_target"
]

print("\nNON-MISSING VALUES")
display(
    scope_matches[metadata_cols]
    .notna()
    .sum()
    .rename("non_missing_matches")
    .to_frame()
)

print("\nUNIQUE REVISED-OVER VALUES")
for col in ["first_inn_revised_overs", "second_inn_revised_overs"]:
    print(f"\n{col}:")
    print(
        scope_matches[col]
        .dropna()
        .value_counts()
        .sort_index()
        .to_string()
    )

print("\nREVISED TARGET VALUE SUMMARY")
display(
    scope_matches["revised_target"]
    .dropna()
    .describe()
    .to_frame("revised_target")
    .round(2)
)

# Join first-innings scores to match metadata
first_innings_scores = (
    innings_score_audit.loc[
        innings_score_audit["innings"] == 1,
        ["match_id", "calculated_total"]
    ]
    .rename(columns={"calculated_total": "first_innings_score"})
)

target_audit = scope_matches.merge(
    first_innings_scores,
    on="match_id",
    how="left",
    validate="one_to_one"
)

target_audit["ordinary_chase_target"] = (
    target_audit["first_innings_score"] + 1
)

print("\nMATCHES WITH REVISED TARGET OR REVISED OVERS")
revised_mask = target_audit[
    metadata_cols
].notna().any(axis=1)

display(
    target_audit.loc[
        revised_mask,
        [
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "result",
            "first_innings_score",
            "ordinary_chase_target",
            "first_inn_revised_overs",
            "second_inn_revised_overs",
            "revised_target"
        ]
    ].sort_values("match_date").head(30)
)

print("\nTARGET COMPARISON")
target_audit["target_diff"] = (
    target_audit["revised_target"]
    - target_audit["ordinary_chase_target"]
)

display(
    target_audit.loc[
        target_audit["revised_target"].notna(),
        [
            "match_id",
            "first_innings_score",
            "ordinary_chase_target",
            "revised_target",
            "target_diff"
        ]
    ].head(30)
)

MATCH METADATA COVERAGE
Matches in primary scope: 813

NON-MISSING VALUES


,non_missing_matches
first_inn_revised_overs,15
second_inn_revised_overs,22
revised_target,22



UNIQUE REVISED-OVER VALUES

first_inn_revised_overs:
first_inn_revised_overs
30.0     1
42.0     1
48.0     1
54.0     1
60.0     1
66.0     3
78.0     1
84.0     1
90.0     2
102.0    2
114.0    1

second_inn_revised_overs:
second_inn_revised_overs
30.0     2
42.0     2
48.0     3
60.0     2
66.0     1
78.0     1
84.0     1
90.0     5
102.0    2
114.0    3

REVISED TARGET VALUE SUMMARY


,revised_target
count,22.00
mean,122.27
std,45.04
min,33.00
25%,94.50
50%,120.50
75%,150.00
max,213.00



MATCHES WITH REVISED TARGET OR REVISED OVERS


,match_id,match_date,home_team,away_team,result,first_innings_score,ordinary_chase_target,first_inn_revised_overs,second_inn_revised_overs,revised_target
8,13612,2022-02-11,Australia,Sri Lanka,Australia Won by 20 runs (D/L method),149,150,NaN,114.0,143.0
141,16753,2022-08-15,Ireland,Afghanistan,Afghanistan Won by 27 Runs,132,133,66.0,NaN,NaN
142,16754,2022-08-17,Ireland,Afghanistan,Ireland Won by 7 Wickets(D/L Method),95,96,NaN,42.0,56.0
150,17395,2022-09-23,India,Australia,India Won by 6 Wickets,90,91,48.0,48.0,91.0
259,19959,2023-03-25,South Africa,West Indies,West Indies won by 3 wickets,131,132,66.0,NaN,NaN
268,21070,2023-03-27,Bangladesh,Ireland,Bangladesh Won by 22 Runs (DLS method),207,208,NaN,48.0,104.0
269,21071,2023-03-29,Bangladesh,Ireland,Bangladesh Won by 77 Runs,202,203,102.0,102.0,203.0
258,21586,2023-05-29,Chennai Super Kings,Gujarat Titans,Chennai Super Kings Won by 5 wickets (DLS met...,214,215,NaN,90.0,171.0
302,28046,2023-12-12,South Africa,India,South Africa won by 5 wickets (DLS),180,181,NaN,90.0,152.0
324,29984,2024-02-25,New Zealand,Australia,Australia Won by 27 Runs (DLS method),118,119,NaN,60.0,126.0



TARGET COMPARISON


,match_id,first_innings_score,ordinary_chase_target,revised_target,target_diff
8,13612,149,150,143.0,-7.0
142,16754,95,96,56.0,-40.0
150,17395,90,91,91.0,0.0
258,21586,214,215,171.0,-44.0
268,21070,207,208,104.0,-104.0
269,21071,202,203,203.0,0.0
302,28046,180,181,152.0,-29.0
324,29984,118,119,126.0,7.0
427,37354,93,94,94.0,0.0
445,36098,108,109,116.0,7.0


In [73]:
# Verify revised-over metadata units against recorded legal-ball counts

innings_limits = innings_ball_audit[
    ["match_id", "innings", "legal_balls_min", "legal_balls_max"]
].copy()

first_limits = innings_limits[
    innings_limits["innings"] == 1
].drop(columns="innings").rename(columns={
    "legal_balls_min": "first_legal_balls_min",
    "legal_balls_max": "first_legal_balls_max"
})

second_limits = innings_limits[
    innings_limits["innings"] == 2
].drop(columns="innings").rename(columns={
    "legal_balls_min": "second_legal_balls_min",
    "legal_balls_max": "second_legal_balls_max"
})

revised_overs_audit = target_audit.merge(
    first_limits, on="match_id", how="left", validate="one_to_one"
).merge(
    second_limits, on="match_id", how="left", validate="one_to_one"
)

# Test the hypothesis that metadata values are legal balls, not overs
revised_overs_audit["first_revised_balls_candidate"] = (
    revised_overs_audit["first_inn_revised_overs"]
)

revised_overs_audit["second_revised_balls_candidate"] = (
    revised_overs_audit["second_inn_revised_overs"]
)

revised_overs_audit["first_limit_violation"] = (
    revised_overs_audit["first_revised_balls_candidate"].notna()
    & (
        revised_overs_audit["first_legal_balls_min"]
        > revised_overs_audit["first_revised_balls_candidate"]
    )
)

revised_overs_audit["second_limit_violation"] = (
    revised_overs_audit["second_revised_balls_candidate"].notna()
    & (
        revised_overs_audit["second_legal_balls_min"]
        > revised_overs_audit["second_revised_balls_candidate"]
    )
)

print("REVISED-OVER UNIT VALIDATION")
print(
    "First-innings limit violations:",
    revised_overs_audit["first_limit_violation"].sum()
)
print(
    "Second-innings limit violations:",
    revised_overs_audit["second_limit_violation"].sum()
)

print("\nREVISED-OVER MATCH DETAILS")
display(
    revised_overs_audit.loc[
        revised_overs_audit[metadata_cols].notna().any(axis=1),
        [
            "match_id", "result",
            "first_inn_revised_overs",
            "first_legal_balls_min",
            "first_legal_balls_max",
            "second_inn_revised_overs",
            "second_legal_balls_min",
            "second_legal_balls_max",
            "revised_target"
        ]
    ].sort_values("match_id")
)

# Identify DLS matches without a recorded revised target
dls_mask = revised_overs_audit["result"].fillna("").str.contains(
    r"D/L|DLS", case=False, regex=True
)

dls_target_audit = revised_overs_audit.loc[
    dls_mask,
    [
        "match_id", "match_date", "result",
        "first_innings_score", "ordinary_chase_target",
        "first_inn_revised_overs", "second_inn_revised_overs",
        "revised_target"
    ]
].copy()

dls_target_audit["revised_target_missing"] = (
    dls_target_audit["revised_target"].isna()
)

print("\nDLS RESULT AUDIT")
print("DLS-labelled matches:", len(dls_target_audit))
print(
    "DLS-labelled matches missing revised target:",
    dls_target_audit["revised_target_missing"].sum()
)

display(dls_target_audit.sort_values("match_date"))

REVISED-OVER UNIT VALIDATION
First-innings limit violations: 0
Second-innings limit violations: 0

REVISED-OVER MATCH DETAILS


,match_id,result,first_inn_revised_overs,first_legal_balls_min,first_legal_balls_max,second_inn_revised_overs,second_legal_balls_min,second_legal_balls_max,revised_target
8,13612,Australia Won by 20 runs (D/L method),NaN,120,120,114.0,114,114,143.0
141,16753,Afghanistan Won by 27 Runs,66.0,66,66,NaN,65,66,NaN
142,16754,Ireland Won by 7 Wickets(D/L Method),NaN,90,90,42.0,40,40,56.0
150,17395,India Won by 6 Wickets,48.0,48,48,48.0,44,44,91.0
259,19959,West Indies won by 3 wickets,66.0,66,66,NaN,63,63,NaN
268,21070,Bangladesh Won by 22 Runs (DLS method),NaN,116,116,48.0,48,48,104.0
269,21071,Bangladesh Won by 77 Runs,102.0,102,102,102.0,102,102,203.0
258,21586,Chennai Super Kings Won by 5 wickets (DLS met...,NaN,120,120,90.0,90,90,171.0
302,28046,South Africa won by 5 wickets (DLS),NaN,117,117,90.0,83,83,152.0
324,29984,Australia Won by 27 Runs (DLS method),NaN,64,64,60.0,60,60,126.0



DLS RESULT AUDIT
DLS-labelled matches: 19
DLS-labelled matches missing revised target: 4


,match_id,match_date,result,first_innings_score,ordinary_chase_target,first_inn_revised_overs,second_inn_revised_overs,revised_target,revised_target_missing
8,13612,2022-02-11,Australia Won by 20 runs (D/L method),149,150,NaN,114.0,143.0,False
142,16754,2022-08-17,Ireland Won by 7 Wickets(D/L Method),95,96,NaN,42.0,56.0,False
268,21070,2023-03-27,Bangladesh Won by 22 Runs (DLS method),207,208,NaN,48.0,104.0,False
187,20182,2023-04-01,Punjab Kings Won by 7 Runs(D/L Method),191,192,NaN,NaN,NaN,True
258,21586,2023-05-29,Chennai Super Kings Won by 5 wickets (DLS met...,214,215,NaN,90.0,171.0,False
279,23674,2023-07-16,Bangladesh Won by 6 Wickets(D/L Method),116,117,NaN,NaN,NaN,True
288,24822,2023-08-18,India Won by 2 Runs (DLS method),139,140,NaN,NaN,NaN,True
302,28046,2023-12-12,South Africa won by 5 wickets (DLS),180,181,NaN,90.0,152.0,False
310,28334,2023-12-31,New Zealand won by 17runs (DLS Method),110,111,NaN,NaN,NaN,True
324,29984,2024-02-25,Australia Won by 27 Runs (DLS method),118,119,NaN,60.0,126.0,False


In [74]:
# Build a verified second-innings target and scheduled-balls table

chase_match_audit = target_audit[
    target_audit["match_id"].isin(primary_match_ids)
].copy()

# Identify DLS/D-L matches from the recorded result
chase_match_audit["is_dls_match"] = (
    chase_match_audit["result"]
    .fillna("")
    .str.contains(r"D/L|DLS", case=False, regex=True)
)

# Assign target only when supported by match metadata
chase_match_audit["chase_target"] = np.nan
chase_match_audit["target_source"] = "unknown"

has_revised_target = chase_match_audit["revised_target"].notna()

ordinary_match = (
    ~chase_match_audit["is_dls_match"]
    & ~has_revised_target
    & chase_match_audit["first_innings_score"].notna()
)

chase_match_audit.loc[
    has_revised_target, "chase_target"
] = chase_match_audit.loc[
    has_revised_target, "revised_target"
]

chase_match_audit.loc[
    has_revised_target, "target_source"
] = "recorded_revised_target"

chase_match_audit.loc[
    ordinary_match, "chase_target"
] = (
    chase_match_audit.loc[
        ordinary_match, "first_innings_score"
    ] + 1
)

chase_match_audit.loc[
    ordinary_match, "target_source"
] = "first_innings_score_plus_one"

# Scheduled legal-ball limit:
# Use recorded revised limit when present.
# Use 120 only for matches not identified as DLS/D-L.
chase_match_audit["scheduled_legal_balls"] = (
    chase_match_audit["second_inn_revised_overs"]
)

standard_full_length = (
    ~chase_match_audit["is_dls_match"]
    & chase_match_audit["scheduled_legal_balls"].isna()
)

chase_match_audit.loc[
    standard_full_length, "scheduled_legal_balls"
] = 120

chase_match_audit["scheduled_balls_source"] = "recorded_revised_limit"
chase_match_audit.loc[
    standard_full_length, "scheduled_balls_source"
] = "standard_120_ball_assumption"

chase_match_audit.loc[
    chase_match_audit["scheduled_legal_balls"].isna(),
    "scheduled_balls_source"
] = "unknown"

# Target and scheduled-balls eligibility are separate
chase_match_audit["target_verified"] = (
    chase_match_audit["chase_target"].notna()
)

chase_match_audit["scheduled_balls_verified"] = (
    chase_match_audit["scheduled_legal_balls"].notna()
)

print("CHASE TARGET AUDIT")
print("Matches:", len(chase_match_audit))
print("\nTarget source counts:")
display(
    chase_match_audit["target_source"]
    .value_counts(dropna=False)
    .to_frame("matches")
)

print("\nScheduled-ball source counts:")
display(
    chase_match_audit["scheduled_balls_source"]
    .value_counts(dropna=False)
    .to_frame("matches")
)

print("\nDLS/D-L matches with missing revised target:")
display(
    chase_match_audit.loc[
        chase_match_audit["is_dls_match"]
        & ~chase_match_audit["target_verified"],
        [
            "match_id", "match_date", "result",
            "first_innings_score", "revised_target",
            "second_inn_revised_overs"
        ]
    ].sort_values("match_date")
)

print("\nMatches with unknown scheduled legal-ball limit:")
display(
    chase_match_audit.loc[
        ~chase_match_audit["scheduled_balls_verified"],
        [
            "match_id", "match_date", "result",
            "chase_target", "target_source",
            "second_inn_revised_overs"
        ]
    ].sort_values("match_date")
)

# Basic consistency checks
assert len(chase_match_audit) == 813
assert (
    chase_match_audit.loc[
        chase_match_audit["is_dls_match"]
        & chase_match_audit["revised_target"].isna(),
        "chase_target"
    ].isna()
).all()

assert (
    chase_match_audit.loc[
        chase_match_audit["target_verified"],
        "chase_target"
    ] > 0
).all()

print("\nChase target audit created.")

CHASE TARGET AUDIT
Matches: 813

Target source counts:


,matches
target_source,
first_innings_score_plus_one,787
recorded_revised_target,22
unknown,4



Scheduled-ball source counts:


,matches
scheduled_balls_source,
standard_120_ball_assumption,787
recorded_revised_limit,22
unknown,4



DLS/D-L matches with missing revised target:


,match_id,match_date,result,first_innings_score,revised_target,second_inn_revised_overs
187,20182,2023-04-01,Punjab Kings Won by 7 Runs(D/L Method),191,NaN,NaN
279,23674,2023-07-16,Bangladesh Won by 6 Wickets(D/L Method),116,NaN,NaN
288,24822,2023-08-18,India Won by 2 Runs (DLS method),139,NaN,NaN
310,28334,2023-12-31,New Zealand won by 17runs (DLS Method),110,NaN,NaN



Matches with unknown scheduled legal-ball limit:


,match_id,match_date,result,chase_target,target_source,second_inn_revised_overs
187,20182,2023-04-01,Punjab Kings Won by 7 Runs(D/L Method),NaN,unknown,NaN
279,23674,2023-07-16,Bangladesh Won by 6 Wickets(D/L Method),NaN,unknown,NaN
288,24822,2023-08-18,India Won by 2 Runs (DLS method),NaN,unknown,NaN
310,28334,2023-12-31,New Zealand won by 17runs (DLS Method),NaN,unknown,NaN



Chase target audit created.


In [76]:
# Build second-innings pressure context using information available
# immediately BEFORE each delivery.

chase_context = eligible_df.loc[
    eligible_df["innings"] == 2
].copy()

chase_metadata = chase_match_audit[
    [
        "match_id",
        "chase_target",
        "target_source",
        "scheduled_legal_balls",
        "scheduled_balls_source",
        "target_verified",
        "scheduled_balls_verified"
    ]
].copy()

chase_context = chase_context.merge(
    chase_metadata,
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Keep only chases with a verified target and scheduled-ball limit.
chase_context["pressure_context_eligible"] = (
    chase_context["target_verified"].fillna(False)
    & chase_context["scheduled_balls_verified"].fillna(False)
)

# Wickets lost BEFORE the current delivery.
chase_context["wickets_lost_before"] = (
    chase_context.groupby(["match_id", "innings"])["is_wicket"]
    .transform(lambda s: s.astype(int).cumsum() - s.astype(int))
)

# Runs required BEFORE the current delivery.
chase_context["runs_required"] = (
    chase_context["chase_target"] - chase_context["innings_score_before"]
)

# Legal balls remaining as a range BEFORE the current delivery.
chase_context["balls_remaining_min"] = (
    chase_context["scheduled_legal_balls"]
    - chase_context["legal_balls_max_before"]
)

chase_context["balls_remaining_max"] = (
    chase_context["scheduled_legal_balls"]
    - chase_context["legal_balls_min_before"]
)

# Required run-rate bounds.
valid_remaining_balls = (
    chase_context["pressure_context_eligible"]
    & (chase_context["balls_remaining_min"] > 0)
    & (chase_context["runs_required"] > 0)
)

chase_context["required_run_rate_min"] = np.nan
chase_context["required_run_rate_max"] = np.nan

# Lower RRR uses the larger possible remaining-ball count.
chase_context.loc[
    valid_remaining_balls, "required_run_rate_min"
] = (
    chase_context.loc[valid_remaining_balls, "runs_required"] * 6
    / chase_context.loc[valid_remaining_balls, "balls_remaining_max"]
)

# Upper RRR uses the smaller possible remaining-ball count.
chase_context.loc[
    valid_remaining_balls, "required_run_rate_max"
] = (
    chase_context.loc[valid_remaining_balls, "runs_required"] * 6
    / chase_context.loc[valid_remaining_balls, "balls_remaining_min"]
)

# Keep only rows with a valid, positive chase requirement and remaining balls.
chase_context["valid_chase_state"] = (
    valid_remaining_balls
    & (chase_context["wickets_lost_before"] < 10)
)

print("CHASE CONTEXT COVERAGE")
print("Second-innings deliveries:", len(chase_context))
print(
    "Deliveries with verified target and schedule:",
    int(chase_context["pressure_context_eligible"].sum())
)
print(
    "Deliveries with valid positive chase state:",
    int(chase_context["valid_chase_state"].sum())
)
print(
    "Deliveries excluded due to unverified match context:",
    int((~chase_context["pressure_context_eligible"]).sum())
)

print("\nREQUIRED RUN-RATE RANGE SUMMARY")
display(
    chase_context.loc[
        chase_context["valid_chase_state"],
        [
            "runs_required",
            "wickets_lost_before",
            "balls_remaining_min",
            "balls_remaining_max",
            "required_run_rate_min",
            "required_run_rate_max"
        ]
    ].describe().round(2)
)

print("\nDELIVERY-LEVEL VALIDATION")
valid_rows = chase_context["valid_chase_state"]

assert (
    chase_context.loc[valid_rows, "runs_required"] > 0
).all()

assert (
    chase_context.loc[valid_rows, "balls_remaining_min"] > 0
).all()

assert (
    chase_context.loc[valid_rows, "balls_remaining_max"]
    >= chase_context.loc[valid_rows, "balls_remaining_min"]
).all()

assert (
    chase_context.loc[valid_rows, "required_run_rate_max"]
    >= chase_context.loc[valid_rows, "required_run_rate_min"]
).all()

assert (
    chase_context.loc[valid_rows, "wickets_lost_before"] < 10
).all()

print("All chase-context validation checks passed.")

C:\Users\yashg\AppData\Local\Temp\ipykernel_17032\2687871669.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chase_context["target_verified"].fillna(False)
C:\Users\yashg\AppData\Local\Temp\ipykernel_17032\2687871669.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & chase_context["scheduled_balls_verified"].fillna(False)


CHASE CONTEXT COVERAGE
Second-innings deliveries: 95146
Deliveries with verified target and schedule: 91508
Deliveries with valid positive chase state: 91453
Deliveries excluded due to unverified match context: 3638

REQUIRED RUN-RATE RANGE SUMMARY


,runs_required,wickets_lost_before,balls_remaining_min,balls_remaining_max,required_run_rate_min,required_run_rate_max
count,91453.00,91453.00,91453.00,91453.00,91453.00,91453.00
mean,102.60,2.71,64.13,64.19,11.37,11.47
std,54.87,2.29,33.15,33.11,13.01,13.76
min,1.00,0.00,1.00,1.00,0.07,0.07
25%,59.00,1.00,36.00,36.00,7.70,7.71
50%,101.00,2.00,65.00,65.00,9.67,9.68
75%,143.00,4.00,93.00,93.00,12.00,12.00
max,305.00,9.00,120.00,120.00,810.00,810.00



DELIVERY-LEVEL VALIDATION
All chase-context validation checks passed.


In [77]:
# Clean boolean handling to avoid pandas FutureWarning
target_ok = chase_context["target_verified"].astype("boolean").fillna(False)
schedule_ok = chase_context["scheduled_balls_verified"].astype("boolean").fillna(False)

chase_context["pressure_context_eligible"] = (
    target_ok & schedule_ok
).astype(bool)

# 1. Identify every match contributing excluded deliveries
excluded_context = chase_context.loc[
    ~chase_context["pressure_context_eligible"]
].copy()

print("UNVERIFIED MATCH CONTEXT")
print("Excluded deliveries:", len(excluded_context))
print("Excluded matches:", excluded_context["match_id"].nunique())

excluded_match_audit = (
    excluded_context.groupby(
        [
            "match_id",
            "target_source",
            "scheduled_balls_source"
        ],
        dropna=False
    )
    .agg(
        excluded_deliveries=("match_id", "size"),
        match_date=("delivery_ist_time", "min")
    )
    .reset_index()
    .sort_values("excluded_deliveries", ascending=False)
)

display(excluded_match_audit)

# 2. Inspect the 20 highest required run-rate observations
print("\nHIGHEST REQUIRED RUN-RATE OBSERVATIONS")

extreme_rrr = (
    chase_context.loc[
        chase_context["valid_chase_state"],
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "innings_score_before",
            "chase_target",
            "runs_required",
            "wickets_lost_before",
            "balls_remaining_min",
            "balls_remaining_max",
            "required_run_rate_min",
            "required_run_rate_max"
        ]
    ]
    .sort_values("required_run_rate_max", ascending=False)
    .head(20)
)

display(extreme_rrr)

# 3. Check the target and score ranges
print("\nTARGET AND SCORE RANGE CHECK")
display(
    chase_context.loc[
        chase_context["valid_chase_state"],
        [
            "chase_target",
            "innings_score_before",
            "runs_required",
            "balls_remaining_min",
            "balls_remaining_max"
        ]
    ].describe().round(2)
)

# 4. Count observations with unusually high RRR
rrr_threshold_counts = pd.DataFrame({
    "threshold": [15, 18, 20, 24, 30, 36, 60, 100],
    "deliveries": [
        int((
            chase_context.loc[
                chase_context["valid_chase_state"],
                "required_run_rate_min"
            ] >= threshold
        ).sum())
        for threshold in [15, 18, 20, 24, 30, 36, 60, 100]
    ]
})

print("\nREQUIRED RUN-RATE THRESHOLD COUNTS")
display(rrr_threshold_counts)

print("\nAudit completed.")

UNVERIFIED MATCH CONTEXT
Excluded deliveries: 3638
Excluded matches: 34


,match_id,target_source,scheduled_balls_source,excluded_deliveries,match_date
30,48026,NaN,NaN,130,2025-11-19 16:19:22 UTC
32,48028,NaN,NaN,130,2025-11-21 16:18:55 UTC
23,48019,NaN,NaN,130,2025-11-16 11:17:42 UTC
20,48016,NaN,NaN,129,2025-11-14 13:17:44 UTC
7,37088,NaN,NaN,126,2024-10-19 15:19:43 UTC
31,48027,NaN,NaN,126,2025-11-21 11:32:21 UTC
33,48029,NaN,NaN,126,2025-11-23 16:20:21 UTC
13,37094,NaN,NaN,125,2024-10-22 15:21:52 UTC
22,48018,NaN,NaN,125,2025-11-15 13:18:17 UTC
19,48015,NaN,NaN,124,2025-11-14 08:33:11 UTC



HIGHEST REQUIRED RUN-RATE OBSERVATIONS


,match_id,innings,overid,over_ball_num,striker_name,bowler_name,innings_score_before,chase_target,runs_required,wickets_lost_before,balls_remaining_min,balls_remaining_max,required_run_rate_min,required_run_rate_max
54644,37278,2,20,6,Tanzim Sakib,Nitish Reddy,163,298.0,135.0,7,1.0,1.0,810.0,810.0
15245,16663,2,20,6,Ibrahim Zadran,Dinesh Karthik,111,213.0,102.0,8,1.0,1.0,612.0,612.0
15933,16684,2,20,6,Hayden Walsh,Tim Southee,119,216.0,97.0,9,1.0,1.0,582.0,582.0
71734,45495,2,20,6,Ayush Shukla,FazalHaq Farooqi,94,189.0,95.0,9,1.0,1.0,570.0,570.0
94087,54213,2,20,6,Prince Yadav,Nathan Smith,172,260.0,88.0,9,1.0,1.0,528.0,528.0
54517,37277,2,20,6,Mustafizur Rahman,Nitish Reddy,135,222.0,87.0,9,1.0,1.0,522.0,522.0
94735,55079,2,20,6,Maheesh Theekshana,Naseem Shah,132,216.0,84.0,7,1.0,1.0,504.0,504.0
30688,21071,2,17,6,Benjamin White,Taskin Ahmed,124,203.0,79.0,9,1.0,1.0,474.0,474.0
29827,20247,2,20,6,Chetan Sakariya,Maheesh Theekshana,146,224.0,78.0,9,1.0,1.0,468.0,468.0
22482,20184,2,20,6,Abdul Samad,Navdeep Saini,127,204.0,77.0,8,1.0,2.0,231.0,462.0



TARGET AND SCORE RANGE CHECK


,chase_target,innings_score_before,runs_required,balls_remaining_min,balls_remaining_max
count,91453.00,91453.00,91453.00,91453.00,91453.00
mean,180.67,78.07,102.60,64.13,64.19
std,36.05,50.96,54.87,33.15,33.11
min,33.00,0.00,1.00,1.00,1.00
25%,157.00,36.00,59.00,36.00,36.00
50%,179.00,74.00,101.00,65.00,65.00
75%,206.00,115.00,143.00,93.00,93.00
max,305.00,264.00,305.00,120.00,120.00



REQUIRED RUN-RATE THRESHOLD COUNTS


,threshold,deliveries
0,15,10931
1,18,6770
2,20,5084
3,24,3492
4,30,2251
5,36,1606
6,60,680
7,100,292



Audit completed.


In [78]:
# Investigate all matches with unverified chase context

# Match-level innings structure from eligible delivery data
match_innings_structure = (
    eligible_df.groupby("match_id")
    .agg(
        innings_count=("innings", "nunique"),
        innings_numbers=("innings", lambda s: sorted(s.unique().tolist())),
        eligible_deliveries=("innings", "size")
    )
    .reset_index()
)

# Attach competition and match metadata
match_details = (
    matches_df[
        [
            "match_id", "comp_id", "match_date",
            "home_team", "away_team", "result"
        ]
    ]
    .merge(
        competitions_df[
            ["comp_id", "comp_name", "match_format", "season"]
        ],
        on="comp_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        match_innings_structure,
        on="match_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        chase_match_audit[
            [
                "match_id", "first_innings_score",
                "chase_target", "target_source",
                "scheduled_legal_balls", "scheduled_balls_source"
            ]
        ],
        on="match_id",
        how="left",
        validate="one_to_one"
    )
)

excluded_match_ids = excluded_context["match_id"].drop_duplicates()

print("ALL MATCHES WITH UNVERIFIED CHASE CONTEXT")
display(
    match_details.loc[
        match_details["match_id"].isin(excluded_match_ids),
        [
            "match_id", "match_date", "comp_name", "match_format",
            "season", "home_team", "away_team", "result",
            "innings_count", "innings_numbers",
            "first_innings_score", "chase_target",
            "target_source", "scheduled_legal_balls",
            "scheduled_balls_source"
        ]
    ].sort_values(["match_format", "match_date"]).to_string(index=False)
)

print("\nUNVERIFIED MATCHES BY FORMAT AND COMPETITION")
display(
    match_details.loc[
        match_details["match_id"].isin(excluded_match_ids)
    ].groupby(
        ["match_format", "comp_name"],
        dropna=False
    )
    .agg(
        matches=("match_id", "nunique"),
        deliveries=("eligible_deliveries", "sum")
    )
    .reset_index()
)

# Inspect extreme required-run-rate cases with full match context
extreme_match_details = extreme_rrr.merge(
    match_details[
        [
            "match_id", "comp_name", "match_format",
            "match_date", "home_team", "away_team", "result"
        ]
    ],
    on="match_id",
    how="left",
    validate="many_to_one"
)

print("\nEXTREME RRR CASES WITH MATCH CONTEXT")
display(
    extreme_match_details[
        [
            "match_id", "match_date", "match_format", "comp_name",
            "home_team", "away_team", "result",
            "overid", "over_ball_num",
            "innings_score_before", "chase_target",
            "runs_required", "wickets_lost_before",
            "balls_remaining_min", "balls_remaining_max",
            "required_run_rate_min", "required_run_rate_max"
        ]
    ].head(20)
)

print("\nAudit complete.")

ALL MATCHES WITH UNVERIFIED CHASE CONTEXT


' match_id match_date                                  comp_name match_format season            home_team             away_team                                  result  innings_count innings_numbers  first_innings_score  chase_target target_source  scheduled_legal_balls scheduled_balls_source\n    20182 2023-04-01                                   IPL 2023          T20   2023         Punjab Kings Kolkata Knight Riders  Punjab Kings Won by 7 Runs(D/L Method)            2.0          [1, 2]                191.0           NaN       unknown                    NaN                unknown\n    37085 2024-10-18      Mens T20 Emerging Teams Asia Cup 2024          T20   2024         Bangladesh A             Hong Kong           Bangladesh A Won by 5 wickets            2.0          [1, 2]                  NaN           NaN           NaN                    NaN                    NaN\n    37086 2024-10-18      Mens T20 Emerging Teams Asia Cup 2024          T20   2024          Sri Lanka A         Afgh


UNVERIFIED MATCHES BY FORMAT AND COMPETITION


,match_format,comp_name,matches,deliveries
0,T20,Asia Cup Rising Stars 2025,15,3454.0
1,T20,IPL 2023,1,222.0
2,T20,Mens T20 Emerging Teams Asia Cup 2024,15,3519.0
3,T20I,Afghanistan in Bangladesh T20I Series 2023,1,211.0
4,T20I,Bangladesh in New Zealand T20I Series 2023,1,218.0
5,T20I,India in Ireland T20I Series 2023,1,170.0



EXTREME RRR CASES WITH MATCH CONTEXT


,match_id,match_date,match_format,comp_name,home_team,away_team,result,overid,over_ball_num,innings_score_before,chase_target,runs_required,wickets_lost_before,balls_remaining_min,balls_remaining_max,required_run_rate_min,required_run_rate_max
0,37278,2024-10-12,T20I,Bangladesh in India T20I Series 2024,India,Bangladesh,India Won by 133 runs,20,6,163,298.0,135.0,7,1.0,1.0,810.0,810.0
1,16663,2022-09-08,T20I,Mens Twenty20 Asia Cup 2022,India,Afghanistan,India Won by 101 Runs,20,6,111,213.0,102.0,8,1.0,1.0,612.0,612.0
2,16684,2022-08-13,T20I,New Zealand in West Indies T20I Series 2022,West Indies,New Zealand,New Zealand Won by 90 Runs,20,6,119,216.0,97.0,9,1.0,1.0,582.0,582.0
3,45495,2025-09-09,T20I,Asia Cup 2025,Afghanistan,Hong Kong,Afghanistan Won by 94 runs,20,6,94,189.0,95.0,9,1.0,1.0,570.0,570.0
4,54213,2026-10-30,T20I,India in New Zealand T20I Series 2026,New Zealand,India,New Zealand Won by 87 runs,20,6,172,260.0,88.0,9,1.0,1.0,528.0,528.0
5,37277,2024-10-09,T20I,Bangladesh in India T20I Series 2024,India,Bangladesh,India Won by 86 runs,20,6,135,222.0,87.0,9,1.0,1.0,522.0,522.0
6,55079,2026-10-11,T20I,Sri Lanka in Pakistan T20I Series 2026,Pakistan,Sri Lanka,Pakistan Won by 82 runs,20,6,132,216.0,84.0,7,1.0,1.0,504.0,504.0
7,21071,2023-03-29,T20I,Ireland in Bangladesh T20I Series 2023,Bangladesh,Ireland,Bangladesh Won by 77 Runs,17,6,124,203.0,79.0,9,1.0,1.0,474.0,474.0
8,20247,2023-05-20,T20,IPL 2023,Delhi Capitals,Chennai Super Kings,Chennai Super Kings Won by 77 runs,20,6,146,224.0,78.0,9,1.0,1.0,468.0,468.0
9,20184,2023-04-02,T20,IPL 2023,Sunrisers Hyderabad,Rajasthan Royals,Rajasthan Royals Won By 72 Runs,20,6,127,204.0,77.0,8,1.0,2.0,231.0,462.0



Audit complete.


In [79]:
# Restrict chase analysis to the agreed primary scope: IPL + T20I
primary_chase = chase_context.loc[
    chase_context["match_id"].isin(primary_match_ids)
].copy()

# Recalculate context eligibility after scope restriction
target_ok = primary_chase["target_verified"].astype("boolean").fillna(False)
schedule_ok = primary_chase["scheduled_balls_verified"].astype("boolean").fillna(False)

primary_chase["pressure_context_eligible"] = (
    target_ok & schedule_ok
).astype(bool)

valid_chases = primary_chase.loc[
    primary_chase["pressure_context_eligible"]
    & (primary_chase["runs_required"] > 0)
    & (primary_chase["balls_remaining_min"] > 0)
    & (primary_chase["wickets_lost_before"] < 10)
].copy()

# Descriptive feasibility indicator:
# Can the required runs be scored at six runs per remaining legal ball?
valid_chases["maximum_six_per_ball_runs"] = (
    valid_chases["balls_remaining_min"] * 6
)
valid_chases["within_six_per_ball_ceiling"] = (
    valid_chases["runs_required"]
    <= valid_chases["maximum_six_per_ball_runs"]
)

print("PRIMARY SCOPE CHASE AUDIT")
print("IPL + T20I chase deliveries:", len(primary_chase))
print("Verified chase-context deliveries:", len(valid_chases))
print(
    "Unverified-context deliveries:",
    int((~primary_chase["pressure_context_eligible"]).sum())
)
print(
    "Unverified matches:",
    primary_chase.loc[
        ~primary_chase["pressure_context_eligible"], "match_id"
    ].nunique()
)

print("\nUNVERIFIED MATCHES IN PRIMARY SCOPE")
display(
    primary_chase.loc[
        ~primary_chase["pressure_context_eligible"],
        ["match_id", "target_source", "scheduled_balls_source"]
    ].drop_duplicates().sort_values("match_id")
)

print("\nREQUIRED RUN RATE DISTRIBUTION")
display(
    valid_chases[
        ["required_run_rate_min", "required_run_rate_max"]
    ].describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]).round(2)
)

print("\nCHASE FEASIBILITY PROFILE")
display(
    valid_chases.groupby("within_six_per_ball_ceiling")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        mean_rrr=("required_run_rate_min", "mean"),
        median_rrr=("required_run_rate_min", "median"),
        mean_wickets_lost=("wickets_lost_before", "mean"),
        mean_balls_remaining=("balls_remaining_min", "mean")
    )
    .round(2)
)

print("\nRRR THRESHOLDS WITHIN THE SIX-PER-BALL CEILING")
thresholds = [8, 10, 12, 15, 18, 20, 24, 30, 36]

display(pd.DataFrame({
    "rrr_threshold": thresholds,
    "deliveries": [
        int((
            (valid_chases["required_run_rate_min"] >= threshold)
            & valid_chases["within_six_per_ball_ceiling"]
        ).sum())
        for threshold in thresholds
    ],
    "matches": [
        valid_chases.loc[
            (valid_chases["required_run_rate_min"] >= threshold)
            & valid_chases["within_six_per_ball_ceiling"],
            "match_id"
        ].nunique()
        for threshold in thresholds
    ]
}))

print("\nAudit complete.")

PRIMARY SCOPE CHASE AUDIT
IPL + T20I chase deliveries: 91846
Verified chase-context deliveries: 91453
Unverified-context deliveries: 338
Unverified matches: 4

UNVERIFIED MATCHES IN PRIMARY SCOPE


,match_id,target_source,scheduled_balls_source
22133,20182,unknown,unknown
32152,23674,unknown,unknown
33170,24822,unknown,unknown
35888,28334,unknown,unknown



REQUIRED RUN RATE DISTRIBUTION


,required_run_rate_min,required_run_rate_max
count,91453.00,91453.00
mean,11.37,11.47
std,13.01,13.76
min,0.07,0.07
50%,9.67,9.68
75%,12.00,12.00
90%,15.86,15.93
95%,20.90,21.00
99%,48.80,51.00
max,810.00,810.00



CHASE FEASIBILITY PROFILE


,deliveries,matches,mean_rrr,median_rrr,mean_wickets_lost,mean_balls_remaining
within_six_per_ball_ceiling,,,,,,
False,1603,274,75.97,54.0,7.41,5.88
True,89850,808,10.22,9.6,2.63,65.17



RRR THRESHOLDS WITHIN THE SIX-PER-BALL CEILING


,rrr_threshold,deliveries,matches
0,8,63794,695
1,10,40583,560
2,12,21646,483
3,15,9328,414
4,18,5169,381
5,20,3483,360
6,24,1895,324
7,30,664,266
8,36,57,48



Audit complete.


In [80]:
# Explore pressure dimensions using only competitive, verified chase states

pressure_pool = valid_chases.loc[
    valid_chases["within_six_per_ball_ceiling"]
].copy()

# Use conservative RRR estimate: it does not assume unresolved deliveries were legal
pressure_pool["rrr_for_classification"] = (
    pressure_pool["required_run_rate_min"]
)

# Group innings stage by remaining legal balls
pressure_pool["innings_stage"] = pd.cut(
    pressure_pool["balls_remaining_min"],
    bins=[0, 30, 60, 90, 120],
    labels=["Final 1-5 overs", "Overs 11-15", "Overs 6-10", "Overs 1-5"],
    include_lowest=True
)

# Group wickets already lost
pressure_pool["wicket_pressure"] = pd.cut(
    pressure_pool["wickets_lost_before"],
    bins=[-1, 2, 5, 9],
    labels=["0-2 wickets lost", "3-5 wickets lost", "6-9 wickets lost"]
)

print("COMPETITIVE CHASE PRESSURE POOL")
print("Deliveries:", len(pressure_pool))
print("Matches:", pressure_pool["match_id"].nunique())

print("\nDELIVERIES BY RRR THRESHOLD AND WICKETS LOST")
rrr_thresholds = [10, 12, 15, 18, 20]

rrr_wicket_rows = []
for threshold in rrr_thresholds:
    for wicket_group in [
        "0-2 wickets lost",
        "3-5 wickets lost",
        "6-9 wickets lost"
    ]:
        subset = pressure_pool.loc[
            (pressure_pool["rrr_for_classification"] >= threshold)
            & (pressure_pool["wicket_pressure"] == wicket_group)
        ]
        rrr_wicket_rows.append({
            "rrr_at_least": threshold,
            "wickets_lost": wicket_group,
            "deliveries": len(subset),
            "matches": subset["match_id"].nunique()
        })

display(pd.DataFrame(rrr_wicket_rows))

print("\nRRR BY INNINGS STAGE")
display(
    pressure_pool.groupby(
        "innings_stage", observed=False
    ).agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        median_rrr=("rrr_for_classification", "median"),
        p75_rrr=("rrr_for_classification", lambda s: s.quantile(0.75)),
        p90_rrr=("rrr_for_classification", lambda s: s.quantile(0.90)),
        mean_wickets_lost=("wickets_lost_before", "mean")
    ).round(2)
)

print("\nWICKETS LOST BY INNINGS STAGE")
display(
    pressure_pool.pivot_table(
        index="innings_stage",
        columns="wicket_pressure",
        values="match_id",
        aggfunc="count",
        observed=False
    ).fillna(0).astype(int)
)

print("\nAudit complete.")

COMPETITIVE CHASE PRESSURE POOL
Deliveries: 89850
Matches: 808

DELIVERIES BY RRR THRESHOLD AND WICKETS LOST


,rrr_at_least,wickets_lost,deliveries,matches
0,10,0-2 wickets lost,14875,381
1,10,3-5 wickets lost,16593,481
2,10,6-9 wickets lost,9115,387
3,12,0-2 wickets lost,3877,146
4,12,3-5 wickets lost,9934,371
5,12,6-9 wickets lost,7835,374
6,15,0-2 wickets lost,372,28
7,15,3-5 wickets lost,3399,202
8,15,6-9 wickets lost,5557,351
9,18,0-2 wickets lost,36,5



RRR BY INNINGS STAGE


,deliveries,matches,median_rrr,p75_rrr,p90_rrr,mean_wickets_lost
innings_stage,,,,,,
Final 1-5 overs,16751,732,12.20,18.00,24.00,5.16
Overs 11-15,23862,792,10.20,12.75,15.22,3.46
Overs 6-10,24555,798,9.28,11.05,12.61,2.05
Overs 1-5,24682,791,9.05,10.45,11.57,0.67



WICKETS LOST BY INNINGS STAGE


wicket_pressure,0-2 wickets lost,3-5 wickets lost,6-9 wickets lost
innings_stage,,,
Final 1-5 overs,1571,7968,7212
Overs 11-15,7627,13044,3191
Overs 6-10,16106,8117,332
Overs 1-5,23614,1068,0



Audit complete.


In [81]:
# Compare candidate pressure definitions across RRR, wickets and innings stage

pressure_pool["rrr_band"] = pd.cut(
    pressure_pool["rrr_for_classification"],
    bins=[0, 10, 12, 15, 18, 10000],
    labels=["Below 10", "10-12", "12-15", "15-18", "18+"],
    right=False
)

# Cross-tabulation: innings stage × wickets lost × RRR band
stage_pressure_audit = (
    pressure_pool.groupby(
        ["innings_stage", "wicket_pressure", "rrr_band"],
        observed=False
    )
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

print("PRESSURE DISTRIBUTION BY STAGE, WICKETS AND RRR")
display(
    stage_pressure_audit.loc[
        stage_pressure_audit["deliveries"] > 0
    ].sort_values(
        ["innings_stage", "wicket_pressure", "rrr_band"]
    )
)

# Compare candidate definitions
candidate_rules = {
    "A: RRR >= 15 and 3+ wickets lost":
        (pressure_pool["rrr_for_classification"] >= 15)
        & (pressure_pool["wickets_lost_before"] >= 3),

    "B: RRR >= 12, 3+ wickets lost, final 5 overs":
        (pressure_pool["rrr_for_classification"] >= 12)
        & (pressure_pool["wickets_lost_before"] >= 3)
        & (pressure_pool["balls_remaining_min"] <= 30),

    "C: RRR >= 12, 3+ wickets lost, final 10 overs":
        (pressure_pool["rrr_for_classification"] >= 12)
        & (pressure_pool["wickets_lost_before"] >= 3)
        & (pressure_pool["balls_remaining_min"] <= 60),

    "D: RRR >= 15, 3+ wickets lost, final 10 overs":
        (pressure_pool["rrr_for_classification"] >= 15)
        & (pressure_pool["wickets_lost_before"] >= 3)
        & (pressure_pool["balls_remaining_min"] <= 60)
}

candidate_summary = []

for rule_name, mask in candidate_rules.items():
    selected = pressure_pool.loc[mask]

    candidate_summary.append({
        "candidate_rule": rule_name,
        "deliveries": len(selected),
        "matches": selected["match_id"].nunique(),
        "batters": selected["striker_id"].nunique(),
        "bowlers": selected["bowler_id"].nunique(),
        "median_rrr": selected["rrr_for_classification"].median(),
        "mean_wickets_lost": selected["wickets_lost_before"].mean()
    })

print("\nCANDIDATE PRESSURE RULE COMPARISON")
display(pd.DataFrame(candidate_summary).round(2))

print("\nAudit complete.")

PRESSURE DISTRIBUTION BY STAGE, WICKETS AND RRR


,innings_stage,wicket_pressure,rrr_band,deliveries,matches
0,Final 1-5 overs,0-2 wickets lost,Below 10,1310,112
1,Final 1-5 overs,0-2 wickets lost,10-12,69,16
2,Final 1-5 overs,0-2 wickets lost,12-15,96,14
3,Final 1-5 overs,0-2 wickets lost,15-18,64,10
4,Final 1-5 overs,0-2 wickets lost,18+,32,4
5,Final 1-5 overs,3-5 wickets lost,Below 10,4023,289
6,Final 1-5 overs,3-5 wickets lost,10-12,967,143
7,Final 1-5 overs,3-5 wickets lost,12-15,1264,170
8,Final 1-5 overs,3-5 wickets lost,15-18,689,127
9,Final 1-5 overs,3-5 wickets lost,18+,1025,110



CANDIDATE PRESSURE RULE COMPARISON


,candidate_rule,deliveries,matches,batters,bowlers,median_rrr,mean_wickets_lost
0,A: RRR >= 15 and 3+ wickets lost,8956,414,456,298,18.54,6.15
1,"B: RRR >= 12, 3+ wickets lost, final 5 overs",8503,456,477,294,17.71,6.24
2,"C: RRR >= 12, 3+ wickets lost, final 10 overs",15266,475,532,343,15.50,5.68
3,"D: RRR >= 15, 3+ wickets lost, final 10 overs",8558,414,451,293,18.71,6.24



Audit complete.


In [82]:
# Provisional primary and sensitivity pressure definitions

pressure_pool["primary_pressure"] = (
    (pressure_pool["rrr_for_classification"] >= 15)
    & (pressure_pool["wickets_lost_before"] >= 3)
    & (pressure_pool["balls_remaining_min"] <= 60)
)

pressure_pool["sensitivity_pressure"] = (
    (pressure_pool["rrr_for_classification"] >= 12)
    & (pressure_pool["wickets_lost_before"] >= 3)
    & (pressure_pool["balls_remaining_min"] <= 30)
)

# Attach competition format for IPL / T20I breakdown
match_format_lookup = (
    matches_df[["match_id", "comp_id"]]
    .merge(
        competitions_df[["comp_id", "match_format"]],
        on="comp_id",
        how="left",
        validate="many_to_one"
    )
    [["match_id", "match_format"]]
    .drop_duplicates()
)

pressure_pool = pressure_pool.merge(
    match_format_lookup,
    on="match_id",
    how="left",
    validate="many_to_one"
)

# Primary and sensitivity sample sizes by format
print("PRESSURE SAMPLE BY FORMAT")
format_pressure_summary = (
    pressure_pool.groupby("match_format")
    .agg(
        chase_deliveries=("match_id", "size"),
        primary_pressure_deliveries=("primary_pressure", "sum"),
        primary_pressure_matches=(
            "primary_pressure",
            lambda s: pressure_pool.loc[s.index[s], "match_id"].nunique()
        ),
        sensitivity_pressure_deliveries=("sensitivity_pressure", "sum"),
        sensitivity_pressure_matches=(
            "sensitivity_pressure",
            lambda s: pressure_pool.loc[s.index[s], "match_id"].nunique()
        )
    )
    .reset_index()
)
display(format_pressure_summary)

# Player exposure to primary pressure situations
primary_df = pressure_pool.loc[
    pressure_pool["primary_pressure"]
].copy()

batter_pressure_exposure = (
    primary_df.groupby(["striker_id", "striker_name"])
    .agg(
        pressure_deliveries=("match_id", "size"),
        pressure_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["pressure_deliveries", "pressure_matches"],
        ascending=False
    )
)

bowler_pressure_exposure = (
    primary_df.groupby(["bowler_id", "bowler_name"])
    .agg(
        pressure_deliveries=("match_id", "size"),
        pressure_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["pressure_deliveries", "pressure_matches"],
        ascending=False
    )
)

print("\nBATTER PRESSURE EXPOSURE")
print("Unique batters:", len(batter_pressure_exposure))
display(batter_pressure_exposure.head(20))

print("\nBOWLER PRESSURE EXPOSURE")
print("Unique bowlers:", len(bowler_pressure_exposure))
display(bowler_pressure_exposure.head(20))

# Exposure distribution: do enough players have repeated opportunities?
print("\nBATTER EXPOSURE DISTRIBUTION")
display(
    batter_pressure_exposure["pressure_deliveries"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .to_frame("pressure_deliveries")
    .round(2)
)

print("\nBOWLER EXPOSURE DISTRIBUTION")
display(
    bowler_pressure_exposure["pressure_deliveries"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .to_frame("pressure_deliveries")
    .round(2)
)

print("\nPLAYER EXPOSURE THRESHOLD COUNTS")
exposure_thresholds = [6, 12, 18, 24, 30, 60]

display(pd.DataFrame({
    "minimum_pressure_deliveries": exposure_thresholds,
    "batters_qualifying": [
        int((batter_pressure_exposure["pressure_deliveries"] >= n).sum())
        for n in exposure_thresholds
    ],
    "bowlers_qualifying": [
        int((bowler_pressure_exposure["pressure_deliveries"] >= n).sum())
        for n in exposure_thresholds
    ]
}))

print("\nAudit complete.")

PRESSURE SAMPLE BY FORMAT


KeyError: 'match_format'

In [83]:
# Inspect format columns and safely attach match_format

print("Existing format-related columns:")
print([
    col for col in pressure_pool.columns
    if "format" in col.lower()
])

# Remove any previous incomplete format merge columns
pressure_pool = pressure_pool.drop(
    columns=[
        col for col in ["match_format", "match_format_x", "match_format_y"]
        if col in pressure_pool.columns
    ]
)

# Build a clean match-to-format lookup
match_format_lookup = (
    matches_df[["match_id", "comp_id"]]
    .merge(
        competitions_df[["comp_id", "match_format"]],
        on="comp_id",
        how="left",
        validate="many_to_one"
    )
    [["match_id", "match_format"]]
    .drop_duplicates()
)

# Attach format to pressure deliveries
pressure_pool = pressure_pool.merge(
    match_format_lookup,
    on="match_id",
    how="left",
    validate="many_to_one"
)

print("\nFormat attachment validation:")
print("Missing formats:", pressure_pool["match_format"].isna().sum())
print("\nFormat counts:")
display(pressure_pool["match_format"].value_counts(dropna=False))

print("\nReady for pressure exposure analysis.")

Existing format-related columns:
['match_format_x', 'match_format_y']

Format attachment validation:
Missing formats: 0

Format counts:


match_format
T20I    48656
T20     41194
Name: count, dtype: int64


Ready for pressure exposure analysis.


In [84]:
# Measure primary and sensitivity pressure exposure by format

primary_df = pressure_pool.loc[
    pressure_pool["primary_pressure"]
].copy()

sensitivity_df = pressure_pool.loc[
    pressure_pool["sensitivity_pressure"]
].copy()

# 1. Overall pressure sample by format
format_rows = []

for fmt in sorted(pressure_pool["match_format"].dropna().unique()):
    all_fmt = pressure_pool.loc[pressure_pool["match_format"] == fmt]
    primary_fmt = primary_df.loc[primary_df["match_format"] == fmt]
    sensitivity_fmt = sensitivity_df.loc[sensitivity_df["match_format"] == fmt]

    format_rows.append({
        "format": fmt,
        "chase_pool_deliveries": len(all_fmt),
        "primary_pressure_deliveries": len(primary_fmt),
        "primary_pressure_matches": primary_fmt["match_id"].nunique(),
        "sensitivity_pressure_deliveries": len(sensitivity_fmt),
        "sensitivity_pressure_matches": sensitivity_fmt["match_id"].nunique()
    })

format_pressure_summary = pd.DataFrame(format_rows)

print("PRESSURE SAMPLE BY FORMAT")
display(format_pressure_summary)

# 2. Batter exposure, separately by format
batter_pressure_exposure = (
    primary_df.groupby(
        ["match_format", "striker_id", "striker_name"]
    )
    .agg(
        pressure_deliveries=("match_id", "size"),
        pressure_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["match_format", "pressure_deliveries", "pressure_matches"],
        ascending=[True, False, False]
    )
)

# 3. Bowler exposure, separately by format
bowler_pressure_exposure = (
    primary_df.groupby(
        ["match_format", "bowler_id", "bowler_name"]
    )
    .agg(
        pressure_deliveries=("match_id", "size"),
        pressure_matches=("match_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["match_format", "pressure_deliveries", "pressure_matches"],
        ascending=[True, False, False]
    )
)

print("\nBATTER PRESSURE EXPOSURE — TOP 20 PER FORMAT")
for fmt in sorted(batter_pressure_exposure["match_format"].unique()):
    print(f"\n{fmt}")
    display(
        batter_pressure_exposure.loc[
            batter_pressure_exposure["match_format"] == fmt
        ].head(20)
    )

print("\nBOWLER PRESSURE EXPOSURE — TOP 20 PER FORMAT")
for fmt in sorted(bowler_pressure_exposure["match_format"].unique()):
    print(f"\n{fmt}")
    display(
        bowler_pressure_exposure.loc[
            bowler_pressure_exposure["match_format"] == fmt
        ].head(20)
    )

# 4. Count players meeting different pressure-delivery thresholds
exposure_thresholds = [6, 12, 18, 24, 30, 60]
threshold_rows = []

for fmt in sorted(primary_df["match_format"].unique()):
    batter_fmt = batter_pressure_exposure.loc[
        batter_pressure_exposure["match_format"] == fmt
    ]
    bowler_fmt = bowler_pressure_exposure.loc[
        bowler_pressure_exposure["match_format"] == fmt
    ]

    for threshold in exposure_thresholds:
        threshold_rows.append({
            "format": fmt,
            "minimum_pressure_deliveries": threshold,
            "batters_qualifying": int(
                (batter_fmt["pressure_deliveries"] >= threshold).sum()
            ),
            "bowlers_qualifying": int(
                (bowler_fmt["pressure_deliveries"] >= threshold).sum()
            )
        })

print("\nPLAYER EXPOSURE THRESHOLD COUNTS")
display(pd.DataFrame(threshold_rows))

print("\nAudit complete.")

PRESSURE SAMPLE BY FORMAT


,format,chase_pool_deliveries,primary_pressure_deliveries,primary_pressure_matches,sensitivity_pressure_deliveries,sensitivity_pressure_matches
0,T20,41194,3990,184,4106,208
1,T20I,48656,4568,230,4397,248



BATTER PRESSURE EXPOSURE — TOP 20 PER FORMAT

T20


,match_format,striker_id,striker_name,pressure_deliveries,pressure_matches
0,T20,8,Mahendra Singh Dhoni,115,13
104,T20,4708,Rinku Singh,112,7
6,T20,152,Ravindra Jadeja,111,13
88,T20,3926,Rahul Tewatia,93,14
62,T20,2756,Shashank Singh,93,10
128,T20,6396,Tim David,87,12
66,T20,2876,Axar Patel,86,12
105,T20,4722,Rashid Khan,82,12
154,T20,7446,Abdul Samad,78,8
80,T20,3644,Kuldeep Yadav,70,11



T20I


,match_format,striker_id,striker_name,pressure_deliveries,pressure_matches
418,T20I,6726,Romario Shepherd,111,13
266,T20I,2633,Jason Holder,89,10
372,T20I,4986,Rovman Powell,78,8
445,T20I,7897,Namboori Tilak Varma,66,5
227,T20I,1077,David Miller,64,5
324,T20I,4101,Mitchell Santner,61,6
257,T20I,2417,Akeal Hosein,60,12
263,T20I,2490,Mohammad Nawaz,59,4
481,T20I,19316,Maheesh Theekshana,53,10
223,T20I,796,Matthew Wade,45,7


... (3 outputs left out)

,match_format,bowler_id,bowler_name,pressure_deliveries,pressure_matches
290,T20I,6464,Arshdeep Singh,140,20
311,T20I,7503,Ravi Bishnoi,114,13
147,T20I,479,Adil Rashid,81,11
194,T20I,2876,Axar Patel,73,12
226,T20I,4101,Mitchell Santner,67,8
279,T20I,6256,Varun Chakravarthy,67,7
285,T20I,6355,Shaheen Afridi,66,9
218,T20I,3653,Avesh Khan,63,9
173,T20I,2261,Harshal Patel,63,8
233,T20I,4409,Sam Curran,62,8



PLAYER EXPOSURE THRESHOLD COUNTS


,format,minimum_pressure_deliveries,batters_qualifying,bowlers_qualifying
0,T20,6,150,120
1,T20,12,99,89
2,T20,18,71,71
3,T20,24,53,59
4,T20,30,41,50
5,T20,60,14,19
6,T20I,6,223,196
7,T20I,12,137,130
8,T20I,18,89,87
9,T20I,24,58,69



Audit complete.


In [85]:
# Evaluate player eligibility using BOTH pressure deliveries and matches

joint_thresholds = [
    (6, 3),
    (12, 4),
    (12, 5),
    (18, 5),
    (18, 6),
    (24, 6),
    (24, 8),
    (30, 8)
]

joint_exposure_rows = []

for fmt in ["T20", "T20I"]:

    batter_fmt = batter_pressure_exposure.loc[
        batter_pressure_exposure["match_format"] == fmt
    ]

    bowler_fmt = bowler_pressure_exposure.loc[
        bowler_pressure_exposure["match_format"] == fmt
    ]

    for min_deliveries, min_matches in joint_thresholds:

        joint_exposure_rows.append({
            "format": fmt,
            "minimum_deliveries": min_deliveries,
            "minimum_matches": min_matches,
            "eligible_batters": int(
                (
                    (batter_fmt["pressure_deliveries"] >= min_deliveries)
                    & (batter_fmt["pressure_matches"] >= min_matches)
                ).sum()
            ),
            "eligible_bowlers": int(
                (
                    (bowler_fmt["pressure_deliveries"] >= min_deliveries)
                    & (bowler_fmt["pressure_matches"] >= min_matches)
                ).sum()
            )
        })

joint_exposure_summary = pd.DataFrame(joint_exposure_rows)

print("JOINT PLAYER EXPOSURE ELIGIBILITY")
display(joint_exposure_summary)

# Inspect how pressure exposure is distributed across matches
print("\nBATTER PRESSURE MATCH DISTRIBUTION")
display(
    batter_pressure_exposure.groupby("match_format")["pressure_matches"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90])
    .round(2)
)

print("\nBOWLER PRESSURE MATCH DISTRIBUTION")
display(
    bowler_pressure_exposure.groupby("match_format")["pressure_matches"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90])
    .round(2)
)

print("\nAudit complete.")

JOINT PLAYER EXPOSURE ELIGIBILITY


,format,minimum_deliveries,minimum_matches,eligible_batters,eligible_bowlers
0,T20,6,3,88,78
1,T20,12,4,61,57
2,T20,12,5,48,50
3,T20,18,5,48,50
4,T20,18,6,32,46
5,T20,24,6,30,46
6,T20,24,8,18,25
7,T20,30,8,17,25
8,T20I,6,3,97,94
9,T20I,12,4,60,64



BATTER PRESSURE MATCH DISTRIBUTION


,count,mean,std,min,25%,50%,75%,90%,max
match_format,,,,,,,,,
T20,209.0,3.09,2.83,1.0,1.0,2.0,4.0,7.0,14.0
T20I,338.0,2.30,1.89,1.0,1.0,2.0,3.0,5.0,13.0



BOWLER PRESSURE MATCH DISTRIBUTION


,count,mean,std,min,25%,50%,75%,90%,max
match_format,,,,,,,,,
T20,139.0,4.29,3.56,1.0,1.0,3.0,6.5,11.0,15.0
T20I,237.0,3.04,2.77,1.0,1.0,2.0,4.0,7.0,20.0



Audit complete.


In [86]:
# Build the baseline population: competitive second-innings chases
# restricted to IPL and T20I

# Attach competition information to the chase deliveries
baseline_chase_df = chase_context.drop(
    columns=[
        col for col in ["match_format", "comp_name"]
        if col in chase_context.columns
    ]
).merge(
    matches_df[["match_id", "comp_id"]],
    on="match_id",
    how="left",
    validate="many_to_one"
).merge(
    competitions_df[["comp_id", "comp_name", "match_format"]],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

# Keep IPL and T20I only
primary_scope_mask = (
    (baseline_chase_df["match_format"] == "T20I")
    | (
        (baseline_chase_df["match_format"] == "T20")
        & baseline_chase_df["comp_name"].str.contains(
            "Indian Premier League|IPL",
            case=False,
            na=False,
            regex=True
        )
    )
)

baseline_chase_df = baseline_chase_df.loc[
    primary_scope_mask
].copy()

# Keep only verified, valid and mathematically competitive chase states
baseline_chase_df["within_six_per_ball_ceiling"] = (
    baseline_chase_df["runs_required"]
    <= baseline_chase_df["balls_remaining_min"] * 6
)

baseline_chase_df = baseline_chase_df.loc[
    baseline_chase_df["pressure_context_eligible"].fillna(False)
    & baseline_chase_df["valid_chase_state"].fillna(False)
    & baseline_chase_df["within_six_per_ball_ceiling"]
].copy()

# Recreate the primary pressure flag on this baseline population
baseline_chase_df["primary_pressure"] = (
    (baseline_chase_df["rrr_for_classification"] >= 15)
    & (baseline_chase_df["wickets_lost_before"] >= 3)
    & (baseline_chase_df["balls_remaining_min"] <= 60)
)

print("BASELINE POPULATION VALIDATION")
print("Total competitive chase deliveries:", len(baseline_chase_df))
print("Matches:", baseline_chase_df["match_id"].nunique())
print("Primary pressure deliveries:", int(baseline_chase_df["primary_pressure"].sum()))

print("\nBASELINE BY FORMAT")
display(
    baseline_chase_df.groupby("match_format")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        pressure_deliveries=("primary_pressure", "sum"),
        batters=("striker_id", "nunique"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

print("\nPRESSURE DELIVERY COUNT CROSS-CHECK")
display(
    pressure_pool.groupby("match_format")
    .agg(
        pressure_deliveries=("primary_pressure", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

print("\nAudit complete.")

KeyError: 'rrr_for_classification'

In [87]:
# Inspect chase-context columns and RRR availability

print("RRR-related columns in chase_context:")
print([
    col for col in chase_context.columns
    if "rrr" in col.lower() or "run_rate" in col.lower()
])

print("\nRRR-related columns in pressure_pool:")
print([
    col for col in pressure_pool.columns
    if "rrr" in col.lower() or "run_rate" in col.lower()
])

print("\nRequired chase-context columns:")
required_columns = [
    "runs_required",
    "balls_remaining_min",
    "balls_remaining_max",
    "required_run_rate_min",
    "required_run_rate_max",
    "valid_chase_state",
    "pressure_context_eligible"
]

for col in required_columns:
    print(f"{col}: {'Available' if col in chase_context.columns else 'MISSING'}")

RRR-related columns in chase_context:
['required_run_rate_min', 'required_run_rate_max']

RRR-related columns in pressure_pool:
['required_run_rate_min', 'required_run_rate_max', 'rrr_for_classification', 'rrr_band']

Required chase-context columns:
runs_required: Available
balls_remaining_min: Available
balls_remaining_max: Available
required_run_rate_min: Available
required_run_rate_max: Available
valid_chase_state: Available
pressure_context_eligible: Available


In [88]:
# Build a clean baseline population from verified competitive chases

baseline_chase_df = chase_context.drop(
    columns=[
        col for col in ["match_format", "comp_name", "comp_id"]
        if col in chase_context.columns
    ]
).merge(
    matches_df[["match_id", "comp_id"]],
    on="match_id",
    how="left",
    validate="many_to_one"
).merge(
    competitions_df[["comp_id", "comp_name", "match_format"]],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

# Keep IPL and T20I only
primary_scope_mask = (
    (baseline_chase_df["match_format"] == "T20I")
    | (
        (baseline_chase_df["match_format"] == "T20")
        & baseline_chase_df["comp_name"].str.contains(
            "Indian Premier League|IPL",
            case=False,
            na=False,
            regex=True
        )
    )
)

baseline_chase_df = baseline_chase_df.loc[
    primary_scope_mask
].copy()

# Use the conservative RRR estimate consistently
baseline_chase_df["rrr_for_classification"] = (
    baseline_chase_df["required_run_rate_max"]
)

# Competitive chase-state validation
baseline_chase_df["within_six_per_ball_ceiling"] = (
    baseline_chase_df["runs_required"]
    <= baseline_chase_df["balls_remaining_min"] * 6
)

baseline_chase_df = baseline_chase_df.loc[
    baseline_chase_df["pressure_context_eligible"].fillna(False)
    & baseline_chase_df["valid_chase_state"].fillna(False)
    & baseline_chase_df["within_six_per_ball_ceiling"]
].copy()

# Apply the primary pressure definition
baseline_chase_df["primary_pressure"] = (
    (baseline_chase_df["rrr_for_classification"] >= 15)
    & (baseline_chase_df["wickets_lost_before"] >= 3)
    & (baseline_chase_df["balls_remaining_min"] <= 60)
)

print("BASELINE POPULATION VALIDATION")
print("Total competitive chase deliveries:", len(baseline_chase_df))
print("Matches:", baseline_chase_df["match_id"].nunique())
print("Primary pressure deliveries:", int(baseline_chase_df["primary_pressure"].sum()))

print("\nBASELINE BY FORMAT")
display(
    baseline_chase_df.groupby("match_format")
    .agg(
        deliveries=("match_id", "size"),
        matches=("match_id", "nunique"),
        pressure_deliveries=("primary_pressure", "sum"),
        batters=("striker_id", "nunique"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

print("\nPRESSURE DELIVERY COUNT CROSS-CHECK")
display(
    pressure_pool.groupby("match_format")
    .agg(
        pressure_deliveries=("primary_pressure", "sum"),
        matches=("match_id", "nunique")
    )
    .reset_index()
)

print("\nAudit complete.")

BASELINE POPULATION VALIDATION
Total competitive chase deliveries: 89850
Matches: 808
Primary pressure deliveries: 8668

BASELINE BY FORMAT


,match_format,deliveries,matches,pressure_deliveries,batters,bowlers
0,T20,41194,361,4050,305,237
1,T20I,48656,447,4618,590,415



PRESSURE DELIVERY COUNT CROSS-CHECK


,match_format,pressure_deliveries,matches
0,T20,3990,361
1,T20I,4568,447



Audit complete.


In [89]:
# Compare the original pressure classification with both RRR bounds

print("RRR CLASSIFICATION COMPARISON")

print(
    "Original RRR equals required_run_rate_min:",
    np.allclose(
        pressure_pool["rrr_for_classification"],
        pressure_pool["required_run_rate_min"],
        equal_nan=True
    )
)

print(
    "Original RRR equals required_run_rate_max:",
    np.allclose(
        pressure_pool["rrr_for_classification"],
        pressure_pool["required_run_rate_max"],
        equal_nan=True
    )
)

# Compare candidate pressure counts using the same pressure-pool rows
rrr_definitions = {
    "Original classification RRR": pressure_pool["rrr_for_classification"],
    "Lower RRR bound": pressure_pool["required_run_rate_min"],
    "Upper RRR bound": pressure_pool["required_run_rate_max"]
}

comparison_rows = []

for label, rrr_values in rrr_definitions.items():
    mask = (
        (rrr_values >= 15)
        & (pressure_pool["wickets_lost_before"] >= 3)
        & (pressure_pool["balls_remaining_min"] <= 60)
    )

    for fmt in ["T20", "T20I"]:
        fmt_mask = pressure_pool["match_format"] == fmt
        selected = pressure_pool.loc[mask & fmt_mask]

        comparison_rows.append({
            "rrr_definition": label,
            "format": fmt,
            "pressure_deliveries": len(selected),
            "pressure_matches": selected["match_id"].nunique()
        })

display(pd.DataFrame(comparison_rows))

# Inspect RRR uncertainty among deliveries near the threshold
near_threshold = pressure_pool.loc[
    pressure_pool["required_run_rate_min"].between(14, 16)
    | pressure_pool["required_run_rate_max"].between(14, 16),
    [
        "match_id",
        "match_format",
        "required_run_rate_min",
        "required_run_rate_max",
        "rrr_for_classification",
        "wickets_lost_before",
        "balls_remaining_min"
    ]
].copy()

print("\nDELIVERIES NEAR RRR THRESHOLD")
print("Rows:", len(near_threshold))
display(near_threshold.head(20))

print("\nDiagnostic complete.")

RRR CLASSIFICATION COMPARISON
Original RRR equals required_run_rate_min: True
Original RRR equals required_run_rate_max: False


,rrr_definition,format,pressure_deliveries,pressure_matches
0,Original classification RRR,T20,3990,184
1,Original classification RRR,T20I,4568,230
2,Lower RRR bound,T20,3990,184
3,Lower RRR bound,T20I,4568,230
4,Upper RRR bound,T20,4050,186
5,Upper RRR bound,T20I,4618,233



DELIVERIES NEAR RRR THRESHOLD
Rows: 4863


,match_id,match_format,required_run_rate_min,required_run_rate_max,rrr_for_classification,wickets_lost_before,balls_remaining_min
185,13512,T20I,14.000000,14.000000,14.000000,7,39.0
186,13512,T20I,14.210526,14.210526,14.210526,7,38.0
189,13512,T20I,14.228571,14.228571,14.228571,7,35.0
190,13512,T20I,14.470588,14.470588,14.470588,7,34.0
192,13512,T20I,14.062500,14.062500,14.062500,7,32.0
193,13512,T20I,14.322581,14.322581,14.322581,7,31.0
194,13512,T20I,14.800000,14.800000,14.800000,7,30.0
195,13512,T20I,15.310345,15.310345,15.310345,8,29.0
196,13512,T20I,15.642857,15.642857,15.642857,8,28.0
197,13512,T20I,15.777778,15.777778,15.777778,8,27.0



Diagnostic complete.


In [90]:
# Use the same RRR definition as the original pressure pool

baseline_chase_df["rrr_for_classification"] = (
    baseline_chase_df["required_run_rate_min"]
)

baseline_chase_df["primary_pressure"] = (
    (baseline_chase_df["rrr_for_classification"] >= 15)
    & (baseline_chase_df["wickets_lost_before"] >= 3)
    & (baseline_chase_df["balls_remaining_min"] <= 60)
)

# Compare counts by format
baseline_check = (
    baseline_chase_df.groupby("match_format")
    .agg(
        baseline_pressure_deliveries=("primary_pressure", "sum"),
        baseline_matches=("match_id", "nunique")
    )
    .reset_index()
)

original_check = (
    pressure_pool.groupby("match_format")
    .agg(
        original_pressure_deliveries=("primary_pressure", "sum"),
        original_matches=("match_id", "nunique")
    )
    .reset_index()
)

comparison = baseline_check.merge(
    original_check,
    on="match_format",
    how="outer",
    validate="one_to_one"
)

comparison["delivery_count_matches"] = (
    comparison["baseline_pressure_deliveries"]
    == comparison["original_pressure_deliveries"]
)

comparison["match_count_matches"] = (
    comparison["baseline_matches"]
    == comparison["original_matches"]
)

print("BASELINE VS ORIGINAL PRESSURE VALIDATION")
display(comparison)

print("\nTotal baseline pressure deliveries:",
      int(baseline_chase_df["primary_pressure"].sum()))

print("Total baseline deliveries:", len(baseline_chase_df))
print("Total baseline matches:", baseline_chase_df["match_id"].nunique())

print("\nAll pressure counts match:",
      comparison["delivery_count_matches"].all())

print("\nAudit complete.")

BASELINE VS ORIGINAL PRESSURE VALIDATION


,match_format,baseline_pressure_deliveries,baseline_matches,original_pressure_deliveries,original_matches,delivery_count_matches,match_count_matches
0,T20,3990,361,3990,361,True,True
1,T20I,4568,447,4568,447,True,True



Total baseline pressure deliveries: 8558
Total baseline deliveries: 89850
Total baseline matches: 808

All pressure counts match: True

Audit complete.
